# To load data from NAS - python sessdfGenerator.py in conda env 

In [4]:
from utils.opconNosepokeFunctions import *
from utils.supplementaryFunctions import *
from scipy.optimize import minimize
from scipy.stats import entropy
from scipy.stats import ttest_rel
%load_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np

from utils.plotSettings import *
%matplotlib qt

from utils.dfLoading import read_df
sessdf = read_df('R:/behavioralData/processed/sessdf-2026-09-25.csv', task_type = ['unstr', 'str', 'dms', 'dls', 'sham'])

# Apply to sessdf
from utils.dfLoading import add_block_groups
sessdf = add_block_groups(sessdf)
sessdf = sessdf[~(sessdf.rewarded_by_wrong_port==True)]
sessdf['day_task'] = sessdf.groupby('animal')['datetime'].transform(
    lambda x: ((x - x.min()).dt.days) + 1
)
# sessdf = sessdf[~(sessdf.animal == 'Alakazam')]

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [136]:
sessdf[sessdf.animal.isin(['Sableye', 'Torchic', 'Togepi', 'Vulpix', 'Whismur', 'Xatu', 'Alakazam', 'Onix', 'Ninetales', 'Jirachi', 'test05022023', 'Emolga', 'Giratina']) & (sessdf.task == 'unstr') & (sessdf.day_task>30)].groupby('animal').count().iloc[:, :15]

,task,block_num,trial,port,reward,trial_rewprob,rewprobfull,poke_in_time,poke_out_time,first_lick_time,last_lick_time,eptime,n_dup81,rewarded_by_wrong_port,datetime
animal,,,,,,,,,,,,,,,
Alakazam,120010,120010,120010,120010,120009,119999,120010,120010,120010,119832,119832,120010,120010,120010,120010
Emolga,151857,151857,151857,151857,151749,151837,151857,151857,151857,151408,151408,151857,151857,151857,151857
Giratina,142845,142845,142845,142845,142833,142845,142845,142845,142845,142799,142799,142845,142845,142845,142845
Jirachi,118906,118906,118906,118906,118897,118906,118906,118906,118906,118574,118574,118906,118906,118906,118906
Sableye,90453,90453,90453,90453,90453,90453,90453,90453,90453,66,66,90453,90453,90453,90453
Togepi,126866,126866,126866,126866,126855,126866,126866,126866,126866,126659,126659,126866,126866,126866,126866
Torchic,138643,138643,138643,138643,138626,138643,138643,138643,138643,138482,138482,138643,138643,138643,138643
Vulpix,133937,133937,133937,133937,133930,133937,133937,133937,133937,133897,133897,133937,133937,133937,133937
Whismur,95240,95240,95240,95240,95233,95240,95240,95240,95240,95094,95094,95240,95240,95240,95240


In [5]:
hsb_6 = pd.read_csv(r"L:\4portProb_structured\HSB_F_006.csv")
hsb_7 = pd.read_csv(r"L:\4portProb_structured\HSB_F_007.csv")
hsb_8 = pd.read_csv(r"L:\4portProb_structured\HSB_F_008.csv")
hsb_9 = pd.read_csv(r"L:\4portProb_structured\HSB_F_009.csv")
hsb_10 = pd.read_csv(r"L:\4portProb_structured\HSB_F_010.csv")
hsb_11 = pd.read_csv(r"L:\4portProb_structured\HSB_F_011.csv")
hsb_12 = pd.read_csv(r"L:\4portProb_structured\HSB_F_012.csv")
hsb_13 = pd.read_csv(r"L:\4portProb_structured\HSB_F_013.csv")
hsb_14 = pd.read_csv(r"L:\4portProb_structured\HSB_F_014.csv")

hsb_6.animal = 'HSB_F_006'
hsb_7.animal = 'HSB_F_007'
hsb_8.animal = 'HSB_F_008'
hsb_9.animal = 'HSB_F_009'
hsb_10.animal = 'HSB_F_010'
hsb_11.animal = 'HSB_F_011'
hsb_12.animal = 'HSB_F_012'
hsb_13.animal = 'HSB_F_013'
hsb_14.animal = 'HSB_F_014'

hsb_data = pd.concat([hsb_6, hsb_7, hsb_8, hsb_9, hsb_10, hsb_11, hsb_12, hsb_13, hsb_14]).reset_index()
hsb_data['trial'] = np.arange(hsb_data.shape[0])
hsb_data['env'] = hsb_data['task']
hsb_data['task'] = 'str'
hsb_data['datetime'] = pd.to_datetime(hsb_data.eptime, unit = 's')
from utils.dfLoading import add_block_groups
hsb_data = hsb_data.rename(columns = {'session':'block_num'})#.drop(columns = "Unnamed: 0")
hsb_data = add_block_groups(hsb_data)
hsb_data = hsb_data.rename(columns = {'sess_block': 'session'})#.drop(columns = "Unnamed: 0")
# ## week relative to each animal-task first date
hsb_data['week_task'] = hsb_data.groupby(['animal', 'task'])['datetime'].transform(
    lambda x: ((x - x.min()).dt.days // 7) + 1
)
hsb_data['day_task'] = hsb_data.groupby('animal')['datetime'].transform(
    lambda x: ((x - x.min()).dt.days) + 1
)
from utils.trializer_v4 import apply_task_rules_from_json
rules_path = r"C:\Users\dlab\banditAnalysis\utils\task_data.json"
hsb_data = apply_task_rules_from_json(hsb_data, rules_path=rules_path)
window = 5
hsb_data['entropy'] = (hsb_data.groupby(['animal', 'session'], as_index = False)
                     .port
                     .rolling(window, center=True)
                     .apply(lambda x: entropy(calc_prob(x), base = 2))
                     .port)
from utils.dfLoading import remove_sessions_with_few_duplicate_trials
hsb_data = remove_sessions_with_few_duplicate_trials(hsb_data)
# logic - if 46 next to 80 then structured, but also how visibly smooth it is

removed ('HSB_F_006', 17, 12)
removed ('HSB_F_006', 18, 68)
removed ('HSB_F_010', 84, 960)
removed ('HSB_F_010', 85, 168)
removed ('HSB_F_010', 87, 60)
removed ('HSB_F_010', 89, 144)
removed ('HSB_F_010', 90, 1392)
removed ('HSB_F_010', 91, 840)
removed ('HSB_F_010', 92, 2420)
removed ('HSB_F_010', 93, 1564)
removed ('HSB_F_010', 94, 1604)
removed ('HSB_F_010', 122, 4320)
removed ('HSB_F_010', 128, 2348)
removed ('HSB_F_010', 250, 1372)
removed ('HSB_F_011', 3, 1180)
removed ('HSB_F_011', 17, 2680)
removed ('HSB_F_011', 18, 2456)
removed ('HSB_F_011', 19, 1708)
removed ('HSB_F_011', 21, 828)
removed ('HSB_F_011', 22, 516)
removed ('HSB_F_011', 23, 300)
removed ('HSB_F_011', 24, 880)
removed ('HSB_F_011', 25, 1352)
removed ('HSB_F_011', 27, 692)
removed ('HSB_F_011', 28, 3180)
removed ('HSB_F_011', 29, 3596)
removed ('HSB_F_011', 35, 4048)
removed ('HSB_F_011', 37, 3720)
removed ('HSB_F_011', 39, 640)
removed ('HSB_F_011', 40, 3716)
removed ('HSB_F_011', 41, 3884)
removed ('HSB_F_011', 

In [54]:
block_an_dict = {'unblocked': ['test05022023', 'Blissey', 'Chikorita', 'Darkrai', 'Eevee',
       'Goldeen', 'Hoppip', 'Inkay', 'Jirachi', 'Kirlia', 'Mesprit',
       'Nidorina', 'Oddish', 'Quilava', 'Raltz', 'Shinx',
       'Togepi', 'Umbreon', 'Vulpix','Wartortle','Zacian',
       'Alakazam', 'Bayleef', 'Cresselia', 'Haxorus'],
       'cued':['Emolga', 'Giratina',
              #   'Haxorus',
                 'Ivysaur',
              'Jigglypuff', 'Lugia', 'Ninetales', 'Onix'],
             # 'Pichu'],
       'uncued' : ['Quaxly','Sableye','Torchic', 'Uxie', 'Vanillish',
              'Whismur', 'Yamper', 'Zorua', 'Azelf', 'Bidoof']}

In [4]:
an_dict = {
    'unstr'  : ['test05022023', 'Blissey', 'Chikorita', 'Darkrai', 'Eevee', 
                'Goldeen', 'Hoppip', 'Inkay', 'Jirachi', 'Kirlia', 'Mesprit',
                'Nidorina', 'Oddish', 'Quilava', 'Raltz', 'Shinx',
                'Togepi', 'Umbreon', 'Vulpix',  'Wartortle', 'Zacian',
                 'Bayleef', 'Cresselia', 'Emolga', 'Giratina',
                'Haxorus', 'Ivysaur', 'Jigglypuff', 'Lugia', 'Ninetales', 'Onix',
                'Pichu', 'Quaxly', 'Sableye', 'Torchic', 'Uxie', 'Vanillish',
                'Whismur', 'Xerneas', 'Yamper', 'Zorua', 'Azelf', 'Bidoof'],
    'dls'    : ['Blissey', 'Darkrai', 'Inkay' , 'Mesprit', 'Raltz', 'Zacian', 'Alakazam', 'Quilava', 'Torchic', 'Uxie', 'Vanillish', 'Yamper', 'Zorua'],
    'dms'    : ['Hoppip', 'Kirlia', 'Nidorina', 'Togepi', 'Xatu', 'Yanma', 'Giratina', 'Ivysaur', 'Lugia', 'Whismur', 'Quaxly'],
    'sham'   : ['test05022023', 'Jirachi', 'Goldeen', 'Phione', 'Umbreon', 'Vulpix', 'Emolga', 'Haxorus', 'Jigglypuff'],
    'ds_2stage' : ['Chikorita', 'Eevee'],
    'ds_4stage': ['Hoppip', 'Blissey'],
    'ds'     : ['Chikorita', 'Eevee', 'Hoppip', 'Blissey'],
    'str'    : ['test05022023', 'Blissey', 'Chikorita', 'Darkrai', 'Eevee'],
    'hsb_str' : ['HSB_F_006', 'HSB_F_007', 'HSB_F_008', 'HSB_F_009', 'HSB_F_010', 'HSB_F_011', 'HSB_F_012', 'HSB_F_013', 'HSB_F_014'],
    'oe_drive' : ['Shinx', 'Dratini', 'Bayleef', 'Onix', 'Pichu', 'Xerneas', 'Bidoof', 'Dwebble']
}

In [56]:
from utils.dfLoading import add_block_groups
sessdf = add_block_groups(sessdf)
sessdf.columns

Index(['animal', 'task', 'block_num', 'trial', 'port', 'reward',
       'trial_rewprob', 'rewprobfull', 'poke_in_time', 'poke_out_time',
       'first_lick_time', 'last_lick_time', 'eptime', 'datetime', 'session',
       'rr', 'entropy', 'week_task', 'sess_bin', 'assigned', 'env',
       'sess_block', 'block_group'],
      dtype='object')

In [74]:
# save all data as .mat files for every animal
# first folder - all data
# from utils.dfLoading import add_block_groups
def add_block_groups(df):
    # make a column called sess_block
    # if session changes within 40min,  number +1
    # first reset index or everything will go crazy
    # df = df.reset_index()
    df['sess_block'] = 0
    thing = df.groupby(['animal', 'task', 'session']).datetime.head(1).astype('datetime64[ns]').diff().dt.seconds>2400
    df.loc[thing[thing == True].index, 'sess_block'] = 1
    df['sess_block'] = df.groupby(['animal', 'task']).sess_block.cumsum()
    # Create a group identifier that increases every time sess_block == 1
    df['block_group'] = df.groupby(['animal', 'task', 'session']).ngroup()

    df['block_group'] = (
        df.groupby(['animal', 'task', 'sess_block'])['session']
        .transform(lambda x: pd.factorize(x)[0] + 1)
    )
    return df
# hsb_data['datetime'] = pd.to_datetime(hsb_data.eptime, unit = 's')
# blocked_df = add_block_groups(hsb_data)
# blocked_df = blocked_df.rename(columns = {'session':'block', 'sess_block': 'session'})#.drop(columns = "Unnamed: 0")
# blocked_df['datetime'] = pd.to_datetime(blocked_df['datetime'])

# ## week relative to each animal-task first date
# blocked_df['week_task'] = blocked_df.groupby(['animal', 'task'])['datetime'].transform(
#     lambda x: ((x - x.min()).dt.days // 7) + 1
# )
cond = 'unstr'
blocked_df = sessdf
# select week 1 rows per animal-task   #(blocked_df['week_task'] == 8)& 
week1_per_animal_task = blocked_df[(blocked_df.task == cond)  & (blocked_df['week_task'] == 8
                                                                 )& (blocked_df.animal.isin(an_dict[cond]))]


In [75]:
l = []
week1_choices = {}
week1_rewards = {}
week1_block_num = {}
week1_rewprob = {}
week1 = []
cols = ['animal', 'session', 'port', 'reward', 'week_task', 'block_group', 'rewprobfull']
for (animal, group) in week1_per_animal_task[cols].groupby('animal'):
    l.append([group.session.nunique(), group.session.value_counts().max()])
    print(l[-1])
    choices = np.ones(shape = l[-1])*np.nan
    rewards = np.ones(shape = l[-1])*np.nan
    block_num = np.ones(shape = l[-1])*np.nan
    rewprob = np.ones(shape = (group.session.nunique(), group.session.value_counts().max(), 4))*np.nan

    for idx, (sess, sess_group) in enumerate(group.groupby('session')):
        if (animal == 'Umbreon'):
            if (sess == 362):
                continue
        l_sess = sess_group.port.to_numpy().shape[0]
        
        choices[idx, :l_sess] = sess_group.port.to_numpy()
        rewards[idx, :l_sess] = sess_group.reward.to_numpy()
        block_num[idx, :l_sess] = sess_group.block_group.to_numpy()
        rewprob[idx, :l_sess, :] = np.array([i.strip('[]').split(' ') for i in sess_group['rewprobfull']], dtype = int)
    week1_choices[animal] = choices
    week1_rewards[animal] = rewards
    week1_block_num[animal] = block_num
    week1_rewprob[animal] = rewprob
    print(animal)

    week1.append({'choices':week1_choices[animal], 'rewards':week1_rewards[animal], 'block_num':week1_block_num[animal], 'rewprob':week1_rewprob[animal], 'animal':animal})


[16, 1236]
Bayleef
[6, 592]
Chikorita
[17, 1109]
Cresselia
[7, 386]
Darkrai
[13, 498]
Eevee
[21, 1307]
Emolga
[21, 1087]
Giratina
[13, 532]
Goldeen
[22, 1092]
Haxorus
[22, 1342]
Hoppip
[9, 1049]
Inkay
[21, 893]
Ivysaur
[22, 1121]
Jigglypuff
[22, 534]
Jirachi
[22, 1514]
Kirlia
[17, 846]
Lugia
[9, 701]
Nidorina
[14, 978]
Oddish
[5, 1586]
Pichu
[9, 1144]
Quaxly
[22, 1094]
Quilava
[21, 709]
Raltz
[22, 811]
Sableye
[18, 359]
Shinx
[20, 1063]
Togepi
[22, 1073]
Torchic
[20, 764]
Umbreon
[12, 1088]
Uxie
[22, 1103]
Vanillish
[6, 499]
Vulpix
[12, 775]
Wartortle
[22, 1001]
Whismur
[5, 810]
Yamper
[20, 1412]
Zacian
[22, 918]
Zorua


In [76]:
from scipy.io import savemat

savemat(r"C:\Users\dlab\OneDrive - Indian Institute of Science\Data\week8_data_20260925_arr.mat", {'week8':week1}, appendmat=True)


# Initial bias

In [ ]:
row_num = 100
cond = 'dls'
nth_port = (sessdf[(sessdf.task == 'unstr') & (sessdf.block_group == 1) & (sessdf.sess_bin>=4)]
                               .groupby(['animal', 'session'])
                               .filter(lambda x: x.reward.size >= 100)
                               .groupby(['animal', 'session'])
                               .nth(row_num))
# sns.displot(data = first_10_ports, x = 'port', col = 'animal', discrete = True,
#             common_norm = False, col_wrap = 7, stat = 'probability', color = 'xkcd:cornflower')
init_port_prop = nth_port.groupby('animal').port.value_counts(normalize = True, sort = True).reset_index()
init_port_prop['highness'] = init_port_prop.groupby('animal').cumcount()+1
init_port_prop = init_port_prop.drop(columns = 'port')
pref_mat = init_port_prop.set_index(['animal', 'highness']).unstack()

In [ ]:
plt.figure()
ax = plt.subplot(121)
ax.bar(np.arange(1,5),nth_port.port.value_counts(normalize=True), color = 'xkcd:cornflower')
ax.set_xticks(np.arange(1,5), np.arange(1,5), color = 'grey', fontsize = 'large')
ax.set_xlabel('Average choice', fontsize = 'x-large')
# ax.set_yticks(color = 'grey', fontsize = 'large')
ax.set_ylabel(f'Probability on trial {row_num}', fontsize = 'x-large')
sns.despine()
plt.tight_layout()
ax = plt.subplot(122)
ax.bar(np.arange(1,5),np.mean(pref_mat, axis = 0), color = 'xkcd:cornflower')
ax.set_xticks(np.arange(1,5), ['High', '', '', 'Low'],color = 'grey', fontsize = 'large')
ax.set_xlabel('Preferred choice', fontsize = 'x-large')
# ax.set_yticks(color = 'grey', fontsize = 'large')
ax.set_ylabel(f'Probability on trial {row_num}', fontsize = 'x-large')
sns.despine()
plt.tight_layout()

In [ ]:
# comparison pairs
cond_l = ["dls", "dms", 'sham']
for cond in cond_l:
    # calc_nth port
    print(cond)

# number of trials

In [125]:
# median trials per session distribution
from scipy.stats import sem
subset_df = hsb_data
mask = (subset_df.task == 'str') & subset_df.animal.isin(an_dict['hsb_str']) & (subset_df.day_task>30)
subset_df = subset_df[mask]
plt.figure()
meds = subset_df.groupby(['animal', 'session'], as_index = False)['trial'].count().groupby('animal').trial.median()#.plot(kind = 'hist', bins = np.arange(0, 1500, 50))
errs = subset_df.groupby(['animal', 'session'], as_index = False)['trial'].count().groupby('animal').trial.sem()
# colors = ['xkcd:peach','xkcd:peach','xkcd:sage red','xkcd:sage red','xkcd:sage red','xkcd:sage red','xkcd:sage red','xkcd:peach','xkcd:peach','xkcd:sage red','xkcd:peach','xkcd:sage red','xkcd:scarlet','xkcd:pumpkin orange','xkcd:pumpkin orange','xkcd:sage red','xkcd:scarlet','xkcd:peach','xkcd:scarlet','xkcd:sage red','xkcd:sage red','xkcd:sage red','xkcd:peach',
# 'xkcd:scarlet','xkcd:scarlet','xkcd:sage red','xkcd:sage red','xkcd:peach','xkcd:pumpkin orange','xkcd:sage red','xkcd:pumpkin orange','xkcd:sage red','xkcd:peach','xkcd:sage red',
# 'xkcd:pumpkin orange','xkcd:sage red',]

# colors = ['xkcd:sage green', 'xkcd:sage green', 'xkcd:emerald green', 'xkcd:emerald green', 'xkcd:sage green', 'xkcd:emerald green', 'xkcd:sage green', 'xkcd:emerald green', 'xkcd:emerald green',]
plt.hist(meds, bins = np.arange(0, 1500, 50))
plt.ylabel('Subject count')
plt.xlabel('Median session length')
plt.title('Bin = 50 trials')

# plt.bar(np.arange(meds.shape[0]), meds, yerr = errs, color = colors)
# plt.xticks(ticks = np.arange(meds.shape[0]), labels = meds.index, rotation = 90)
# plt.axhline(500, linestyle = '--', linewidth = 0.75, color = 'r')
plt.grid(visible = True)
plt.tight_layout()

In [129]:
# groupby cond and animal
from utils.plotSettings import *
countdf = sessdf.groupby(['animal', 'task'],
                         as_index = False)['session'].value_counts().rename(columns = {'count':'sesscount'})

cond = 'sham'
an_list = sessdf[sessdf.task == cond].animal.sort_values().unique()

he = countdf[countdf.task.isin([cond, 'unstr'])& 
             (countdf.animal.isin(an_list))].groupby(['animal', 'task'], as_index = False).sesscount.median()

y = np.vstack([he[he.task == 'unstr'].sesscount.values, he[he.task == cond].sesscount.values]).T

# plot now!
fig = plt.figure(figsize = (3.5, 4.5))
ax = plt.subplot(111)
ax.bar([0, 1], [np.mean(y[:, 0]), np.mean(y[:, 1])], width = 0.4, color = ['xkcd:cornflower', 'xkcd:slate grey'], alpha = 0.5)
ax.plot([0]*len(y[:, 0]), y[:, 0], 'o', color = 'xkcd:cornflower', alpha = 0.7)
ax.plot([1]*len(y[:, 1]), y[:, 1], 'o', color = 'xkcd:slate grey', alpha = 0.7)

# for lines between points
for i in range(y.shape[0]):
    ax.plot(y[i, :], color = 'xkcd:slate grey', linewidth = 0.7, alpha = 0.3)

# sns.pointplot(data = countdf[countdf.task.isin([cond, 'unstr'])& (countdf.animal.isin(an_list))],
#                 order = ['unstr', cond], hue_order =['unstr', cond], 
#               y = 'sesscount', x = 'task', hue = 'task', errorbar = None, estimator = 'mean', 
#             palette = ['xkcd:cornflower', 'xkcd:slate grey'], dodge = False, ax = ax)


# formatting
ax.set_xticks([0, 1], ["Pre-lesion","Post-lesion"], fontsize = 'large', color = 'grey')
ax.tick_params(axis='both', colors='grey')
# ax.set_ylabel()
# ax.legend(False)
ax.set_ylabel('Number of trials in session', fontsize = 'x-large')
ax.set_xlabel('Condition', fontsize = 'x-large')
ax.set_xlim(-0.3, 1.3)


from scipy.stats import ttest_rel
ttest_rel(y[:, 0], y[:, 1])

annot = True
if annot == True:
  # Perform t-tests
  t_stat1, p_val1 = ttest_rel(y[:, 0], y[:, 1])
  # Add annotations
  max_y = max(np.concatenate([y[:, 0], y[:, 1]]))+20

  if p_val1:
      plt.plot([0, 1], [max_y + 0.02, max_y + 0.02], color='black', linewidth=1)
      plt.text(0.5, max_y + 20, f'p={p_val1:.3f}', ha='center', fontsize=10)

sns.despine()
plt.tight_layout()

# Transition matrices

In [57]:
from utils.dfLoading import add_shift_info
sessdf = add_shift_info(sessdf)
hsb_data = add_shift_info(hsb_data)

c:\Users\dlab\banditAnalysis\utils\dfLoading.py:177: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['shift_t1'] = (df['choice_t2']==df['port']).replace({True: 0, False: 1})
c:\Users\dlab\banditAnalysis\utils\dfLoading.py:178: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df
c:\Users\dlab\banditAnalysis\utils\dfLoading.py:177: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to th

In [8]:
# full tm - all trials 
%matplotlib qt
ind = 1
trialsinsess = 100

cond = 'unstr'
# an_list = an_dict[cond]
an_list = block_an_dict['cued']+block_an_dict['uncued']
from utils.plotSettings import *
parula = get_parula_cmap()
cmap = parula
mat = np.ones((len(an_list), 4, 4))*np.nan
# fig = plt.figure(figsize = (15, 10))
tempdf = sessdf.groupby(['animal','block_num']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','block_num']).head(trialsinsess)
mask = (tempdf.task==cond) & tempdf.animal.isin(an_list) & (tempdf.sess_bin>=4)
for animal, group in tempdf[mask].groupby('animal'):

    # ax = plt.subplot(6, 8, ind)
    # sns.heatmap(pd.crosstab(group['port'],
    #                         group['choice_t1'],
    #                         normalize = 'index'),
    #                         # dropna = False),
    #             ax = ax, cmap = cmap,
    #             annot = True, fmt = '.2f',square = True, vmax = 0.17,
    #             xticklabels = np.arange(1,5), yticklabels = np.arange(1,5),
    #             # mask = np.eye(4, dtype = bool),
    #             vmin = 0)
    # ax.set_title(animal)
    
    if (animal == 'Yamper') & (group.task.unique()[0] == 'dls'):
        mat_y = np.vstack([pd.crosstab(group['port'],
                            group['choice_t1'],
                            normalize = 'index').to_numpy(), np.zeros((1, 3))*np.nan])
        mat_y = np.hstack([mat_y, np.zeros((4, 1))*np.nan])
        mat[ind-1] = mat_y
    else:
        mat[ind-1] = pd.crosstab(group['port'],
                            group['choice_t1'],
                            normalize = 'index')
    
    ind+=1

plt.tight_layout()
plt.figure()
ax = plt.subplot(111)
sns.heatmap(np.nanmedian(mat, axis = 0), annot = False, fmt = '.2f', cmap = parula, xticklabels = np.arange(1,5), square = True,
            yticklabels = np.arange(1,5), vmin = 0, vmax = 0.2, cbar_kws={'label': 'Transition probability'},
            # mask = np.eye(4),
            ax = ax)
# ax.patch.set_facecolor('white')
ax.set_xlabel('Choice at trial t+1', fontsize = 'x-large')
ax.set_ylabel('Choice at trial t', fontsize = 'x-large')
ax.tick_params(axis = u'both', which = u'both', length = 0, labelsize = 'large', color = 'grey')
plt.title(f'Choice at switches - Animal data, n = {mat.shape[0]}')

# plt.suptitle('Animal data', fontsize = 16)    
plt.tight_layout()
# plt.savefig('C:/Users/dlab/rishika_sim/plots/20231210/tm_all_noswitch.png', dpi = 300)

In [7]:
tempdf = hsb_data.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
ind = 1
# plt.figure()
cond = 'hsb_str'
an_list = an_dict[cond]
mat = np.ones((len(an_list), 4, 4))*np.nan

for animal, group in tempdf.groupby('animal'):

    # ax = plt.subplot(6, 8, ind)
    # sns.heatmap(pd.crosstab(group['port'],
    #                         group['choice_t1'],
    #                         normalize = 'index'),
    #                         # dropna = False),
    #             ax = ax, cmap = cmap,
    #             annot = True, fmt = '.2f',square = True, vmax = 0.17,
    #             xticklabels = np.arange(1,5), yticklabels = np.arange(1,5),
    #             # mask = np.eye(4, dtype = bool),
    #             vmin = 0)
    mat[ind-1] = pd.crosstab(group['port'],
                            group['choice_t1'],
                            normalize = 'index')
    
    ind+=1
    # ax.set_title(animal)
plt.tight_layout()
plt.figure()
ax = plt.subplot(111)
sns.heatmap(np.nanmedian(mat, axis = 0), annot = False, fmt = '.2f', cmap = parula, xticklabels = np.arange(1,5), square = True,
            yticklabels = np.arange(1,5), vmin = 0, vmax = 0.2, cbar_kws={'label': 'Transition probability'},
            # mask = np.eye(4),
            ax = ax)
# ax.patch.set_facecolor('white')
ax.set_xlabel('Choice at trial t+1', fontsize = 'x-large')
ax.set_ylabel('Choice at trial t', fontsize = 'x-large')
ax.tick_params(axis = u'both', which = u'both', length = 0, labelsize = 'large', color = 'grey')
plt.title(f'Choice at switches - Animal data, n = {mat.shape[0]}')

# plt.suptitle('Animal data', fontsize = 16)    
plt.tight_layout()

In [ ]:
# full tm - all trials 
%matplotlib qt
ind = 1
trialsinsess = 100

cond = 'unstr'
an_list = an_dict[cond]
# an_list = block_an_dict['cued']+block_an_dict['uncued']
from utils.plotSettings import *
parula = get_parula_cmap()
cmap = parula
mat = np.ones((len(an_list), 4, 4))*np.nan
fig = plt.figure(figsize = (15, 10))
tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
mask = (tempdf.task==cond) & tempdf.animal.isin(an_list) & (tempdf.sess_bin>=3)
for animal, group in tempdf[mask].groupby('animal'):

    ax = plt.subplot(6, 8, ind)
    mask0 = (group.reward == 0)
    mask1 = (group.reward == 1)
    r_mat = pd.crosstab(group.loc[mask1, 'port'], group.loc[mask1, 'choice_t1'], normalize = 'index')
    nr_mat = pd.crosstab(group.loc[mask0, 'port'], group.loc[mask0, 'choice_t1'], normalize = 'index')
    sns.heatmap(r_mat - nr_mat,
                ax = ax, cmap = 'seismic',
                annot = True, fmt = '.2f',square = True, center = 0,
                xticklabels = np.arange(1,5), yticklabels = np.arange(1,5),
                # mask = np.eye(4, dtype = bool),
                vmin = 0)
    ax.set_title(animal)
    
    if (animal == 'Yamper') & (group.task.unique()[0] == 'dls'):
        mat_y = np.vstack([pd.crosstab(group['port'],
                            group['choice_t1'],
                            normalize = 'index').to_numpy(), np.zeros((1, 3))*np.nan])
        mat_y = np.hstack([mat_y, np.zeros((4, 1))*np.nan])
        mat[ind-1] = mat_y
    else:
        mat[ind-1] = r_mat  - nr_mat
    
    ind+=1

plt.tight_layout()
plt.figure()
ax = plt.subplot(111)
sns.heatmap(np.nanmedian(mat, axis = 0), annot = False, fmt = '.2f', cmap = 'seismic', xticklabels = np.arange(1,5), square = True,
            yticklabels = np.arange(1,5), center=0, cbar_kws={'label': 'Transition probability'},
            # mask = np.eye(4),
            ax = ax)
# ax.patch.set_facecolor('white')
ax.set_xlabel('Choice at trial t+1', fontsize = 'x-large')
ax.set_ylabel('Choice at trial t', fontsize = 'x-large')
ax.tick_params(axis = u'both', which = u'both', length = 0, labelsize = 'large', color = 'grey')
plt.title(f'Choice at switches - Animal data, n = {mat.shape[0]}')

# plt.suptitle('Animal data', fontsize = 16)    
plt.tight_layout()
# plt.savefig('C:/Users/dlab/rishika_sim/plots/20231210/tm_all_noswitch.png', dpi = 300)

# Entropy


## Overall entropy per rat per task

In [ ]:
#################################### rr ######################################
# set_cwd('/home/rishika/sim/')
%matplotlib qt
# %matplotlib inline
window = 7
trialsinsess = 100

from utils.dfLoading import subset_trials, add_shift_info, add_block_groups
subset_df = add_block_groups(sessdf)
subset_df = subset_df[(subset_df.sess_bin>= 4)& (subset_df.block_group == 1) & (subset_df.animal.isin(block_an_dict['cued']+block_an_dict['uncued']))]
# subset_df = add_shift_info(df)
subset_df = subset_trials(subset_df, trialsinsess=trialsinsess, head_trials=trialsinsess)

hsb_data = hsb_data.sort_values(['animal', 'session', 'eptime'])
subset_df_hsb = add_block_groups(hsb_data)
subset_df_hsb = subset_df_hsb[(subset_df_hsb.block_group == 1)]
subset_df_hsb = subset_trials(subset_df_hsb, trialsinsess=trialsinsess, head_trials = trialsinsess)

def process_data_blockwise(sessdf, animal, task='unstr'):
    subset_df = sessdf[sessdf.animal == animal]
    num_sess = subset_df[subset_df.task == task].sess_block.unique().shape[0]
    num_t = subset_df[subset_df.task == task].groupby('sess_block').trial.count().max()
    actions = np.ones(shape = (num_sess, num_t))*np.nan
    rewards = np.ones(shape = (num_sess, num_t))*np.nan
    block_id = np.ones(shape = (num_sess, num_t))*np.nan

    for ind, (_, group) in enumerate(subset_df[subset_df.task == task].groupby('sess_block')):
        actions[ind] = np.concatenate([group.port, np.ones(shape = (num_t - group.port.shape[0]))*np.nan])
        rewards[ind] = np.concatenate([group.reward, np.ones(shape = (num_t - group.reward.shape[0]))*np.nan])
        block_id[ind] = np.concatenate([group.block_group, np.ones(shape = (num_t - group.block_group.shape[0]))*np.nan])
    print('processed blockwise')
    return np.array([actions, rewards, block_id])

# # for getting entropy trialwise
# from utils.dfAnalysisHelpers import subset_metric_to_numpy
# bias_dict = {}
# for task in ['unstr', 'str', 'dms', 'dls', 'sham']:
#     for ind, (an, group) in enumerate(subset_df[subset_df.task == task].groupby('animal')):
#         L = subset_metric_to_numpy(group, 'port')
#         bias_dict[an, task] = [entropy(calc_prob(L[:, col]), base = 2) for col in range(L.shape[1])]
# block_id =1 

# for overall - should be easier if data is processed blockwise first.
def calc_bias_per_sess(actions, mask = (block_id == 1)):
    selected_ind = np.where(mask, actions, np.nan)
    return np.array([entropy(calc_prob(row[~np.isnan(row)]), base = 2) for row in selected_ind if row[~np.isnan(row)].shape[0]!=0])


bias_dict_unstr = {}
bias_dict_str_in_unstr = {}
# for task in ['unstr', 'str']:
for ind, an in enumerate(subset_df[subset_df.task == 'unstr'].animal.unique()):
    actions, _, block_id = process_data_blockwise(subset_df[subset_df.env == 'unstr'], an, 'unstr')
    bias_dict_unstr[an] = calc_bias_per_sess(actions, block_id == 1)
    if subset_df[(subset_df.env == 'str') & (subset_df.animal == an)].shape[0]>0:
        actions, _, block_id = process_data_blockwise(subset_df[subset_df.env == 'str'], an, 'unstr')
        bias_dict_str_in_unstr[an] = calc_bias_per_sess(actions, block_id == 1)

bias_dict_str = {}
bias_dict_unstr_in_str = {}
bias_dict_str_all = {}
for an in subset_df_hsb[subset_df_hsb.task == 'str'].animal.unique():
    actions, _, block_id = process_data_blockwise(subset_df_hsb, an, 'str')
    bias_dict_str_all[an] = calc_bias_per_sess(actions, block_id == 1)
    actions, _, block_id = process_data_blockwise(subset_df_hsb[subset_df_hsb.env == 'str'], an, 'str')
    bias_dict_str[an] = calc_bias_per_sess(actions, block_id == 1)
    actions, _, block_id = process_data_blockwise(subset_df_hsb[subset_df_hsb.env == 'unstr'], an, 'str')
    bias_dict_unstr_in_str[an] = calc_bias_per_sess(actions, block_id==1)

processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise
processed blockwise


In [ ]:
temp_ = []
n = 0
# for bias_dict in [bias_dict_unstr, bias_dict_unstr_in_str]:
for an in bias_dict_str_all.keys():
    plt.bar(n, np.mean(bias_dict_str_all[an]))
        # temp_.append(np.mean(bias_dict[an]))
    
    # plt.bar(x = n, height = np.mean(temp_), yerr = sem(temp_))
    n+=1

NameError: name 'bias_dict_str_all' is not defined

## Sliding window entropy/Switch rate

In [14]:
#################################### moving entropy ######################################
%matplotlib qt
# %matplotlib inline
trialsinsess = 100

from utils.dfLoading import subset_trials, add_shift_info, add_block_groups
subset_df = add_block_groups(sessdf)
subset_df = subset_df[(subset_df.day_task>30)& (subset_df.animal.isin(block_an_dict['uncued']+block_an_dict['cued']))]
subset_df = add_shift_info(subset_df)
# subset_df = add_shift_info(subset_df)

# subset_df_unbl = add_block_groups(sessdf)
# subset_df_unbl = subset_df_unbl[(subset_df_unbl.sess_bin>=4) & (subset_df_unbl.block_group == 2) & (subset_df_unbl.animal.isin(block_an_dict['uncued']+block_an_dict['cued']))]
# subset_df_unbl = add_shift_info(subset_df_unbl)
# subset_df_unbl = add_shift_info(subset_df_unbl)

# hsb_data_sorted = hsb_data.sort_values(['animal','eptime', 'session'])
subset_df_hsb = add_block_groups(hsb_data)
subset_df_hsb = subset_df_hsb[(subset_df_hsb.day_task>30)]
subset_df_hsb = add_shift_info(subset_df_hsb)

# window = 
from utils.supplementaryFunctions import calc_prob

window = 5
# subset_df['entropy'] = (subset_df.groupby(['animal', 'session'], as_index = False)
#                      .port
#                      .rolling(window, center=True)
#                      .apply(lambda x: entropy(calc_prob(x), base = 2))
#                      .port)
# subset_df_unbl['entropy'] = (subset_df_unbl.groupby(['animal', 'session'], as_index = False)
#                      .port
#                      .rolling(window, center=True)
#                      .apply(lambda x: entropy(calc_prob(x), base = 2))
#                      .port)
# subset_df_hsb['entropy'] = (subset_df_hsb.groupby(['animal', 'session'], as_index = False)
#                      .port
#                      .rolling(window, center=True)
#                      .apply(lambda x: entropy(calc_prob(x), base = 2))
#                      .port)

subset_df = subset_trials(subset_df, trialsinsess=trialsinsess, head_trials = trialsinsess)
subset_df_hsb = subset_trials(subset_df_hsb, trialsinsess=trialsinsess, head_trials = trialsinsess)
# subset_df_unbl = subset_trials(subset_df_unbl, trialsinsess=trialsinsess, head_trials = trialsinsess)


from utils.dfAnalysisHelpers import get_sessionwise_averages
entropy_mean_dict = get_sessionwise_averages(subset_df, metric= 'shift_t0', tasks = ['unstr', 'dms', 'dls'])
entropy_mean_dict_hsb = get_sessionwise_averages(subset_df_hsb, metric= 'shift_t0', tasks = ['unstr', 'str'])

# entropy_mean_dict_hsb_unstr = session_averager(subset_df_hsb[subset_df_hsb.env == 'unstr'], metric= 'entropy', tasks = ['str'])
# entropy_mean_dict_hsb_str = session_averager(subset_df_hsb[subset_df_hsb.env == 'str'], metric= 'entropy', tasks = ['str'])

# entropy_mean_dict_pure_unstr = get_sessionwise_averages(subset_df[subset_df.env == 'unstr'], metric= 'entropy', tasks = ['unstr'])
# entropy_mean_dict_str_in_unstr = get_sessionwise_averages(subset_df[subset_df.env == 'str'], metric= 'entropy', tasks = ['unstr'])

# entropy_mean_dict_unbl = get_sessionwise_averages(subset_df_unbl, metric = 'entropy', tasks = ['unstr'])


c:\Users\dlab\banditAnalysis\utils\dfLoading.py:171: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['choice_t1'] = df.groupby(['animal','session']).port.shift(-1)
c:\Users\dlab\banditAnalysis\utils\dfLoading.py:172: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['choice_t2'] = df.groupby(['animal','session']).port.shift(-2)
c:\Users\dlab\banditAnalysis\utils\dfLoading.py:175: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexe

In [18]:
from scipy.optimize import curve_fit
from scipy.stats import sem
from itertools import permutations
# plot average reponse
from utils.plotSettings import *
plt.figure(figsize = (5,4.5))
temp_unstr, temp_str = [], []

# set properties
smooth = False
include_hsb = False
window = 5
# label1, label2 = 'Pre-lesion', 'Post-lesion'
label1, label2 = 'Uncorrelated', 'Correlated'
# label1, label2 = 'Uncued', 'Cued'
# label1, label2 = 'Block 1', 'Block 2'
# label1, label2 = 'Blocked', 'Unblocked'
# color1, color2 =  sns.color_palette('crest')[0],  sns.color_palette('crest')[4]
color1, color2 = 'xkcd:barney purple', 'xkcd:emerald green'
# color1, color2 = 'xkcd:cornflower', 'xkcd:slate grey'

# color1, color2 = 'xkcd:olive green', 'xkcd:emerald green'
# color1, color2 = 'xkcd:pumpkin orange', 'xkcd:dark orange'
# # color1, color2 = '#7a2048', '#408ec6'


# set data for plotting
for (animal, env) in entropy_mean_dict.keys():
    if env == 'unstr':
        temp_unstr.append(entropy_mean_dict[(animal, env)])

for (animal, env) in entropy_mean_dict_hsb.keys():
     if env == 'str':
        temp_str.append(entropy_mean_dict_hsb[(animal, env)])


temp_unstr = np.array(temp_unstr)
# temp_unstr[:, 99] = np.nan
# temp_str = np.array(temp_str)
# temp_str[:, 99] = np.nan

# smoothing details
if smooth:           
    for i, row in enumerate(temp_unstr):
        temp_unstr[i] = np.convolve(row, np.ones(window)/window, 'same')
        try:
            temp_str[i] = np.convolve(temp_str[i], np.ones(window)/window, 'same')
        except IndexError:
            pass
    
    temp_unstr = np.array(temp_unstr)
    temp_str = np.array(temp_str)
    # 
    # works only for odd windows
    temp_unstr[:, 0:(window//2)] = np.nan
    temp_unstr[:, (trialsinsess-window//2):trialsinsess] = np.nan
    
    temp_str[:, 0:(window//2)] = np.nan
    temp_str[:, (trialsinsess-window//2):trialsinsess] = np.nan

# take mean of smoothed data   
mean_unstr = (np.nanmean(np.array(temp_unstr), axis = 0))
mean_str = (np.nanmean(np.array(temp_str), axis = 0))

# plot line 1
plt.plot(np.arange(trialsinsess), mean_unstr, color = color1, label = label1)#, marker = '.') 
plt.fill_between(np.arange(trialsinsess), mean_unstr+sem(temp_unstr, nan_policy = 'omit'),
                mean_unstr-sem(temp_unstr, nan_policy = 'omit'), color = color1, alpha = 0.2)

# plot line 2 (or comment)
plt.plot(np.arange(trialsinsess), mean_str, color = color2, label = label2)
plt.fill_between(np.arange(trialsinsess), mean_str+sem(temp_str, nan_policy = 'omit'), 
                mean_str-sem(temp_str, nan_policy = 'omit'), color = color2, alpha = 0.2)

# plot significance
# from scipy.stats import ttest_ind
# pvals = ttest_ind(temp_str, temp_unstr)[1]
# pvals_ind = [ind for ind, i in enumerate(pvals) if i < 0.05]
# plt.scatter(pvals_ind, [0.45]*len(pvals_ind), marker = '.', color = 'r', alpha = 0.4) #(6, 2, 60) for asterisks

# chance calc
# plt.axhline(np.mean(np.unique(list(permutations([80, 46, 15, 10]))+list(permutations([46, 80, 46, 15])), axis = 0))/100,
#             color = color1, linestyle = '--')
# plt.axhline(np.mean([[80, 46, 15, 10], [46, 80, 46, 15], [15, 46, 80, 46], [10, 15, 46, 80]])/100, color = color2, linestyle = '--')

# plot settings
plt.xlabel('Trials in session', fontsize = 'x-large')
plt.ylabel('Entropy (bits)', fontsize = 'x-large')
plt.xlim(0, trialsinsess)
# plt.ylim(0.4, 0.72)
plt.xticks([0, 25, 50, 75, 100], [1, '', '', '', 100], color = 'grey', fontsize = 'large')
# plt.yticks([0, 0.2, 0.4, 0.6, 0.8, 1], [0, '', '', '', '', '1.0'], color = 'grey', fontsize = 'large')
# plt.title(f'n = {temp_unstr.shape[0]}')
plt.legend(loc = 'upper left')
leg = plt.legend(frameon=False)
for line, text in zip(leg.get_lines(), leg.get_texts()):
    text.set_color(line.get_color())
sns.despine()
# sns.set_context('talk')
plt.tight_layout()
# plt.savefig('C:/Users/dlab/OneDrive - Indian Institute of Science/Drawings/awp1/entropy_n_21.png', dpi = 300)


# if fitting exp curve is reqd
def logFunc(x, a, b):
    return a + b*np.log(x)

# popt_str, _ = curve_fit(logFunc, np.arange(150), temp_str, nan_policy = 'omit')
# popt_unstr, _ = curve_fit(logFunc, np.arange(trialsinsess), mean_unstr, nan_policy = 'omit')
# plt.plot(np.arange(150), logFunc(np.arange(150), *popt_str), '#408ec6', linestyle= 'dotted')
# plt.plot(np.arange(trialsinsess), logFunc(np.arange(trialsinsess), *popt_unstr), color1, linestyle= 'dotted')

In [11]:
# blockwise entropy/switch rate diff groups
from scipy.stats import sem
from utils.plotSettings import *

# (block_group, 'first'/'last', n_trials) in x-axis order
SEGMENTS = [(1, 'first', 100), (1, 'last', 25),
            (2, 'first', 100), (2, 'last', 25),
            (3, 'first', 25)]
seg_lens = [n for _, _, n in SEGMENTS]
seg_edges = np.cumsum([0] + seg_lens)
n_trials = seg_edges[-1]   # 275
ALLOW_OVERLAP = False      # False: a block needs >= first_n + last_n trials so first/last don't share trials

# set properties
smooth = False
window = 5                    # odd number so the window is centred
smooth_within_blocks = False  # True: don't smooth across segment boundaries

# minimum trials each block needs
need = {}
for bg, _, n in SEGMENTS:
    need[bg] = need.get(bg, 0) + n if not ALLOW_OVERLAP else max(need.get(bg, 0), n)

def session_matrix(df, sess_col='session'):
    """Returns {animal: n_sessions x n_trials reward matrix}; only sessions where every block is long enough."""
    out = {}
    df = df.sort_values('datetime')
    for animal, adf in df.groupby('animal'):
        rows = []
        for _, sdf in adf.groupby(sess_col):
            blocks = {bg: sdf.loc[sdf.block_group == bg, 'entropy'].to_numpy() for bg in need}
            if any(len(blocks[bg]) < need[bg] for bg in need):
                continue
            parts = [blocks[bg][:n] if which == 'first' else blocks[bg][-n:] for bg, which, n in SEGMENTS]
            rows.append(np.concatenate(parts))
        if rows:
            out[animal] = np.vstack(rows).astype(float)
    return out

unstr_mats = session_matrix(sessdf[(sessdf.day_task > 30) & (sessdf.task == 'unstr') & (sessdf.animal.isin(block_an_dict['cued']+block_an_dict['uncued']))], sess_col='session')
str_mats   = session_matrix(hsb_data[(hsb_data.day_task > 30) & (hsb_data.task == 'str')], sess_col='session')

def smooth_rows(mat, window, within_blocks=True):
    """Moving average; near segment edges, averages only over trials inside the segment (no NaNs)."""
    out = np.full_like(mat, np.nan)
    edges = seg_edges if within_blocks else [0, mat.shape[1]]
    k = np.ones(window)
    for s, e in zip(edges[:-1], edges[1:]):
        seg = mat[:, s:e]
        norm = np.convolve(np.ones(seg.shape[1]), k, 'same')   # trials actually inside the window
        out[:, s:e] = np.array([np.convolve(r, k, 'same') for r in seg]) / norm
    return out

# average over sessions (rows) per animal -> animals x n_trials
temp_unstr = np.array([m.mean(axis=0) for m in unstr_mats.values()])
temp_str   = np.array([m.mean(axis=0) for m in str_mats.values()])

if smooth:
    temp_unstr = smooth_rows(temp_unstr, window, smooth_within_blocks)
    temp_str = smooth_rows(temp_str, window, smooth_within_blocks)

mean_unstr = np.nanmean(temp_unstr, axis=0)
mean_str = np.nanmean(temp_str, axis=0)

# plot
plt.figure(figsize=(9, 4.5))
ax = plt.gca()
color1, color2 = 'xkcd:barney purple', 'xkcd:emerald green'
label1, label2 = 'Uncorrelated', 'Correlated'
x = np.arange(n_trials)

# split lines only at axis breaks (first 100 -> last 25), not at block switches
break_pos = [b for (_, which, _), b in zip(SEGMENTS[1:], seg_edges[1:-1]) if which == 'last']
switch_pos = [b for (_, which, _), b in zip(SEGMENTS[1:], seg_edges[1:-1]) if which == 'first']
runs = list(zip([0] + break_pos, break_pos + [n_trials]))

for i, (s, e) in enumerate(runs):
    for mean, temp, c, lab in [(mean_unstr, temp_unstr, color1, label1), (mean_str, temp_str, color2, label2)]:
        err = sem(temp[:, s:e], nan_policy='omit')
        plt.plot(x[s:e], mean[s:e], color=c, label=lab if i == 0 else None)
        plt.fill_between(x[s:e], mean[s:e] + err, mean[s:e] - err, color=c, alpha=0.2)

# block switches: dashed vertical line
for b in switch_pos:
    plt.axvline(b, color='grey', linestyle='--', linewidth=0.75)

# axis break where the middle of a block is skipped
gap = 2   # half-width of the break, in trials
trans = ax.get_xaxis_transform()   # x in data units, y in axes units
for b in break_pos:
    # patch that cuts the x-axis spine
    ax.add_patch(plt.Rectangle((b - gap, -0.02), 2 * gap, 0.04, transform=trans,
                               facecolor=plt.gcf().get_facecolor(), edgecolor='none',
                               clip_on=False, zorder=3))
    # the two diagonal break marks
    ax.plot([b - gap, b + gap], [0, 0], transform=trans, linestyle='none',
            marker=[(-1, -1.5), (1, 1.5)], markersize=8, markeredgewidth=1,
            color='k', clip_on=False, zorder=4)

plt.xlabel('Trials in session', fontsize='x-large')
plt.ylabel('Entropy (bits)', fontsize='x-large')
plt.xlim(0, n_trials)
plt.xticks(np.arange(0, n_trials + 1, 25), color='grey', fontsize='large')
leg = plt.legend(frameon=False, loc='upper left')
for line, text in zip(leg.get_lines(), leg.get_texts()):
    text.set_color(line.get_color())
plt.tight_layout()


C:\Users\dlab\AppData\Local\Temp\ipykernel_22024\3950800155.py:62: RuntimeWarning: Mean of empty slice
  mean_unstr = np.nanmean(temp_unstr, axis=0)
C:\Users\dlab\AppData\Local\Temp\ipykernel_22024\3950800155.py:63: RuntimeWarning: Mean of empty slice
  mean_str = np.nanmean(temp_str, axis=0)
C:\Users\dlab\AppData\Local\Temp\ipykernel_22024\3950800155.py:79: SmallSampleWarning: After omitting NaNs, one or more axis-slices of one or more sample arguments is too small; corresponding elements of returned arrays will be NaN. See documentation for sample size requirements.
  err = sem(temp[:, s:e], nan_policy='omit')


In [13]:
def block_counts(df, sess_col='session'):
    rows = []
    for (animal, sess), sdf in df.groupby(['animal', sess_col]):
        lens = sdf.groupby('block_group').size()
        if any(lens.get(bg, 0) < need[bg] for bg in need):
            continue   # session excluded from the plot
        for bg, n in sdf.groupby('block_group')['block_num'].nunique().items():
            rows.append((animal, sess, bg, n))
    out = pd.DataFrame(rows, columns=['animal', 'session', 'block_group', 'n_blocks'])
    return out.assign(group=np.where(out.block_group == 1, 'block_group 1', 'other'))

unstr_df = sessdf[(sessdf.day_task > 30) & (sessdf.task == 'unstr') &
                  (sessdf.animal.isin(block_an_dict['cued'] + block_an_dict['uncued']))]
str_df = hsb_data[(hsb_data.day_task > 30) & (hsb_data.task == 'str')]

for name, d in [('Uncorrelated (sessdf)', unstr_df), ('Correlated (hsb_data)', str_df)]:
    bc = block_counts(d)
    print(name)
    print(bc.groupby('group').n_blocks.sum(), '\n')
    print(bc.groupby('block_group').n_blocks.sum(), '\n')   # every block_group listed separately


Uncorrelated (sessdf)
group
block_group 1    1712
other            6635
Name: n_blocks, dtype: int64 

block_group
1     1712
2     1712
3     1712
4     1471
5      955
6      458
7      204
8       88
9       32
10       3
Name: n_blocks, dtype: int64 

Correlated (hsb_data)
group
block_group 1    1579
other            8139
Name: n_blocks, dtype: int64 

block_group
1     1579
2     1579
3     1579
4     1537
5     1395
6     1086
7      653
8      257
9       52
10       1
Name: n_blocks, dtype: int64 



## Entropy in block 2 and beyond

In [ ]:
# for all groups which have a second block, plot first block entropy for first 100 trials of session
# find if 2nd or greater block exists for animals
block_2_exists = tempdf[(tempdf.block_group>=2) & (tempdf.task == 'unstr')].groupby(['animal', 'sess_block']).size().reset_index()[['animal', 'sess_block']]

# remove animals with less than 10
block_2_exists = block_2_exists[block_2_exists.animal.isin(block_2_exists.animal.value_counts()[block_2_exists.animal.value_counts()>10].index)]

# find first blocks for these sessions
all_blocked_groups = tempdf[tempdf.task == 'unstr'].merge(block_2_exists, on = ['animal', 'sess_block'])
first_blocks = all_blocked_groups[all_blocked_groups.block_group == 1]
other_blocks = all_blocked_groups[all_blocked_groups.block_group > 1]

# convert to numpy array
entropy2_dict_block_1 = {}
for an, group in first_blocks.groupby('animal'):
        g = group.groupby('session').cumcount()
        L = (group.set_index(['session', g])
                .unstack(fill_value=0)
                .stack(future_stack= True).groupby(level=0)
                .apply(lambda x: x.entropy.values.tolist())
                .tolist())
        entropy2_dict_block_1[an] = np.mean(L, axis = 0)
entropy2_dict_block_others = {}
for an, group in other_blocks.groupby('animal'):
        g = group.groupby('session').cumcount()
        L = (group.set_index(['session', g])
                .unstack(fill_value=0)
                .stack(future_stack= True).groupby(level=0)
                .apply(lambda x: x.entropy.values.tolist())
                .tolist())
        entropy2_dict_block_others[an] = np.mean(L, axis = 0)


In [ ]:
plt.figure(figsize = (5, 4.5))
prev_sess_size = 25
y = np.zeros((len(entropy2_dict_block_1.keys()), 100+prev_sess_size))
for ind, an in enumerate(entropy2_dict_block_1.keys()):
    y[ind] = np.concatenate([entropy2_dict_block_1[an][-prev_sess_size:], entropy2_dict_block_others[an]])
    plt.plot(y[ind], color = 'xkcd:cornflower', alpha = 0.3, lw = 0.75)
plt.plot(np.nanmean(y, axis = 0), color = 'xkcd:cornflower', lw = 2.25)
plt.xlabel('Block position', fontsize = 'x-large')
plt.ylabel('Entropy (bits)', fontsize = 'x-large')
plt.ylim(0, 0.4)
plt.xticks(np.arange(0, 126, 25), np.arange(-25, 101, 25), color = 'grey', fontsize = 'large')
plt.axvline(25, lw = 0.75, color = 'k', linestyle = '--')
plt.yticks(color = 'grey', fontsize = 'large')
sns.despine()
plt.tight_layout()

C:\Users\dlab\AppData\Local\Temp\ipykernel_19448\2989309347.py:7: RuntimeWarning: Mean of empty slice
  plt.plot(np.nanmean(y, axis = 0), color = 'xkcd:cornflower', lw = 2.25)


# Regret

In [ ]:
#################################### regret ######################################
%matplotlib qt

trialsinsess = 10

from utils.dfLoading import subset_trials, add_shift_info, add_block_groups
subset_df = add_block_groups(sessdf)
subset_df = subset_df[(subset_df.block_group == 1) & (subset_df.animal.isin(block_an_dict['cued']+block_an_dict['uncued']))]
# subset_df = add_shift_info(df)
subset_df = subset_trials(subset_df, trialsinsess=trialsinsess, head_trials = trialsinsess)

hsb_data = hsb_data.sort_values(['animal', 'session', 'eptime'])
subset_df_hsb = add_block_groups(hsb_data)
subset_df_hsb = subset_df_hsb[(subset_df_hsb.block_group == 1)]
subset_df_hsb = subset_trials(subset_df_hsb, trialsinsess=trialsinsess, head_trials = trialsinsess)


from utils.supplementaryFunctions import calc_prob
window = 5
subset_df['regret'] = abs(subset_df['rewprob'] - 80)
subset_df_hsb['regret'] = abs(subset_df_hsb['rewprob'] - 80)


from utils.dfAnalysisHelpers import session_averager
reg_mean_dict = session_averager(subset_df, metric= 'regret', tasks = ['unstr', 'str', 'dms', 'dls', 'sham', 'ds'])
reg_mean_dict_hsb = session_averager(subset_df_hsb, metric= 'regret', tasks = ['unstr', 'str'])           

In [ ]:
from scipy.optimize import curve_fit
from scipy.stats import sem
from itertools import permutations
# plot average reponse
from utils.plotSettings import *
plt.figure(figsize = (5,4.5))
temp_unstr, temp_str = [], []

# set properties
smooth = False
include_hsb = True
window = 3
# label1, label2 = 'Pre-lesion', 'Post-lesion'
label1, label2 = 'Unstructured', 'Structured'
# color1, color2 = 'xkcd:cornflower', 'xkcd:slate grey'
color1, color2 = 'xkcd:pumpkin orange', 'xkcd:emerald green'
# # color1, color2 = '#7a2048', '#408ec6'

joint_rr_mean_dict = reg_mean_dict_hsb | reg_mean_dict
an_list = an_dict['hsb_str'] + an_dict['unstr']

# set data for plotting
for (animal, env) in reg_mean_dict.keys():
    if animal in an_list:
        if env == 'unstr':
            temp_unstr.append(reg_mean_dict[(animal, env)])
            print(animal, env)
        elif (env == 'str'):
            temp_str.append(reg_mean_dict[(animal, env)])
temp_unstr = np.array(temp_unstr)
temp_str = np.array(temp_str)

# smoothing details
if smooth:           
    for i, row in enumerate(temp_unstr):
        temp_unstr[i] = np.convolve(row, np.ones(window)/window, 'same')
        temp_str[i] = np.convolve(temp_str[i], np.ones(window)/window, 'same')
    
    temp_unstr = (1-np.array(temp_unstr)/100)
    temp_str = (1-np.array(temp_str)/100)
    # 
    # works only for odd windows
    temp_unstr[:, 0:(window//2)] = np.nan
    temp_unstr[:, (trialsinsess-window//2):trialsinsess] = np.nan
    
    temp_str[:, 0:(window//2)] = np.nan
    temp_str[:, (trialsinsess-window//2):trialsinsess] = np.nan


if include_hsb:
    temp_str_hsb, temp_unstr_hsb = [], []
    for (animal, env) in reg_mean_dict_hsb.keys():
        if animal in an_list:
            if env == 'unstr':
                temp_unstr_hsb.append(reg_mean_dict_hsb[(animal, env)])
            elif (env == 'str'):
                temp_str_hsb.append(reg_mean_dict_hsb[(animal, env)])
    temp_unstr_hsb = (1-np.array(temp_unstr_hsb)/100)
    temp_str_hsb = (1-np.array(temp_str_hsb)/100)
    # currently unsmoothed only
    mean_str_hsb = np.nanmean(np.array(temp_str_hsb), axis = 0)
    mean_unstr_hsb = np.nanmean(np.array(temp_unstr_hsb), axis = 0)
    color3, color4 = 'xkcd:dark orange', 'xkcd:olive green'
    # plt.plot(np.arange(trialsinsess), mean_unstr_hsb, color3, label = label1+'_HSB')#, marker = '.') 
    # plt.fill_between(np.arange(trialsinsess), mean_unstr_hsb+sem(temp_unstr_hsb, nan_policy = 'omit'),
    #                 mean_unstr_hsb-sem(temp_unstr_hsb, nan_policy = 'omit'), color = color3, alpha = 0.2)
    plt.plot(np.arange(trialsinsess), mean_str_hsb, color4, label = label2+'_HSB')
    plt.fill_between(np.arange(trialsinsess), mean_str_hsb+sem(temp_str_hsb, nan_policy = 'omit'), 
                    mean_str_hsb-sem(temp_str_hsb, nan_policy = 'omit'), color = color4, alpha = 0.2)


# take mean of smoothed data   
mean_unstr = (np.nanmean(np.array(temp_unstr), axis = 0))
mean_str = (np.nanmean(np.array(temp_str), axis = 0))

# plot line 1
plt.plot(np.arange(trialsinsess), mean_unstr, color1, label = label1)#, marker = '.') 
plt.fill_between(np.arange(trialsinsess), mean_unstr+sem(temp_unstr, nan_policy = 'omit'),
                mean_unstr-sem(temp_unstr, nan_policy = 'omit'), color = color1, alpha = 0.2)

# plot line 2 (or comment)
# plt.plot(np.arange(trialsinsess), mean_str, color2, label = label2)
# plt.fill_between(np.arange(trialsinsess), mean_str+sem(temp_str, nan_policy = 'omit'), 
#                 mean_str-sem(temp_str, nan_policy = 'omit'), color = color2, alpha = 0.2)

# plot significance
# pvals = ttest_rel(temp_str, temp_unstr)[1]
# pvals_ind = [ind for ind, i in enumerate(pvals) if i < 0.05]
# plt.scatter(pvals_ind, [0.45]*len(pvals_ind), marker = '.', color = 'r', alpha = 0.4) #(6, 2, 60) for asterisks

# chance calc
# plt.axhline(np.mean(np.unique(list(permutations([80, 46, 15, 10]))+list(permutations([46, 80, 46, 15])), axis = 0))/100,
#             color = 'k', linestyle = '--')
# plt.axhline(np.mean([[80, 46, 15, 10], [46, 80, 46, 15], [15, 46, 80, 46], [10, 15, 46, 80]])/100, color = color2)

# plot settings
plt.xlabel('Trials in session', fontsize = 'x-large')
plt.ylabel('Performance index', fontsize = 'x-large')
plt.xlim(0, trialsinsess)
# plt.ylim(0.4, 0.8)
# plt.xticks([0, 25, 50, 75, 100], [1, '', '', '', 100], color = 'grey', fontsize = 'large')
# plt.yticks([0.4, 0.5, 0.6, 0.7], [0.4, '', '', 0.7], color = 'grey', fontsize = 'large')
plt.title(f'n = {temp_unstr.shape[0]}')
plt.legend(loc = 'upper left')
sns.despine()
# sns.set_context('talk')
plt.tight_layout()
# plt.savefig('C:/Users/dlab/OneDrive - Indian Institute of Science/Drawings/awp1/entropy_n_21.png', dpi = 300)


# if fitting exp curve is reqd
def logFunc(x, a, b):
    return a + b*np.log(x)

# popt_str, _ = curve_fit(logFunc, np.arange(150), temp_str, nan_policy = 'omit')
# popt_unstr, _ = curve_fit(logFunc, np.arange(trialsinsess), mean_unstr, nan_policy = 'omit')
# plt.plot(np.arange(150), logFunc(np.arange(150), *popt_str), '#408ec6', linestyle= 'dotted')
# plt.plot(np.arange(trialsinsess), logFunc(np.arange(trialsinsess), *popt_unstr), color1, linestyle= 'dotted')

Emolga unstr
Giratina unstr
Ivysaur unstr
Jigglypuff unstr
Lugia unstr
Ninetales unstr
Onix unstr
Pichu unstr
Quaxly unstr
Torchic unstr
Uxie unstr
Vanillish unstr
Whismur unstr
Yamper unstr
Zorua unstr


C:\Users\dlab\AppData\Local\Temp\ipykernel_8424\3656398011.py:62: RuntimeWarning: Mean of empty slice
  mean_unstr_hsb = np.nanmean(np.array(temp_unstr_hsb), axis = 0)
C:\Users\dlab\AppData\Local\Temp\ipykernel_8424\3656398011.py:74: RuntimeWarning: Mean of empty slice
  mean_str = (np.nanmean(np.array(temp_str), axis = 0))


In [ ]:
%matplotlib qt
from scipy.stats import sem
# plot average reponse
plt.figure(figsize = (5,4.5))
temp_unstr, temp_str = [], []

# set properties
smooth = True
window = 3
label1 = 'Pre-lesion'
label2 = 'Post-lesion'
# label1 = 'Unstructured'
# label2 = 'Structured'
color1 = 'xkcd:cornflower'
color2 = 'xkcd:slate grey'
# color1 = '#7a2048', 
# color1 = 'xkcd:pumpkin orange'
# color2 = 'xkcd:emerald green'
# color2 = '#408ec6'

# set data for plotting
cond = 'ds_2stage'
for (animal, env) in reg_mean_dict.keys():
    if animal in an_dict[cond]:
        if env == 'unstr':
            temp_unstr.append(reg_mean_dict[(animal, env)])
        elif (env == 'ds'):
            temp_str.append(reg_mean_dict[(animal, env)])

# calculate performance index
temp_unstr = (1-np.array(temp_unstr)/100)
temp_str = (1-np.array(temp_str)/100)
# temp_unstr = np.array(temp_unstr)
# temp_str = np.array(temp_str)

# smoothing details
if smooth:           
    for i, row in enumerate(temp_unstr):
        temp_unstr[i] = np.convolve(row, np.ones(window)/window, 'same')
        temp_str[i] = np.convolve(temp_str[i], np.ones(window)/window, 'same')
    
    # works only for odd windows
    temp_unstr[:, 0:(window//2)] = np.nan
    temp_unstr[:, (trialsinsess-window//2):trialsinsess] = np.nan
    
    temp_str[:, 0:(window//2)] = np.nan
    temp_str[:, (trialsinsess-window//2):trialsinsess] = np.nan

# temp_str = [np.convolve(temp_str[i], np.ones(window)/window, 'same') for i, row in enumerate(temp_str)]
# temp_str[:, 0:(window//2)] = np.nan
# temp_str[:, (trialsinsess-window//2):trialsinsess] = np.nan

# take mean of smoothed data   
mean_unstr = (np.mean(np.array(temp_unstr), axis = 0))
mean_str = (np.mean(np.array(temp_str), axis = 0))

# plot line 1
plt.plot(np.arange(trialsinsess), mean_unstr, color1, label = label1) 
plt.fill_between(np.arange(trialsinsess), mean_unstr+sem(temp_unstr, nan_policy = 'omit'),
                 mean_unstr-sem(temp_unstr, nan_policy = 'omit'), color = color1, alpha = 0.2)

# plot line 2 (or comment)
plt.plot(np.arange(trialsinsess), mean_str, color2, label = label2)
plt.fill_between(np.arange(trialsinsess), mean_str+sem(temp_str, nan_policy = 'omit'), 
                 mean_str-sem(temp_str, nan_policy = 'omit'), color = color2, alpha = 0.2)
# for row in temp_str:
#     plt.plot(row, color = color2, alpha  = 1)

# plot significance
# pvals = ttest_rel(temp_str, temp_unstr)[1]
# pvals_ind = [ind for ind, i in enumerate(pvals) if i < 0.05]
# plt.scatter(pvals_ind, [0.65]*len(pvals_ind), marker = '.', color = 'r', alpha = 0.4) #(6, 2, 60) for asterisks

# plot settings
plt.xlabel('Trials in session', fontsize = 'x-large')
plt.ylabel('Performance index', fontsize = 'x-large')
plt.xlim(0, trialsinsess)
plt.axhline(0.625, color='k', linestyle = '--') # chance calc by
plt.ylim(0.6, 0.8)
plt.xticks([0, 25, 50, 75, 100], [1, '', '', '', 100], color = 'grey', fontsize = 'large')
plt.yticks([0.6, 0.65, 0.7, 0.75, 0.8, 0.85], ['0.60', '', '', '', '', 0.85], color = 'grey', fontsize = 'large')
# plt.ylim(0, 0.8)
# plt.legend(loc='upper left')
plt.title(f'n = {temp_unstr.shape[0]}')
sns.despine()
# sns.set_context('talk')
plt.tight_layout()
# plt.savefig('C:/Users/dlab/OneDrive - Indian Institute of Science/Drawings/awp1/entropy_n_21.png', dpi = 300)

C:\Users\dlab\AppData\Local\Temp\ipykernel_21396\3896998169.py:59: SmallSampleWarning: After omitting NaNs, one or more axis-slices of one or more sample arguments is too small; corresponding elements of returned arrays will be NaN. See documentation for sample size requirements.
  plt.fill_between(np.arange(trialsinsess), mean_unstr+sem(temp_unstr, nan_policy = 'omit'),
C:\Users\dlab\AppData\Local\Temp\ipykernel_21396\3896998169.py:60: SmallSampleWarning: After omitting NaNs, one or more axis-slices of one or more sample arguments is too small; corresponding elements of returned arrays will be NaN. See documentation for sample size requirements.
  mean_unstr-sem(temp_unstr, nan_policy = 'omit'), color = color1, alpha = 0.2)
C:\Users\dlab\AppData\Local\Temp\ipykernel_21396\3896998169.py:64: SmallSampleWarning: After omitting NaNs, one or more axis-slices of one or more sample arguments is too small; corresponding elements of returned arrays will be NaN. See documentation for sample siz

In [ ]:
# if fitting exp curve is reqd
def logF(x, a, b, c):
    return a*np.log(x+b)+c

def f(x, a, b, c):
    '''exponential fxn: a is amplitude, b is decay rate, c is offset'''
    return a*np.exp(-b*x) + c

popt_str, _ = curve_fit(f, np.arange(trialsinsess), mean_str, nan_policy = 'omit')
popt_unstr, _ = curve_fit(f, np.arange(trialsinsess), mean_unstr, nan_policy = 'omit')
plt.plot(np.arange(trialsinsess), f(np.arange(trialsinsess), *popt_str), color2, linestyle= 'dotted')
plt.plot(np.arange(trialsinsess), f(np.arange(trialsinsess), *popt_unstr), color1, linestyle= 'dotted')

In [ ]:
# logic - if 46 next to 80 then structured, but also how visibly smooth it is
l_str_sess = [
    '[10 15 46 80]',
    '[10 15 80 46]',
    '[10 46 80 15]',
    '[10 80 46 15]',
    '[15 10 46 80]',
    '[15 10 80 46]', 
    '[15 46 46 80]', 
    '[15 46 80 10]',
    '[15 46 80 46]',
    '[15 80 46 10]',
    '[15 80 46 46]',
    '[46 46 80 15]',
    '[46 80 10 15]',
    '[46 80 15 10]',
    '[46 80 46 15]',
    '[80 46 10 15]',
    '[80 46 15 10]',
    '[80 46 46 15]'
]
sessdf['assigned'] = sessdf.rewprobfull.isin(l_str_sess)
sessdf.assigned.replace({True: 'str', False: 'unstr'}, inplace = True)

struc = ['[80 46 15 10]', '[46 80 46 15]', '[15 46 80 46]', '[10 15 46 80]']
sessdf['env'] = sessdf.rewprobfull.isin(struc)
sessdf.env.replace({True: 'str', False: 'unstr'}, inplace = True)

In [ ]:
%matplotlib qt
from scipy.signal import find_peaks
plt.figure(figsize = (20, 10))
unq_prob, _ = np.array(np.unique(sessdf[(sessdf.animal == 'Blissey')].rewprobfull,
          return_counts = True))

for i in range(unq_prob.shape[0]):
    ax = plt.subplot(4, 9, i+1)
    rp = [int(o) for o in unq_prob[i]]
    pk = find_peaks(rp, height = 10)
    ax.plot(rp)
    ax.set_title(f'{pk[0].shape[0]}, {pk[1]["peak_heights"]}')
plt.tight_layout()

# Reward rate analysis

In [55]:
#################################### rr ######################################
%matplotlib qt
window = 7
trialsinsess = 100

from utils.dfLoading import subset_trials, add_shift_info, add_block_groups
subset_df = add_block_groups(sessdf)
subset_df = subset_df[(subset_df.day_task>30) & (subset_df.animal.isin(block_an_dict['uncued']+block_an_dict['cued']))]#+block_an_dict['cued']))]
subset_df = subset_trials(subset_df, trialsinsess=trialsinsess, head_trials = trialsinsess)

# subset_df_unbl = add_block_groups(sessdf)
# subset_df_unbl = subset_df_unbl[(subset_df_unbl.day_task>30) & (subset_df_unbl.block_group == 1) & (subset_df_unbl.animal.isin(block_an_dict['unblocked']))]
# subset_df_unbl = subset_trials(subset_df_unbl, trialsinsess=trialsinsess, head_trials = trialsinsess)
# subset_df_unbl = add_shift_info(subset_df_unbl)


# hsb_data_sorted = hsb_data.sort_values(['animal', 'eptime', 'session'])
subset_df_hsb = add_block_groups(hsb_data)
subset_df_hsb = subset_df_hsb[(subset_df_hsb.day_task>30)]
subset_df_hsb = subset_trials(subset_df_hsb, trialsinsess=trialsinsess, head_trials = trialsinsess)

# window = 
# subset_df['rr'] = subset_df.groupby(['animal', 'session'], as_index = False).reward.rolling(window, center=True).mean().reward
# subset_df_hsb['rr'] = subset_df_hsb.groupby(['animal', 'session'], as_index = False).reward.rolling(window, center=True).mean().reward
# subset_df_unbl['rr'] = subset_df_unbl.groupby(['animal', 'session'], as_index = False).reward.rolling(window, center=True).mean().reward


from utils.dfAnalysisHelpers import get_sessionwise_averages
rr_mean_dict = get_sessionwise_averages(subset_df, metric= 'reward', tasks = ['unstr', 'dms', 'dls', 'sham'])
rr_mean_dict_hsb = get_sessionwise_averages(subset_df_hsb, metric= 'reward', tasks = ['str'])

rr_mean_dict_hsb_unstr = get_sessionwise_averages(subset_df_hsb[subset_df_hsb.env == 'unstr'], metric= 'reward', tasks = ['str'])
rr_mean_dict_hsb_str = get_sessionwise_averages(subset_df_hsb[subset_df_hsb.env == 'str'], metric= 'reward', tasks = ['str'])

rr_mean_dict_pure_unstr = get_sessionwise_averages(subset_df[subset_df.env == 'unstr'], metric= 'reward', tasks = ['unstr'])
rr_mean_dict_str_in_unstr = get_sessionwise_averages(subset_df[subset_df.env == 'str'], metric= 'reward', tasks = ['unstr'])

# rr_mean_dict_unbl = get_sessionwise_averages(subset_df_unbl, metric = 'rr', tasks = ['unstr'])


In [75]:
from scipy.optimize import curve_fit
%matplotlib qt
from scipy.stats import sem
from itertools import permutations
# plot average reponse
from utils.plotSettings import *
plt.figure(figsize = (5,4.5))
temp_unstr, temp_str = [], []

# set properties
smooth = True
window = 3
# label1, label2 = 'Pre-lesion', 'Post-lesion'
label1, label2 = 'Uncorrelated', 'Correlated'
# label1, label2 = 'Block 1', 'Block 2'
color1, color2 = 'xkcd:barney purple', 'xkcd:emerald green'
# color1, color2 = 'xkcd:cornflower', 'xkcd:pink'
# color1, color2 = sns.color_palette('crest')[0], sns.color_palette('crest')[4]
# color1, color2 = 'xkcd:pumpkin orange', 'xkcd:emerald green'
# color1, color2 = 'xkcd:olive green', 'xkcd:emerald green'
# # color1, color2 = '#7a2048', '#408ec6'
sel_animal = 'Blissey'

# set data for plotting
for (animal, env) in rr_mean_dict.keys():
    # if animal == sel_animal:
    if (env == 'unstr'): #& (animal in an_dict['dls']):
        temp_unstr.append(rr_mean_dict[(animal, env)])

for (animal, env) in rr_mean_dict_hsb.keys():
    # if animal == sel_animal:
    # if env == 'dls':
    temp_str.append(rr_mean_dict_hsb[(animal, env)])


temp_unstr = np.array(temp_unstr)
temp_str = np.array(temp_str)

# smoothing details
if smooth:           
    for i, row in enumerate(temp_unstr):
        temp_unstr[i] = np.convolve(row, np.ones(window)/window, 'same')
        try:
            temp_str[i] = np.convolve(temp_str[i], np.ones(window)/window, 'same')
        except IndexError:
            pass
    
    temp_unstr = np.array(temp_unstr)
    temp_str = np.array(temp_str)
    # 
    # works only for odd windows
    temp_unstr[:, 0:(window//2)] = np.nan
    temp_unstr[:, (trialsinsess-window//2):trialsinsess] = np.nan
    
    temp_str[:, 0:(window//2)] = np.nan
    temp_str[:, (trialsinsess-window//2):trialsinsess] = np.nan

# take mean of smoothed data   
mean_unstr = (np.nanmean(np.array(temp_unstr), axis = 0))
mean_str = (np.nanmean(np.array(temp_str), axis = 0))

# plot line 1
plt.plot(np.arange(trialsinsess), mean_unstr, color = color1, label = label1)#, marker = '.') 
plt.fill_between(np.arange(trialsinsess), mean_unstr+sem(temp_unstr, nan_policy = 'omit'),
                mean_unstr-sem(temp_unstr, nan_policy = 'omit'), color = color1, alpha = 0.2)

# plot line 2 (or comment)
plt.plot(np.arange(trialsinsess), mean_str, color = color2, label = label2)
plt.fill_between(np.arange(trialsinsess), mean_str+sem(temp_str, nan_policy = 'omit'), 
                mean_str-sem(temp_str, nan_policy = 'omit'), color = color2, alpha = 0.2)

# plot significance
# from scipy.stats import ttest_ind
# pvals = ttest_ind(temp_str, temp_unstr)[1]
# pvals_ind = [ind for ind, i in enumerate(pvals) if i < 0.05]
# plt.scatter(pvals_ind, [0.45]*len(pvals_ind), marker = '.', color = 'r', alpha = 0.4) #(6, 2, 60) for asterisks

# chance calc
prob_comb = np.unique(list(permutations([80, 46, 15, 10]))+list(permutations([46, 80, 46, 15])), axis = 0)
str_prob_comb = np.array([[80, 46, 15, 10], [46, 80, 46, 15], [15, 46, 80, 46], [10, 15, 46, 80]])
unstr_prob_comb = prob_comb[~(prob_comb[:, None, :] == str_prob_comb[None, :, :]).all(axis = 2).any(axis = 1)]
hsb_prob_comb = [42.06]
rs_prob_comb = [42.25]
# np.mean(unstr_prob_comb)
plt.axhline(np.mean(rs_prob_comb)/100, color = color1, linestyle = '--')
plt.axhline(np.mean(hsb_prob_comb)/100, color = color2, linestyle = '--')

# plot settings
plt.xlabel('Trials in session', fontsize = 'x-large')
plt.ylabel('Reward rate', fontsize = 'x-large')
plt.xlim(0, trialsinsess)
plt.ylim(0.4, 0.75)
plt.xticks([0, 25, 50, 75, 100], [1, '', '', '', 100], color = 'grey', fontsize = 'large')
plt.yticks([0.4, 0.5, 0.6, 0.7], [0.4, '', '', 0.7], color = 'grey', fontsize = 'large')
# plt.title(f'n = {temp_unstr.shape[0]}')

leg = plt.legend(frameon=False, loc = 'upper left')
for line, text in zip(leg.get_lines(), leg.get_texts()):
    text.set_color(line.get_color())
sns.despine()
# sns.set_context('talk')
plt.tight_layout()
# plt.savefig('C:/Users/dlab/OneDrive - Indian Institute of Science/Drawings/awp1/entropy_n_21.png', dpi = 300)


# if fitting exp curve is reqd
def logFunc(x, a, b):
    return a + b*np.log(x)

# popt_str, _ = curve_fit(logFunc, np.arange(150), temp_str, nan_policy = 'omit')
# popt_unstr, _ = curve_fit(logFunc, np.arange(trialsinsess), mean_unstr, nan_policy = 'omit')
# plt.plot(np.arange(150), logFunc(np.arange(150), *popt_str), '#408ec6', linestyle= 'dotted')
# plt.plot(np.arange(trialsinsess), logFunc(np.arange(trialsinsess), *popt_unstr), color1, linestyle= 'dotted')

C:\Users\dlab\AppData\Local\Temp\ipykernel_22024\3414676009.py:59: RuntimeWarning: Mean of empty slice
  mean_unstr = (np.nanmean(np.array(temp_unstr), axis = 0))
C:\Users\dlab\AppData\Local\Temp\ipykernel_22024\3414676009.py:60: RuntimeWarning: Mean of empty slice
  mean_str = (np.nanmean(np.array(temp_str), axis = 0))
C:\Users\dlab\AppData\Local\Temp\ipykernel_22024\3414676009.py:64: SmallSampleWarning: After omitting NaNs, one or more axis-slices of one or more sample arguments is too small; corresponding elements of returned arrays will be NaN. See documentation for sample size requirements.
  plt.fill_between(np.arange(trialsinsess), mean_unstr+sem(temp_unstr, nan_policy = 'omit'),
C:\Users\dlab\AppData\Local\Temp\ipykernel_22024\3414676009.py:65: SmallSampleWarning: After omitting NaNs, one or more axis-slices of one or more sample arguments is too small; corresponding elements of returned arrays will be NaN. See documentation for sample size requirements.
  mean_unstr-sem(temp_u

In [79]:
# blockwise rr diff groups
from scipy.stats import sem
from utils.plotSettings import *

# (block_group, 'first'/'last', n_trials) in x-axis order
SEGMENTS = [(1, 'first', 100), (1, 'last', 25),
            (2, 'first', 100)]
seg_lens = [n for _, _, n in SEGMENTS]
seg_edges = np.cumsum([0] + seg_lens)
n_trials = seg_edges[-1]   # 275
ALLOW_OVERLAP = False      # False: a block needs >= first_n + last_n trials so first/last don't share trials

# set properties
smooth = False
window = 5                    # odd number so the window is centred
smooth_within_blocks = False  # True: don't smooth across segment boundaries

# minimum trials each block needs
need = {}
for bg, _, n in SEGMENTS:
    need[bg] = need.get(bg, 0) + n if not ALLOW_OVERLAP else max(need.get(bg, 0), n)

def session_matrix(df, sess_col='session'):
    """Returns {animal: n_sessions x n_trials reward matrix}; only sessions where every block is long enough."""
    out = {}
    df = df.sort_values('datetime')
    for animal, adf in df.groupby('animal'):
        rows = []
        for _, sdf in adf.groupby(sess_col):
            blocks = {bg: sdf.loc[sdf.block_group == bg, 'reward'].to_numpy() for bg in need}
            if any(len(blocks[bg]) < need[bg] for bg in need):
                continue
            parts = [blocks[bg][:n] if which == 'first' else blocks[bg][-n:] for bg, which, n in SEGMENTS]
            rows.append(np.concatenate(parts))
        if rows:
            out[animal] = np.vstack(rows).astype(float)
    return out

unstr_mats = session_matrix(sessdf[(sessdf.day_task > 30) & (sessdf.task == 'unstr') & (sessdf.animal.isin(block_an_dict['cued']+block_an_dict['uncued']))], sess_col='session')
str_mats   = session_matrix(hsb_data[(hsb_data.day_task > 30) & (hsb_data.task == 'str')], sess_col='session')

def smooth_rows(mat, window, within_blocks=True):
    """Moving average; near segment edges, averages only over trials inside the segment (no NaNs)."""
    out = np.full_like(mat, np.nan)
    edges = seg_edges if within_blocks else [0, mat.shape[1]]
    k = np.ones(window)
    for s, e in zip(edges[:-1], edges[1:]):
        seg = mat[:, s:e]
        norm = np.convolve(np.ones(seg.shape[1]), k, 'same')   # trials actually inside the window
        out[:, s:e] = np.array([np.convolve(r, k, 'same') for r in seg]) / norm
    return out

# average over sessions (rows) per animal -> animals x n_trials
temp_unstr = np.array([m.mean(axis=0) for m in unstr_mats.values()])
temp_str   = np.array([m.mean(axis=0) for m in str_mats.values()])

if smooth:
    temp_unstr = smooth_rows(temp_unstr, window, smooth_within_blocks)
    temp_str = smooth_rows(temp_str, window, smooth_within_blocks)

mean_unstr = np.nanmean(temp_unstr, axis=0)
mean_str = np.nanmean(temp_str, axis=0)

# plot
plt.figure(figsize=(9, 4.5))
ax = plt.gca()
color1, color2 = 'xkcd:barney purple', 'xkcd:emerald green'
label1, label2 = 'Uncorrelated', 'Correlated'

# split lines only at axis breaks (first 100 -> last 25), not at block switches
break_pos = [b for (_, which, _), b in zip(SEGMENTS[1:], seg_edges[1:-1]) if which == 'last']
switch_pos = [b for (_, which, _), b in zip(SEGMENTS[1:], seg_edges[1:-1]) if which == 'first']
runs = list(zip([0] + break_pos, break_pos + [n_trials]))

# shift x after each axis break so the data has a visible gap there
gap = 2.5   # half-width of the break, in trials
def xmap(t):
    return np.asarray(t) + 2 * gap * np.searchsorted(break_pos, t, side='right')
x = xmap(np.arange(n_trials))

for i, (s, e) in enumerate(runs):
    for mean, temp, c, lab in [(mean_unstr, temp_unstr, color1, label1), (mean_str, temp_str, color2, label2)]:
        err = sem(temp[:, s:e], nan_policy='omit')
        plt.plot(x[s:e], mean[s:e], color=c, label=lab if i == 0 else None)
        plt.fill_between(x[s:e], mean[s:e] + err, mean[s:e] - err, color=c, alpha=0.2)

# block switches: dashed vertical line
for b in switch_pos:
    plt.axvline(xmap(b), color='grey', linestyle='--', linewidth=0.75)

# axis break where the middle of a block is skipped
trans = ax.get_xaxis_transform()   # x in data units, y in axes units
for b in xmap(break_pos) - gap - 0.5:   # centre of each gap in the data
    # patch that cuts the x-axis spine
    ax.add_patch(plt.Rectangle((b - gap, -0.02), 2 * gap, 0.04, transform=trans,
                               facecolor=plt.gcf().get_facecolor(), edgecolor='none',
                               clip_on=False, zorder=3))
    # the two diagonal break marks
    ax.plot([b - gap, b + gap], [0, 0], transform=trans, linestyle='none',
            marker=[(-1, -1.5), (1, 1.5)], markersize=8, markeredgewidth=1,
            color='k', clip_on=False, zorder=4)

plt.xlabel('Trials in session', fontsize='x-large')
plt.ylabel('Reward rate', fontsize='x-large')
plt.xlim(0, xmap(n_trials - 1) + 1)
plt.axhline(42.25/100, color = color1, linestyle = '--')
plt.axhline(42.06/100, color = color2, linestyle = '--')
ticks = np.arange(0, n_trials + 1, 25)
plt.xticks(xmap(ticks), ticks, color='grey', fontsize='large')
leg = plt.legend(frameon=False, loc='upper left')
for line, text in zip(leg.get_lines(), leg.get_texts()):
    text.set_color(line.get_color())
plt.tight_layout()


In [55]:
from scipy.optimize import curve_fit
from scipy.stats import sem
from itertools import permutations
# plot average reponse
from utils.plotSettings import *
plt.figure(figsize = (5,4.5))
temp_unstr, temp_str = [], []

# set properties
smooth = False
include_hsb = True
window = 3
# label1, label2 = 'Pre-lesion', 'Post-lesion'
label1, label2 = 'Unstructured', 'Structured'
# color1, color2 = 'xkcd:cornflower', 'xkcd:slate grey'
color1, color2 = 'xkcd:pumpkin orange', 'xkcd:emerald green'
# # color1, color2 = '#7a2048', '#408ec6'

joint_rr_mean_dict = rr_mean_dict_hsb | rr_mean_dict
an_list = an_dict['hsb_str'] + an_dict['unstr']

# set data for plotting
for (animal, env) in rr_mean_dict.keys():
    if animal in an_list:
        if env == 'unstr':
            temp_unstr.append(rr_mean_dict[(animal, env)])
            print(animal, env)
        elif (env == 'str'):
            temp_str.append(rr_mean_dict[(animal, env)])
temp_unstr = np.array(temp_unstr)
temp_str = np.array(temp_str)

# smoothing details
if smooth:           
    for i, row in enumerate(temp_unstr):
        temp_unstr[i] = np.convolve(row, np.ones(window)/window, 'same')
        temp_str[i] = np.convolve(temp_str[i], np.ones(window)/window, 'same')
    
    temp_unstr = np.array(temp_unstr)
    temp_str = np.array(temp_str)
    # 
    # works only for odd windows
    temp_unstr[:, 0:(window//2)] = np.nan
    temp_unstr[:, (trialsinsess-window//2):trialsinsess] = np.nan
    
    temp_str[:, 0:(window//2)] = np.nan
    temp_str[:, (trialsinsess-window//2):trialsinsess] = np.nan


if include_hsb:
    temp_str_hsb, temp_unstr_hsb = [], []
    for (animal, env) in rr_mean_dict_hsb.keys():
        if animal in an_list:
            if env == 'unstr':
                temp_unstr_hsb.append(rr_mean_dict_hsb[(animal, env)])
            elif (env == 'str'):
                temp_str_hsb.append(rr_mean_dict_hsb[(animal, env)])
    temp_unstr_hsb = np.array(temp_unstr_hsb)
    temp_str_hsb = np.array(temp_str_hsb)
    # currently unsmoothed only
    mean_str_hsb = np.nanmean(np.array(temp_str_hsb), axis = 0)
    mean_unstr_hsb = np.nanmean(np.array(temp_unstr_hsb), axis = 0)
    color3, color4 = 'xkcd:dark orange', 'xkcd:olive green'
    # plt.plot(np.arange(trialsinsess), mean_unstr_hsb, color3, label = label1+'_HSB')#, marker = '.') 
    # plt.fill_between(np.arange(trialsinsess), mean_unstr_hsb+sem(temp_unstr_hsb, nan_policy = 'omit'),
    #                 mean_unstr_hsb-sem(temp_unstr_hsb, nan_policy = 'omit'), color = color3, alpha = 0.2)
    plt.plot(np.arange(trialsinsess), mean_str_hsb, color4, label = label2+'_HSB')
    plt.fill_between(np.arange(trialsinsess), mean_str_hsb+sem(temp_str_hsb, nan_policy = 'omit'), 
                    mean_str_hsb-sem(temp_str_hsb, nan_policy = 'omit'), color = color4, alpha = 0.2)


# take mean of smoothed data   
mean_unstr = (np.nanmean(np.array(temp_unstr), axis = 0))
mean_str = (np.nanmean(np.array(temp_str), axis = 0))

# plot line 1
plt.plot(np.arange(trialsinsess), mean_unstr, color1, label = label1)#, marker = '.') 
plt.fill_between(np.arange(trialsinsess), mean_unstr+sem(temp_unstr, nan_policy = 'omit'),
                mean_unstr-sem(temp_unstr, nan_policy = 'omit'), color = color1, alpha = 0.2)

# plot line 2 (or comment)
# plt.plot(np.arange(trialsinsess), mean_str, color2, label = label2)
# plt.fill_between(np.arange(trialsinsess), mean_str+sem(temp_str, nan_policy = 'omit'), 
#                 mean_str-sem(temp_str, nan_policy = 'omit'), color = color2, alpha = 0.2)

# plot significance
# pvals = ttest_rel(temp_str, temp_unstr)[1]
# pvals_ind = [ind for ind, i in enumerate(pvals) if i < 0.05]
# plt.scatter(pvals_ind, [0.45]*len(pvals_ind), marker = '.', color = 'r', alpha = 0.4) #(6, 2, 60) for asterisks

# chance calc
plt.axhline(np.mean(np.unique(list(permutations([80, 46, 15, 10]))+list(permutations([46, 80, 46, 15])), axis = 0))/100,
            color = 'k', linestyle = '--')
plt.axhline(np.mean([[80, 46, 15, 10], [46, 80, 46, 15], [15, 46, 80, 46], [10, 15, 46, 80]])/100, color = color2)

# plot settings
plt.xlabel('Trials in session', fontsize = 'x-large')
plt.ylabel('Reward rate', fontsize = 'x-large')
plt.xlim(0, trialsinsess)
plt.ylim(0.4, 0.7)
# plt.xticks([0, 25, 50, 75, 100], [1, '', '', '', 100], color = 'grey', fontsize = 'large')
plt.yticks([0.4, 0.5, 0.6, 0.7], [0.4, '', '', 0.7], color = 'grey', fontsize = 'large')
plt.title(f'n = {temp_unstr.shape[0]}')
plt.legend(loc = 'upper left')
sns.despine()
# sns.set_context('talk')
plt.tight_layout()
# plt.savefig('C:/Users/dlab/OneDrive - Indian Institute of Science/Drawings/awp1/entropy_n_21.png', dpi = 300)


# if fitting exp curve is reqd
def logFunc(x, a, b):
    return a + b*np.log(x)

# popt_str, _ = curve_fit(logFunc, np.arange(150), temp_str, nan_policy = 'omit')
# popt_unstr, _ = curve_fit(logFunc, np.arange(trialsinsess), mean_unstr, nan_policy = 'omit')
# plt.plot(np.arange(150), logFunc(np.arange(150), *popt_str), '#408ec6', linestyle= 'dotted')
# plt.plot(np.arange(trialsinsess), logFunc(np.arange(trialsinsess), *popt_unstr), color1, linestyle= 'dotted')

Emolga unstr
Giratina unstr
Ivysaur unstr
Jigglypuff unstr
Lugia unstr
Quaxly unstr
Torchic unstr
Uxie unstr
Vanillish unstr
Whismur unstr
Zorua unstr


C:\Users\dlab\AppData\Local\Temp\ipykernel_7644\473177550.py:61: RuntimeWarning: Mean of empty slice
  mean_str_hsb = np.nanmean(np.array(temp_str_hsb), axis = 0)
C:\Users\dlab\AppData\Local\Temp\ipykernel_7644\473177550.py:62: RuntimeWarning: Mean of empty slice
  mean_unstr_hsb = np.nanmean(np.array(temp_unstr_hsb), axis = 0)
C:\Users\dlab\AppData\Local\Temp\ipykernel_7644\473177550.py:68: SmallSampleWarning: After omitting NaNs, one or more axis-slices of one or more sample arguments is too small; corresponding elements of returned arrays will be NaN. See documentation for sample size requirements.
  plt.fill_between(np.arange(trialsinsess), mean_str_hsb+sem(temp_str_hsb, nan_policy = 'omit'),
C:\Users\dlab\AppData\Local\Temp\ipykernel_7644\473177550.py:69: SmallSampleWarning: After omitting NaNs, one or more axis-slices of one or more sample arguments is too small; corresponding elements of returned arrays will be NaN. See documentation for sample size requirements.
  mean_str_hsb-

In [ ]:
temp_unstr, temp_str = [], []
for (animal, env) in rr_mean_dict.keys():
    if animal in fish:
        if env == 'unstr':
            temp_unstr.append(rr_mean_dict[(animal, env)])
        elif env == 'str':
            temp_str.append(rr_mean_dict[(animal, env)])

ttest_rel(np.nanmean(np.array(temp_str)[:, :25], axis = 1),
          np.nanmean(np.array(temp_str)[:, -25:], axis = 1), nan_policy = 'omit')

plt.figure(figsize = (3.5,4.5))
rr_df = pd.DataFrame([np.nanmean(np.array(temp_str), axis = 1), np.nanmean(np.array(temp_unstr), axis = 1)],
                     index = ['Structured', 'Unstructured'])
plt.scatter(np.zeros(5), rr_df.loc['Structured'], color = '#408ec6', alpha = 0.5)
plt.scatter(np.ones(5), rr_df.loc['Unstructured'], color = '#7a2048', alpha = 0.5)
plt.plot(rr_df, '-', c = 'grey', alpha = 0.5)
sns.pointplot(rr_df.T, join = False, palette = ['#408ec6','#7a2048'], errorbar = None)
sns.despine()
plt.xlim([-0.4,1.4])
# ax.set_xticks([0,1])
plt.xlabel('Session')
# plt.xticks([0,1],['Structured', 'End'])
plt.ylabel('Reward rate')
plt.title('Reward rate across conditions', wrap = True)
plt.tight_layout()

In [ ]:
# plot 
window = 7
trialsinsess = 100
fig = plt.figure(figsize=(10,10))
rr_mean_dict_unstr = {}
# rem_les = sessdf[~(sessdf.task.isin(['dms', 'dls', 'sham']))]
fish = ['test05022023', 'Blissey', 'Chikorita', 'Darkrai', 'Eevee']

colors ={'str':'#1e2761', 'unstr':'#7a2048', 'dls':'xkcd:gold', 'dms':'xkcd:red', 'sham':'b', 'dms_str':'xkcd:orange'} 
for i in ['unstr', 'str']:

    for ind, (animal, group) in enumerate(sessdf[sessdf.animal.isin(fish)].groupby('animal')):
        ax = plt.subplot(4, 7, ind+1)
        
        # remove duplicates - if a session has duplicates remove it 
        # group = group.loc[~(sessdf.groupby('animal').get_group(animal)
        #                           .duplicated(subset = 'trialstart', keep = False))]
        
        # filter by sessions>= length
        filtered = (group[group.env.isin(['str']) & (group.task.isin([i]))]
                    .groupby('session')).filter(lambda x: x.reward.size >= trialsinsess)

        # take only those sessions
        filtered = filtered.groupby('session').head(trialsinsess)
        if filtered.empty==True:
            continue
            
        # convert to numpy array
        g = filtered.groupby('session').cumcount()
        L = np.array(filtered.set_index(['session',g])
               .unstack(fill_value=0)
               .stack().groupby(level=0)
               .apply(lambda x: x.rr.values.tolist())
               .tolist())
        
        print(L.shape, animal, i)
        # mean
        rr_mean = np.mean(L, axis=0)
        # reg_sem = np.std(L)
        rr_mean_dict_unstr[animal, i] = rr_mean
        ax.plot(rr_mean, label = i, c = colors[i])
        # ax.fill_between(np.arange(trialsinsess), reg_sem+reg_mean, reg_mean-reg_sem, color = colors[i], alpha = 0.2)
        ax.set_title(animal)
        # ax.legend(['Pure unstructured','__', 'Structured (in unstr)'])
        ax.set_xlabel('Trial')

sns.despine()
fig.supylabel('Reward rate, averaged across sessions')
plt.tight_layout()

(49, 100) Blissey unstr
(17, 100) Chikorita unstr
(19, 100) Darkrai unstr
(9, 100) Eevee unstr
(18, 100) test05022023 unstr
(177, 100) Blissey str
(136, 100) Chikorita str
(65, 100) Darkrai str
(209, 100) Eevee str
(104, 100) test05022023 str


In [ ]:
# plot average reponse and add structured in structured also
plt.figure(figsize = (5,4.5))
temp_unstr, temp_str, temp_str_str = [], [], []

# set properties
smooth = True
window = 3
label1 = 'Structured (in unstr)'
label2 = 'Structured (in str)'
# label3 = 'Structured (in str)'
color1 = 'xkcd:azure'
color2 = 'xkcd:slate grey'
# color1 = '#7a2048', 
color1 = 'xkcd:olive'
color2 = 'xkcd:emerald green'
# color3 = 'xkcd:magenta'
# color2 = '#408ec6'

# set data for plotting
# fish = ['Blissey', 'Darkrai', 'Mesprit', 'Quilava', 'Raltz', 'Inkay']
# fish = ['Hoppip', 'Kirlia', 'Nidorina', 'Togepi']
# fish = ['test05022023', 'Jirachi', 'Goldeen', 'Phione']
fish = ['test05022023', 'Blissey', 'Chikorita', 'Darkrai', 'Eevee']
for (animal, env) in rr_mean_dict_unstr.keys():
    if animal in fish:
        if env == 'unstr':
            temp_unstr.append(rr_mean_dict_unstr[(animal, env)])
        elif (env == 'str'):
            temp_str.append(rr_mean_dict_unstr[(animal, env)])
            # temp_str_str.append(rr_mean_dict[(animal, env)])

# calculate performance index
# temp_unstr = (1-np.array(temp_unstr)/100)
# temp_str = (1-np.array(temp_str)/100)
temp_unstr = np.array(temp_unstr)
temp_str = np.array(temp_str)
# temp_str_str = np.array(temp_str_str)

# smoothing details
if smooth:           
    for i, row in enumerate(temp_unstr):
        temp_unstr[i] = np.convolve(row, np.ones(window)/window, 'same')
        temp_str[i] = np.convolve(temp_str[i], np.ones(window)/window, 'same')
        # temp_str_str[i] = np.convolve(temp_str_str[i], np.ones(window)/window, 'same')
    # 
    # works only for odd windows
    temp_unstr[:, 0:(window//2)] = np.nan
    temp_unstr[:, (trialsinsess-window//2):trialsinsess] = np.nan
    
    temp_str[:, 0:(window//2)] = np.nan
    temp_str[:, (trialsinsess-window//2):trialsinsess] = np.nan

    # temp_str_str[:, 0:(window//2)] = np.nan
    # temp_str_str[:, (trialsinsess-window//2):trialsinsess] = np.nan

# take mean of smoothed data   
mean_unstr = (np.mean(np.array(temp_unstr), axis = 0))
mean_str = (np.mean(np.array(temp_str), axis = 0))
# mean_str_str = (np.mean(np.array(temp_str_str), axis = 0))

# plot line 1
plt.plot(np.arange(trialsinsess), mean_unstr, color1, label = label1) 
plt.fill_between(np.arange(trialsinsess), mean_unstr+sem(temp_unstr, nan_policy = 'omit'),
                 mean_unstr-sem(temp_unstr, nan_policy = 'omit'), color = color1, alpha = 0.2)

# plot line 2 (or comment)
plt.plot(np.arange(trialsinsess), mean_str, color2, label = label2)
plt.fill_between(np.arange(trialsinsess), mean_str+sem(temp_str, nan_policy = 'omit'), 
                 mean_str-sem(temp_str, nan_policy = 'omit'), color = color2, alpha = 0.2)

# plot line 3 (or comment)
# plt.plot(np.arange(trialsinsess), mean_str_str, color3, label = label3)
# plt.fill_between(np.arange(trialsinsess), mean_str_str+sem(temp_str_str, nan_policy = 'omit'), 
#                  mean_str_str-sem(temp_str_str, nan_policy = 'omit'), color = color3, alpha = 0.2)

# plot significance
# pvals = ttest_rel(temp_str, temp_unstr)[1]
# pvals_ind = [ind for ind, i in enumerate(pvals) if i < 0.05]
# plt.scatter(pvals_ind, [0.65]*len(pvals_ind), marker = '.', color = 'r', alpha = 0.4) #(6, 2, 60) for asterisks

# plot settings
plt.xlabel('Trials in session', fontsize = 'x-large')
plt.ylabel('Reward rate', fontsize = 'x-large')
plt.xlim(0, trialsinsess)
# plt.axhline(0.625, color='k', linestyle = '--') # chance calc by
# plt.ylim(0.6, 0.95)
plt.xticks([0, 25, 50, 75, 100], [1, '', '', '', 100], color = 'grey', fontsize = 'large')
plt.ylim(0.4, 0.7)

plt.yticks([0.4, 0.5, 0.6, 0.7], [0.4, '', '', 0.7], color = 'grey', fontsize = 'large')

plt.legend(loc='upper left')
sns.despine()
# sns.set_context('talk')
plt.tight_layout()
# plt.savefig('C:/Users/dlab/OneDrive - Indian Institute of Science/Drawings/awp1/entropy_n_21.png', dpi = 300)

## lesion-weekwise rr

In [274]:
import json
trialsinsess = 100
metric = 'reward'
days_back = 29
# DLS : raspberry, DMS: kelly green
# Load lesion dates from task_data.json
with open('utils/task_data.json', 'r') as f:
    task_data = json.load(f)

cond_list = ['dls', 'dms', 'sham']
daywise_dict_pre = {}
daywise_dict_post = {}
data_mask = (sessdf.block_group == 1)

for cond in cond_list:
    # Create a mapping of animal to first lesion date (first "after" date for each animal)
    lesion_dates = {}
    for rule in task_data['rules']:
        animal = rule['animal']
        if animal not in lesion_dates:
            if animal in an_dict[cond]:
                if animal != 'Zacian': # fix this ASAP
                    lesion_dates[animal] = pd.to_datetime(rule['after'])

    # Calculate reward rate per session and days relative to lesion
    reward_data = []
    daywise_average_pre = np.zeros((len(lesion_dates), days_back+1, trialsinsess))*np.nan
    daywise_average_post = np.zeros((len(lesion_dates), days_back+1, trialsinsess))*np.nan

    for an_idx, (animal, lesion_date) in enumerate(lesion_dates.items()):
        subset_df = sessdf[data_mask & (sessdf.animal==animal)]
        from utils.dfLoading import subset_trials
        
        if len(subset_df) == 0:
            continue
        # something wrong with selected pre dates
        # last_pre_date = subset_df[subset_df.datetime<lesion_dates[animal]]['datetime'].tail(1)
        # print(animal, last_pre_date,)
        first_pre_date = last_pre_date - pd.to_timedelta(str(days_back)+'day')
        first_post_date = lesion_dates[animal]
        last_pre_date = first_post_date - pd.to_timedelta('16day')
        first_pre_date = last_pre_date - pd.to_timedelta(str(days_back)+'day')
        last_post_date = first_post_date + pd.to_timedelta(str(days_back)+'day')

        # pre_subset_df = subset_df[(subset_df['datetime']>first_pre_date.values[0]) & (subset_df['datetime']<last_pre_date.values[0])]
        pre_subset_df = subset_df[(subset_df['datetime']>first_pre_date) & (subset_df['datetime']<last_pre_date)]
        post_subset_df = subset_df[(subset_df['datetime']>first_post_date) & (subset_df['datetime']<last_post_date)]
        pre_subset_df = subset_trials(pre_subset_df, trialsinsess=trialsinsess, head_trials=trialsinsess) 
        post_subset_df = subset_trials(post_subset_df, trialsinsess=trialsinsess, head_trials=trialsinsess) 
        # for every day, get avg reward over sessions? 
        from utils.dfAnalysisHelpers import get_sessionwise_averages
        
        for d_idx, (day, group) in enumerate(pre_subset_df.groupby(pre_subset_df['datetime'].dt.date)):
            daywise_average_pre[an_idx, d_idx, :] = np.array(list(get_sessionwise_averages(group, metric = metric).values())).flatten()

        for d_idx, (day, group) in enumerate(post_subset_df.groupby(post_subset_df['datetime'].dt.date)):
            daywise_average_post[an_idx, d_idx, :] = np.array(list(get_sessionwise_averages(group, metric = metric).values())).flatten()
    # store for each condition        
    daywise_dict_pre[cond] = daywise_average_pre
    daywise_dict_post[cond] = daywise_average_post

c:\Users\dlab\banditAnalysis\utils\dfAnalysisHelpers.py:24: RuntimeWarning: Mean of empty slice
  metric_dict[an] = np.nanmean(L, axis = 0)


In [266]:
import json
trialsinsess = 100
metric = 'rr'
days_back = 3
# DLS : raspberry, DMS: kelly green
# Load lesion dates from task_data.json
with open('utils/task_data.json', 'r') as f:
    task_data = json.load(f)

cond_list = ['dls', 'dms', 'sham']
daywise_dict_pre = {}
daywise_dict_post = {}
data_mask = (sessdf.block_group == 1)

for cond in cond_list:
    # Create a mapping of animal to first lesion date (first "after" date for each animal)
    lesion_dates = {}
    for rule in task_data['rules']:
        animal = rule['animal']
        if animal not in lesion_dates:
            if animal in an_dict[cond]:
                if animal not in ['Zacian']:# 'Alakazam', 'Nidorina', 'Lugia', 'Quaxly', 'Whismur']: # fix this ASAP
                    lesion_dates[animal] = pd.to_datetime(rule['after'])

    # Calculate reward rate per session and days relative to lesion
    reward_data = []
    daywise_average_pre = np.zeros((len(lesion_dates), days_back+1, trialsinsess))*np.nan
    daywise_average_post = np.zeros((len(lesion_dates), days_back+1, trialsinsess))*np.nan

    for an_idx, (animal, lesion_date) in enumerate(lesion_dates.items()):
        subset_df = sessdf[data_mask & (sessdf.animal==animal)]
        from utils.dfLoading import subset_trials
        
        if len(subset_df) == 0:
            continue
        # something wrong with selected pre dates
        # last_pre_date = subset_df[subset_df.datetime<lesion_dates[animal]]['datetime'].tail(1)
        # print(animal, last_pre_date,)
        first_pre_date = last_pre_date - pd.to_timedelta(str(days_back)+'W')
        first_post_date = lesion_dates[animal]
        last_pre_date = first_post_date - pd.to_timedelta('16day')
        first_pre_date = last_pre_date - pd.to_timedelta(str(days_back)+'W')
        last_post_date = first_post_date + pd.to_timedelta(str(days_back)+'W')

        # pre_subset_df = subset_df[(subset_df['datetime']>first_pre_date.values[0]) & (subset_df['datetime']<last_pre_date.values[0])]
        pre_subset_df = subset_df[(subset_df['datetime']>first_pre_date) & (subset_df['datetime']<last_pre_date)]
        post_subset_df = subset_df[(subset_df['datetime']>first_post_date) & (subset_df['datetime']<last_post_date)]
        pre_subset_df = subset_trials(pre_subset_df, trialsinsess=trialsinsess, head_trials=trialsinsess) 
        post_subset_df = subset_trials(post_subset_df, trialsinsess=trialsinsess, head_trials=trialsinsess) 
        # for every day, get avg reward over sessions? 
        from utils.dfAnalysisHelpers import get_sessionwise_averages
        
        for d_idx, (day, group) in enumerate(pre_subset_df.groupby(pd.Grouper(key='datetime', freq='W'))):
            if group.empty:
                continue
            daywise_average_pre[an_idx, d_idx, :] = np.array(list(get_sessionwise_averages(group, metric = 'reward').values())).flatten()

        for d_idx, (day, group) in enumerate(post_subset_df.groupby(pd.Grouper(key='datetime', freq='W'))):
            if group.empty:
                continue
            daywise_average_post[an_idx, d_idx, :] = np.array(list(get_sessionwise_averages(group, metric = 'reward').values())).flatten()

        print(animal)
    # store for each condition        
    daywise_dict_pre[cond] = daywise_average_pre
    daywise_dict_post[cond] = daywise_average_post

Blissey
Darkrai
Inkay
Mesprit
Quilava
Raltz
Alakazam
Torchic
Uxie
Vanillish
Yamper
Zorua
Hoppip
Kirlia
Nidorina
Togepi
Xatu
Yanma
Giratina
Ivysaur
Lugia
Quaxly
Whismur
test05022023
Jirachi
Goldeen
Phione
Umbreon
Vulpix
Emolga
Haxorus
Jigglypuff


In [287]:
from scipy.stats import sem
from itertools import permutations
plt.figure(figsize=(6, 3))
cond_list = ['sham','dls', 'dms']
colors = {'dls':'xkcd:faded red', 'dms':'xkcd:soft green', 'sham':'xkcd:slate grey'}
labels = {'dls':'DLS (n = 12)', 'dms': 'DMS (n = 11)', 'sham':'Sham (n = 9)'}
# trial_range = 
pn = 1
for cond in cond_list:
    ax = plt.subplot(1,1,1)
    daywise_average_pre = daywise_dict_pre[cond]
    daywise_average_post = daywise_dict_post[cond]

    ax.plot(
        np.arange(daywise_average_pre.shape[1]),
        np.nanmean(daywise_average_pre, axis = (0,2)),
        color = colors[cond], label = labels[cond]
        )
    ax.fill_between(
        np.arange(daywise_average_pre.shape[1]),
        np.nanmean(daywise_average_pre, axis = (0,2))-sem(np.nanmean(daywise_average_pre, axis = 2), axis = 0, nan_policy='omit'),
        np.nanmean(daywise_average_pre, axis = (0,2))+sem(np.nanmean(daywise_average_pre, axis = 2), axis = 0, nan_policy='omit'),
        color = colors[cond], alpha = 0.3
        )

    ax.plot(
        np.arange(daywise_average_post.shape[1])+daywise_average_pre.shape[1],
        np.nanmean(daywise_average_post, axis = (0,2)),
        color = colors[cond],
        ) 
    ax.fill_between(
        np.arange(daywise_average_post.shape[1])+daywise_average_pre.shape[1],
        np.nanmean(daywise_average_post, axis = (0,2))-sem(np.nanmean(daywise_average_post, axis = 2), axis = 0, nan_policy='omit'),
        np.nanmean(daywise_average_post, axis = (0,2))+sem(np.nanmean(daywise_average_post, axis = 2), axis = 0, nan_policy='omit'),
        color = colors[cond], alpha = 0.3
        )   

    # ax.set_xticks(
    #     np.arange(daywise_average_post.shape[1]+daywise_average_pre.shape[1], step=5),
    #     np.arange(-daywise_average_pre.shape[1],daywise_average_post.shape[1],5)
    #     )
    ax.set_xlim(-1,(days_back)*2)
    ax.set_ylim(0.4, 0.9)
    ax.set_xticks(
        np.arange(-1, (days_back)*2, step=5),
        np.arange(-days_back-1,days_back+1,5)
        )


    ax.axvline(days_back, color = 'k', alpha = 0.5, linewidth = 0.75, linestyle = '-.')
    prob_comb = np.array(list(permutations([80, 46, 15, 10]))+list(permutations([46, 80, 46, 15])))
    ax.axhline(np.mean(prob_comb)/100, color = 'k', linestyle = '--')
    ax.set_xlabel('Days from lesion')
    ax.set_ylabel('Reward rate')
    # leg = plt.legend(frameon=False, loc = 'upper left')
    # for line, text in zip(leg.get_lines(), leg.get_texts()):
    #     text.set_color(line.get_color())
    pn+=1


plt.tight_layout()

C:\Users\dlab\AppData\Local\Temp\ipykernel_6796\291511003.py:16: RuntimeWarning: Mean of empty slice
  np.nanmean(daywise_average_pre, axis = (0,2)),
C:\Users\dlab\AppData\Local\Temp\ipykernel_6796\291511003.py:21: RuntimeWarning: Mean of empty slice
  np.nanmean(daywise_average_pre, axis = (0,2))-sem(np.nanmean(daywise_average_pre, axis = 2), axis = 0, nan_policy='omit'),
C:\Users\dlab\AppData\Local\Temp\ipykernel_6796\291511003.py:21: SmallSampleWarning: After omitting NaNs, one or more axis-slices of one or more sample arguments is too small; corresponding elements of returned arrays will be NaN. See documentation for sample size requirements.
  np.nanmean(daywise_average_pre, axis = (0,2))-sem(np.nanmean(daywise_average_pre, axis = 2), axis = 0, nan_policy='omit'),
C:\Users\dlab\AppData\Local\Temp\ipykernel_6796\291511003.py:22: RuntimeWarning: Mean of empty slice
  np.nanmean(daywise_average_pre, axis = (0,2))+sem(np.nanmean(daywise_average_pre, axis = 2), axis = 0, nan_policy='om

In [ ]:
# Calculate reward rate per session (session is identified by sess_block)
    # for sess_block, group in animal_data.groupby('sess_block'):
    #     session_date = group['datetime'].iloc[0]
    #     days_since_lesion = (session_date - lesion_date).days
    #     weeks_since_lesion = days_since_lesion / 7
    #     reward_rate = group['reward'].sum() / len(group)
        
#     reward_data.append({
#         'animal': animal,
#         'date': session_date,
#         'days_since_lesion': days_since_lesion,
#         'weeks_since_lesion': weeks_since_lesion,
#         'reward_rate': reward_rate,
#         'sess_block': sess_block
#     })

# reward_df = pd.DataFrame(reward_data)

# # Weekly aggregation: weeks -4 to +4 around lesion
# weekly_data = []
# for animal in reward_df['animal'].unique():
#     animal_reward_df = reward_df[reward_df['animal'] == animal]
    
#     # Group into weeks relative to lesion
#     # Negative weeks: -4 to -1
#     # Positive weeks: +1 to +4
#     for week_offset in range(-4, 5):
#         if week_offset == 0:  # Skip week 0 (transition week)
#             continue
        
#         # Get all sessions in this week window
#         if week_offset < 0:
#             # Before lesion
#             week_mask = (animal_reward_df['weeks_since_lesion'] < week_offset) & \
#                        (animal_reward_df['weeks_since_lesion'] >= week_offset - 1)
#         else:
#             # After lesion
#             week_mask = (animal_reward_df['weeks_since_lesion'] >= week_offset) & \
#                        (animal_reward_df['weeks_since_lesion'] < week_offset + 1)
        
#         week_data = animal_reward_df[week_mask]
#         if len(week_data) > 0:
#             weekly_data.append({
#                 'animal': animal,
#                 'week_relative_to_lesion': week_offset,
#                 'mean_reward_rate': week_data['reward_rate'].mean(),
#                 'sem_reward_rate': week_data['reward_rate'].sem(),
#                 'n_sessions': len(week_data)
#             })

# weekly_df = pd.DataFrame(weekly_data)

# # Plot 1: Weekly reward rates (-4 to +4 weeks)
# fig, ax = plt.subplots(figsize=(10, 6))

# # Calculate mean and SEM across animals for each week
# week_summary = weekly_df.groupby('week_relative_to_lesion').agg({
#     'mean_reward_rate': ['mean', 'sem'],
#     'n_sessions': 'sum'
# }).reset_index()
# week_summary.columns = ['week_relative_to_lesion', 'mean_rr', 'sem_rr', 'n_sessions']

# # Create x-axis positions with gap at lesion
# x_positions = []
# x_labels = []
# x_vals = []
# current_x = 0

# # Pre-lesion weeks (-4 to -1)
# for week in sorted(week_summary[week_summary['week_relative_to_lesion'] < 0]['week_relative_to_lesion'].unique()):
#     x_positions.append(current_x)
#     x_labels.append(f'{int(week)} weeks')
#     x_vals.append(week)
#     current_x += 1

# # Gap for lesion
# lesion_x = current_x + 0.5
# current_x += 1.5

# # Post-lesion weeks (1 to 4)
# for week in sorted(week_summary[week_summary['week_relative_to_lesion'] > 0]['week_relative_to_lesion'].unique()):
#     x_positions.append(current_x)
#     x_labels.append(f'Week {int(week)}')
#     x_vals.append(week)
#     current_x += 1

# # Plot data
# colors = []
# for i, (idx, row) in enumerate(week_summary.iterrows()):
#     if row['week_relative_to_lesion'] < 0:
#         color = 'steelblue'
#     else:
#         color = 'coral'
#     colors.append(color)
#     x_idx = x_positions[i]
#     ax.errorbar(x_idx, row['mean_rr'], yerr=row['sem_rr'], fmt='o', 
#                markersize=8, capsize=5, color=color, linewidth=2)

# # Draw vertical line at lesion
# ax.axvline(x=lesion_x, color='red', linestyle='--', linewidth=2, label='Lesion')

# ax.set_ylabel('Average Reward Rate', fontsize=12)
# ax.set_xlabel('Time Relative to Lesion', fontsize=12)
# ax.set_title('Reward Rate Before and After Lesion (Weekly Averages)', fontsize=14)
# ax.set_xticks(x_positions + [lesion_x])
# ax.set_xticklabels(x_labels + ['Lesion'], rotation=45, ha='right')
# ax.set_ylim([0, 1])
# ax.grid(axis='y', alpha=0.3)
# ax.legend()
# plt.tight_layout()
# plt.show()


# Daily aggregation: days -15 to +15 around lesion
daily_data = []
for animal in reward_df['animal'].unique():
    animal_reward_df = reward_df[reward_df['animal'] == animal]
    
    # Group into days relative to lesion
    for day_offset in range(-15, 16):
        if day_offset == 0:  # Skip day 0 (lesion day)
            continue
        
        # Get all sessions on this day (within 24 hours)
        if day_offset < 0:
            # Before lesion
            day_mask = (animal_reward_df['days_since_lesion'] < day_offset) & \
                      (animal_reward_df['days_since_lesion'] >= day_offset - 1)
        else:
            # After lesion
            day_mask = (animal_reward_df['days_since_lesion'] >= day_offset) & \
                      (animal_reward_df['days_since_lesion'] < day_offset + 1)
        
        day_data_subset = animal_reward_df[day_mask]
        if len(day_data_subset) > 0:
            daily_data.append({
                'animal': animal,
                'day_relative_to_lesion': day_offset,
                'mean_reward_rate': day_data_subset['reward_rate'].mean(),
                'sem_reward_rate': day_data_subset['reward_rate'].sem(),
                'n_sessions': len(day_data_subset)
            })

daily_df = pd.DataFrame(daily_data)

# Plot 2: Daily reward rates (-15 to +15 days)
fig, ax = plt.subplots(figsize=(14, 6))

# Calculate mean and SEM across animals for each day
day_summary = daily_df.groupby('day_relative_to_lesion').agg({
    'mean_reward_rate': ['mean', 'sem'],
    'n_sessions': 'sum'
}).reset_index()
day_summary.columns = ['day_relative_to_lesion', 'mean_rr', 'sem_rr', 'n_sessions']

# Create x-axis positions with gap at lesion
x_positions = []
x_labels = []
x_vals = []
current_x = 0

# Pre-lesion days (-15 to -1)
for day in sorted(day_summary[day_summary['day_relative_to_lesion'] < 0]['day_relative_to_lesion'].unique()):
    x_positions.append(current_x)
    x_labels.append(f'{int(day)}')
    x_vals.append(day)
    current_x += 1

# Gap for lesion
lesion_x = current_x + 0.5
current_x += 1.5

# Post-lesion days (1 to 15)
for day in sorted(day_summary[day_summary['day_relative_to_lesion'] > 0]['day_relative_to_lesion'].unique()):
    x_positions.append(current_x)
    x_labels.append(f'{int(day)}')
    x_vals.append(day)
    current_x += 1

# Plot data
for i, (idx, row) in enumerate(day_summary.iterrows()):
    if row['day_relative_to_lesion'] < 0:
        color = 'steelblue'
    else:
        color = 'coral'
    x_idx = x_positions[i]
    ax.errorbar(x_idx, row['mean_rr'], yerr=row['sem_rr'], fmt='o', 
               markersize=5, capsize=3, color=color, linewidth=1, alpha=0.7)

# Draw vertical line at lesion
ax.axvline(x=lesion_x, color='red', linestyle='--', linewidth=2, label='Lesion')

ax.set_ylabel('Average Reward Rate', fontsize=12)
ax.set_xlabel('Days Relative to Lesion', fontsize=12)
ax.set_title('Reward Rate Before and After Lesion (Daily Averages)', fontsize=14)

# Set x-tick labels (show every 3rd label for clarity)
tick_positions = x_positions[::3] + [lesion_x]
tick_labels = x_labels[::3] + ['Lesion']
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels, rotation=45, ha='right')

ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


C:\Users\dlab\AppData\Local\Temp\ipykernel_6796\2007472009.py:94: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# Verify data summary
print(f"Total animals with lesion dates: {len(lesion_dates)}")
print(f"Total reward data points: {len(reward_df)}")
print(f"\nAnimals with data (first 10):")
for i, animal in enumerate(list(reward_df['animal'].unique())[:10]):
    animal_data = reward_df[reward_df['animal'] == animal]
    print(f"  {animal}: {len(animal_data)} sessions, days range: {animal_data['days_since_lesion'].min()} to {animal_data['days_since_lesion'].max()}")

print(f"\nWeekly data points: {len(weekly_df)}")
print(f"Daily data points: {len(daily_df)}")

# Show sample of weekly data
print("\nSample weekly data:")
print(weekly_df.head(10))


Total animals with lesion dates: 40
Total reward data points: 8106

Animals with data (first 10):
  Chikorita: 187 sessions, days range: -171 to -16
  Eevee: 233 sessions, days range: -184 to -30
  Blissey: 383 sessions, days range: -226 to 6
  Darkrai: 256 sessions, days range: -231 to -14
  test05022023: 193 sessions, days range: -227 to 30
  Hoppip: 140 sessions, days range: -79 to -15
  Kirlia: 188 sessions, days range: -78 to 1
  Inkay: 319 sessions, days range: -142 to -14
  Jirachi: 334 sessions, days range: -143 to 70
  Mesprit: 332 sessions, days range: -179 to 9

Weekly data points: 137
Daily data points: 159

Sample weekly data:
      animal  week_relative_to_lesion  mean_reward_rate  sem_reward_rate  \
0  Chikorita                       -3          0.570301         0.032432   
1  Chikorita                       -2          0.663707         0.038610   
2      Eevee                       -4          0.637489         0.045406   
3    Blissey                       -4          0

# Bias analysis

In [ ]:
# stolen from Sanika
#plot number of times each port was chosen/total trials in block as a function of rewprob at that port, for all trials indf; average across blocks
port_choices = [1, 2, 3, 4]
#group by unique rewprob values, find unique rewprob values across all ports first
colors_ports = ['red', 'green', 'blue', 'purple']
unique_rewprobs = np.unique(sessdf['trial_rewprob'])
unique_rewprobs = np.unique(hsb_data['rewprob'])

# print(unique_rewprobs)
#remove nan from unique_rewprobs
unique_rewprobs = unique_rewprobs[~np.isnan(unique_rewprobs)]
# print(unique_rewprobs)
#remove rows with nan rewprob
# sessdf = sessdf[~sessdf['rewprob'].isna()]
# plt.figure(figsize=(20, 10))

################## get dataset for which to plot bias ###########
cond = 'str'
mask = (sessdf.task == cond) #& (sessdf.animal.isin(block_an_dict['cued']+block_an_dict['uncued'])) & (sessdf.sess_bin >= 4) #& (sessdf.block_group == 1)
mask = (hsb_data.task == cond) 
# nanimals = sessdf[mask].animal.nunique()
nanimals = hsb_data[mask].animal.nunique()


################# plotting #####################################
pn = 1
plt.figure(figsize = (10,7))

#remove rows with nan rewprob
for an, tempdf in hsb_data[mask].groupby('animal'):

    tempdf = tempdf[~tempdf['rewprob'].isna()]

    #for every block, calculate number of times each port was chosen/total trials in block for each unique rewprob value
    nblocks = tempdf['session'].nunique()
    unique_rewprobs = np.unique(tempdf['rewprob'])
    unique_rewprobs = unique_rewprobs[~np.isnan(unique_rewprobs)]
    port_choice_matrix = np.zeros((len(unique_rewprobs), len(port_choices), nblocks))
    #for all blocks
    #save all rewprobfull per block in a list
    for block, num in enumerate(tempdf['session'].unique()):
        block_data = tempdf[tempdf['session'] == num]
        #for all ports
        for p, port in enumerate(port_choices):
            port_data = block_data[block_data['port'] == port]
            #for all unique rewprobs
            for r, rewprob in enumerate(unique_rewprobs):
                n_choices = len(port_data[port_data['rewprob'] == rewprob])
                port_choice_matrix[r, p, block] = n_choices

    bias_matrix = np.zeros((port_choice_matrix.shape[0], port_choice_matrix.shape[1]))
    counter = np.zeros((port_choice_matrix.shape[0], port_choice_matrix.shape[1]))

    for p in range(port_choice_matrix.shape[1]):
        for r in range(port_choice_matrix.shape[0]):
            bias_matrix[r, p] = 0
            for b in range(port_choice_matrix.shape[2]):
                ntrls = np.sum(port_choice_matrix[:,:,b])
                if ntrls != 0:
                    bias_matrix[r, p] += port_choice_matrix[r, p, b] / ntrls
                if port_choice_matrix[r, p, b] > 0:
                    counter[r, p] += 1

    #divide fin_mat by counter
    for p in range(port_choice_matrix.shape[1]):
        for r in range(port_choice_matrix.shape[0]):
            if counter[r, p] > 0:
                bias_matrix[r, p] = bias_matrix[r, p] / counter[r, p]
    print('Bias matrix:', bias_matrix)


    #plot as line plots for each port
    # plt.figure(figsize=(7, 6))
    ax = plt.subplot(4,7,pn)
    for p, port in enumerate(port_choices):
        ax.plot([0,1,2,3], bias_matrix[:, p], label=f'Port {port}', marker='o', linewidth=2, color=colors_ports[p])
    ax.set_xlabel('Rew. prob.')
    ax.set_xticks([0,1,2,3], labels=[int(r) for r in unique_rewprobs])
    # ax.set_ylabel(f'Fraction chosen (averaged across {nblocks} blocks)')
    ax.set_ylabel(f'{nblocks} blocks')
    ax.set_title(f'{an}')
    ax.set_ylim(0, 0.9)
    pn+=1
    
plt.tight_layout()

    # n_rows, n_cols = bias_matrix.shape
    # rpf = [10,15,46,80]
    # labels = [i+1 for i in range(n_cols)]

    # # Angles for radar
    # angles = np.linspace(0, 2*np.pi, n_cols, endpoint=False)
    # angles = np.concatenate([angles, [angles[0]]])  # close loop

    # fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
    # for i in range(n_rows):
    #     values = bias_matrix[i]
    #     values = np.concatenate([values, [values[0]]])  # close loop

    #     ax.plot(angles, values, linewidth=2, label=f'prob {rpf[i]}', color=colors_ports[i])
    #     # ax.fill(angles, values, alpha=0.15)

    # ax.set_thetagrids(np.degrees(angles[:-1]), labels)
    # #reduce the yticks to 3
    # # ax.set_yticks([])
    # ax.legend(loc='upper right', bbox_to_anchor=(1.25, 1.1))
    # plt.show()

Bias matrix: [[0.06494296 0.05914116 0.08195179 0.05173698]
 [0.06310336 0.06773632 0.11206134 0.04884528]
 [0.18498843 0.14441011 0.37162824 0.11589102]
 [0.76340264 0.61646558 0.85494773 0.583495  ]]
Bias matrix: [[0.09406571 0.13403269 0.1004383  0.07659053]
 [0.12022489 0.15920754 0.12650895 0.08383014]
 [0.27031397 0.56630252 0.29650079 0.23303671]
 [0.50901617 0.73812711 0.60349081 0.66374297]]
Bias matrix: [[0.05871822 0.04737651 0.06417405 0.05891269]
 [0.05896504 0.07213547 0.07299523 0.05539487]
 [0.1738672  0.22200145 0.22793455 0.11780724]
 [0.72655949 0.75418949 0.76481747 0.69363064]]
Bias matrix: [[0.0705844  0.08966961 0.07593678 0.05041175]
 [0.07078351 0.09643287 0.07110962 0.04984585]
 [0.24160591 0.22786856 0.28568366 0.07641842]
 [0.80956904 0.7076102  0.69043239 0.49226186]]
Bias matrix: [[0.03999495 0.04435599 0.08818695 0.07871469]
 [0.05607511 0.04158927 0.11457992 0.07540143]
 [0.21855011 0.10931187 0.25459558 0.16323655]
 [0.74772015 0.52797808 0.77339968 0.7

# predict avg performance based on reward probability - 25-09-2024

In [ ]:
#################################### perf grouped by rp ######################################
# set_cwd('/home/rishika/sim/')
%matplotlib qt
window = 7
trialsinsess = 100
fig = plt.figure(figsize=(20,10))
# sessdf['rr'] = sessdf.groupby(['animal', 'session'], as_index = False).reward.rolling(window, center=True).mean().reward
rr_mean_dict_rp = {}
# rem_les = sessdf[~(sessdf.task.isin(['dms', 'dls', 'sham']))]

colors ={'str':'#028f1e55', 'unstr':'#fb7d0755', 'dls':'xkcd:gold', 'dms':'xkcd:red', 'sham':'b', 'ds':'xkcd:orange'} 
for i in ['unstr']:

    for ind, (rpfull_string, group) in enumerate(sessdf.groupby('rewprobfull')):
        ax = plt.subplot(5, 8,  ind+1)
        
        # remove duplicates
        # if animal == 'test05022023':
        #     group = group.loc[~(sessdf.groupby('animal').get_group('test05022023')
        #                               .duplicated(subset = 'trialstart', keep = False))]
        
        # filter by sessions>= length
        filtered = (group[group.task.isin([i])]
                    .groupby(['animal','session'])).filter(lambda x: x.reward.size >= trialsinsess)

        # take only those sessions
        filtered = filtered.groupby(['animal','session']).head(trialsinsess)
        
        # if animal=='Nidorina':
        #     filtered = filtered[filtered.datetime.astype('datetime64[ns]')<datetime.datetime(2024, 5, 8)]
        
        if filtered.empty==True:
            continue

        rp_type = filtered.assigned.iloc[0]
        
        # convert to numpy array
        g = filtered.groupby(['animal','session']).cumcount()
        L = np.array(filtered.set_index(['animal','session',g])
               .unstack(fill_value=0)
               .stack().groupby(level = ['animal', 'session'])
               .apply(lambda x: x.rr.values.tolist())
               .tolist())
        
        # mean
        rr_mean = np.mean(L, axis=0)
        rr_mean_dict_rp[rpfull_string, rp_type] = rr_mean
        ax.plot(rr_mean, label = i, c = 'k')
        ax.set_title(rpfull_string.strip('[]'))
        # ax.legend()
        ax.set_facecolor(colors[rp_type])
        # ax.set_xlabel('Trial')
        ax.axhline(np.array(rpfull_string.strip('[]').split(), dtype = int).mean()/100, linestyle = '--', color = 'k')
        ax.set_ylim(0.3, 0.8)

sns.despine()
fig.supylabel('Reward rate, averaged across sessions')
plt.tight_layout()

AttributeError: 'DataFrame' object has no attribute 'assigned'

In [ ]:
ax = plt.subplot(111)

dat = pd.DataFrame(rr_mean_dict_rp).mean().sort_values().reset_index()
dat.rename(columns = {'level_0':'rp', 'level_1':'rp_type', 0:'mu'}, inplace = True)
sns.barplot(dat, x = 'rp', y = 'mu', hue = 'rp_type', palette = [color1, color2])
plt.xticks(rotation=90, ha='right')
plt.tight_layout()
sns.despine()

In [ ]:
ax = plt.subplot(111)

dat = pd.DataFrame(rr_mean_dict_rp).head(50).mean().sort_values().reset_index()
dat.rename(columns = {'level_0':'rp', 'level_1':'rp_type', 0:'mu'}, inplace = True)
sns.barplot(dat, x = 'rp', y = 'mu', hue = 'rp_type', palette = [color2, color1])
plt.xticks(rotation=90, ha='right')
plt.tight_layout()
plt.title('first 50')
sns.despine()

In [ ]:
ax = plt.subplot(111)

dat = pd.DataFrame(rr_mean_dict_rp).tail(50).mean().sort_values().reset_index()
dat.rename(columns = {'level_0':'rp', 'level_1':'rp_type', 0:'mu'}, inplace = True)
sns.barplot(dat, x = 'rp', y = 'mu', hue = 'rp_type', palette = [color1, color2])
plt.xticks(rotation=90, ha='right')
plt.tight_layout()
plt.title('last 50')
sns.despine()

C:\Users\dlab\miniforge3\envs\rishika\lib\site-packages\ipykernel\eventloops.py:145: UserWarning: The figure layout has changed to tight
  el.exec() if hasattr(el, "exec") else el.exec_()


In [ ]:
ax = plt.subplot(111)

dat = pd.DataFrame(rr_mean_dict_rp).tail(15).max().sort_values().reset_index()
dat.rename(columns = {'level_0':'rp', 'level_1':'rp_type', 0:'mu'}, inplace = True)
sns.barplot(dat, x = 'rp', y = 'mu', hue = 'rp_type', palette = [color1, color2])
plt.xticks(rotation=90, ha='right')
plt.tight_layout()
plt.title('max of last 15')
sns.despine()

C:\Users\dlab\miniforge3\envs\rishika\lib\site-packages\ipykernel\eventloops.py:145: UserWarning: The figure layout has changed to tight
  el.exec() if hasattr(el, "exec") else el.exec_()


In [ ]:
ax = plt.subplot(111)

dat = pd.DataFrame(rr_mean_dict_rp).median().sort_values().reset_index()
dat.rename(columns = {'level_0':'rp', 'level_1':'rp_type', 0:'mu'}, inplace = True)
sns.barplot(dat, x = 'rp', y = 'mu', hue = 'rp_type', palette = [color2, color1])
plt.xticks(rotation=90, ha='right')
plt.tight_layout()
plt.title('median')
sns.despine()

C:\Users\dlab\miniforge3\envs\rishika\lib\site-packages\ipykernel\eventloops.py:145: UserWarning: The figure layout has changed to tight
  el.exec() if hasattr(el, "exec") else el.exec_()


In [ ]:
# plot average reponse
plt.figure(figsize = (5,4.5))
temp_unstr, temp_str = [], []

# set properties
smooth = True
window = 3
label1 = 'Unstructured'
label2 = 'Structured'
color1 = 'xkcd:azure'
color2 = 'xkcd:slate grey'
# color1 = '#7a2048', 
color1 = 'xkcd:pumpkin orange'
color2 = 'xkcd:emerald green'
# color2 = '#408ec6'

# set data for plotting
# fish = ['Blissey', 'Darkrai', 'Mesprit', 'Quilava', 'Raltz', 'Inkay']
# fish = ['Hoppip', 'Kirlia', 'Nidorina', 'Togepi']
# fish = ['test05022023', 'Jirachi', 'Goldeen', 'Phione']
fish = ['test05022023', 'Blissey', 'Chikorita', 'Darkrai', 'Eevee']
for (animal, env) in rr_mean_dict_rp.keys():
    # if animal in fish:
    if env == 'unstr':
        temp_unstr.append(rr_mean_dict_rp[(animal, env)])
    elif (env == 'str'):
        temp_str.append(rr_mean_dict_rp[(animal, env)])

# calculate performance index
temp_unstr = np.array(temp_unstr)
temp_str = np.array(temp_str)

# smoothing details
if smooth == True:           
    for i, row in enumerate(temp_unstr):
        temp_unstr[i] = np.convolve(row, np.ones(window)/window, 'same')
        temp_str[i] = np.convolve(temp_str[i], np.ones(window)/window, 'same')
    
    # works only for odd windows
    temp_unstr[:, 0:(window//2)] = np.nan
    temp_unstr[:, (trialsinsess-window//2):trialsinsess] = np.nan
    
    temp_str[:, 0:(window//2)] = np.nan
    temp_str[:, (trialsinsess-window//2):trialsinsess] = np.nan

# take mean of smoothed data   
mean_unstr = (np.mean(np.array(temp_unstr), axis = 0))
mean_str = (np.mean(np.array(temp_str), axis = 0))

# plot line 1
plt.plot(np.arange(trialsinsess), mean_unstr, color1, label = label1) 
plt.fill_between(np.arange(trialsinsess), mean_unstr+sem(temp_unstr, nan_policy = 'omit'),
                 mean_unstr-sem(temp_unstr, nan_policy = 'omit'), color = color1, alpha = 0.2)

# plot line 2 (or comment)
plt.plot(np.arange(trialsinsess), mean_str, color2, label = label2)
plt.fill_between(np.arange(trialsinsess), mean_str+sem(temp_str, nan_policy = 'omit'), 
                 mean_str-sem(temp_str, nan_policy = 'omit'), color = color2, alpha = 0.2)

# plot significance
# pvals = ttest_rel(temp_str, temp_unstr)[1]
# pvals_ind = [ind for ind, i in enumerate(pvals) if i < 0.05]
# plt.scatter(pvals_ind, [0.65]*len(pvals_ind), marker = '.', color = 'r', alpha = 0.4) #(6, 2, 60) for asterisks

# plot settings
plt.xlabel('Trials in session', fontsize = 'x-large')
plt.ylabel('Performance index', fontsize = 'x-large')
plt.xlim(0, trialsinsess)
plt.axhline(0.625, color='k', linestyle = '--') # chance calc by
# plt.ylim(0.6, 0.85)
plt.xticks([0, 25, 50, 75, 100], [1, '', '', '', 100], color = 'grey', fontsize = 'large')
# plt.yticks([0.6, 0.65, 0.7, 0.75, 0.8, 0.85], ['0.60', '', '','','', 0.85], color = 'grey', fontsize = 'large')
# plt.ylim(0, 0.8)
plt.legend(loc='upper left')
sns.despine()
# sns.set_context('talk')
plt.tight_layout()
# plt.savefig('C:/Users/dlab/OneDrive - Indian Institute of Science/Drawings/awp1/entropy_n_21.png', dpi = 300)

IndexError: index 0 is out of bounds for axis 0 with size 0

# new rewprob-matched analysis

In [ ]:
results = []

# Process for each animal and offset in a single loop
for n in range(-20, 21):
    if n in [-2, -1, 0, 1, 2]:
        continue

    sessdf['entropy_shifted'] = (
        sessdf.groupby(['animal', 'session'])['entropy'].shift(-n)
    )

    ported = (
        sessdf[sessdf['task'] == 'unstr']
        .groupby(['animal', 'session', 'rewprob', 'reward'], as_index=False)['entropy_shifted']
        .mean()
    )

    grouped = ported.groupby(['animal', 'rewprob', 'reward'], as_index=False)['entropy_shifted'].mean()
    grouped['offset'] = n

    # Append to results
    results.append(grouped)

# Combine results
final_df = pd.concat(results, ignore_index=True)

In [ ]:
rew_an_dict, unrew_an_dict, diff_dict = {}, {}, {}
# Add NaNs in the middle
nan_series = pd.Series([np.nan] * 5, index=range(-2, 3))

for an in final_df.animal.unique():
    # calc for each animal, for each rp
    for rewprob in [10, 15, 46, 80]:
        unrew = (final_df[(final_df['rewprob'] == rewprob) & (final_df['reward'] == 0) & (final_df['animal']==an)]
            .set_index('offset')['entropy_shifted'])
        rew = (final_df[(final_df['rewprob'] == rewprob) & (final_df['reward'] == 1) & (final_df['animal']==an)]
            .set_index('offset')['entropy_shifted'])
    
        rew_an_dict[(an, rewprob)] = pd.concat([rew.iloc[:18], nan_series, rew.iloc[18:]])
        unrew_an_dict[(an, rewprob)] = pd.concat([unrew.iloc[:18], nan_series, unrew.iloc[18:]])
        diff_dict[(an, rewprob)] = unrew_an_dict[(an, rewprob)] - rew_an_dict[(an, rewprob)]
    # if diff_dict[(an, rewprob)].isna().sum() > 0:
    #     print(an, rewprob)
data = pd.DataFrame(diff_dict).T.reset_index().groupby('level_1')[np.arange(-20, 21)].mean()

NameError: name 'pd' is not defined

In [ ]:
colors = sns.color_palette("crest", as_cmap=True)(np.linspace(0, 1, 4))

for ind, rewprob in enumerate([10, 15, 46, 80]):
    plt.plot(data.T[ind], '.-', color = colors[ind], label = rewprob)
plt.legend()
sns.despine()
plt.tight_layout()

In [ ]:
# make a filter for sessions to have one of each rp in that session
# rpfull = sessdf.rewprobfull
# sessdf['num_rp'] = [len(np.unique(np.array(x.strip('[]').split(), dtype = int))) for x in rpfull] 

# mask = (
#     sessdf
#     .groupby(['animal', 'session'], as_index=False)
#     .apply(lambda g: g['rewprob'].nunique() == g['num_rp'].iloc[0])
# )
results = []
mask.columns = ['animal', 'session', 'is_valid']
cond = 'sham'
an_list = an_dict[cond]

filtered_sessdf = sessdf.merge(mask[mask['is_valid']], on=['animal', 'session'], how='inner')
filtered_sessdf = filtered_sessdf[(filtered_sessdf.animal.isin(an_list))]
# Process for each animal and offset in a single loop
for n in range(-20, 21):
    if n in [-2, -1, 0, 1, 2]:
        continue

    filtered_sessdf['entropy_shifted'] = (
        filtered_sessdf.groupby(['animal', 'session'])['entropy'].shift(-n)
    )

    ported = (
        filtered_sessdf[filtered_sessdf['task'] == 'unstr']
        .groupby(['animal', 'session', 'rewprob', 'reward'], as_index=False)['entropy_shifted']
        .mean()
    )

    grouped = ported.groupby(['animal', 'rewprob', 'reward'], as_index=False)['entropy_shifted'].mean()
    grouped['offset'] = n

    # Append to results
    results.append(grouped)

# Combine results
final_df = pd.concat(results, ignore_index=True)

rew_an_dict, unrew_an_dict, diff_dict = {}, {}, {}
# Add NaNs in the middle
nan_series = pd.Series([np.nan] * 5, index=range(-2, 3))

for an in final_df.animal.unique():
    # calc for each animal, for each rp
    for rewprob in [10, 15, 46, 80]:
        unrew = (final_df[(final_df['rewprob'] == rewprob) & (final_df['reward'] == 0) & (final_df['animal']==an)]
            .set_index('offset')['entropy_shifted'])
        rew = (final_df[(final_df['rewprob'] == rewprob) & (final_df['reward'] == 1) & (final_df['animal']==an)]
            .set_index('offset')['entropy_shifted'])
    
        rew_an_dict[(an, rewprob)] = pd.concat([rew.iloc[:18], nan_series, rew.iloc[18:]])
        unrew_an_dict[(an, rewprob)] = pd.concat([unrew.iloc[:18], nan_series, unrew.iloc[18:]])
        diff_dict[(an, rewprob)] = unrew_an_dict[(an, rewprob)] - rew_an_dict[(an, rewprob)]
data = pd.DataFrame(diff_dict).T.reset_index().groupby('level_1')[np.arange(-20, 21)].mean().T.to_numpy()

colors = sns.color_palette("crest", as_cmap=True)(np.linspace(0, 1, 4))

In [ ]:
fig = plt.figure()
for ind, rewprob in enumerate([10, 15, 46, 80]):
    plt.plot(data.T[ind], '.-', color = colors[ind], label = rewprob)
plt.legend()
# plt.xticks(np.arange(-20, 20))
plt.xlabel('Trials')
plt.ylabel('Entropy no reward trials - reward trials')
plt.ylim(-0.1, 0.25)
sns.despine()
plt.tight_layout()

In [ ]:
%matplotlib qt
plt.figure(figsize = (20, 10))
itera = 1
pn=1
col = 0
ax = plt.subplot(7,4,pn)
for key in diff_dict.keys():
    
    if itera % 4==0:
        ax = plt.subplot(7,4,pn)
        ax.set_title(key)
        pn+=1
        col = 0
    ax.plot(diff_dict[key], '.-', color = colors[col])
    col += 1
    itera +=1
sns.despine()
plt.tight_layout()

### why do DLS-lesioned animals only choose a few ports? are they able to perform perfectly (pre-lesion levels) in those sessions?

In [ ]:
sessdf['high_port'] = sessdf['rewprobfull'].apply(lambda x: np.where(np.array(x.strip('[]').split())=='80')[0][0]+1)

In [ ]:
plt.figure()
pn = 1
trialsinsess = 25
rr_port_dict = {}
for rat in sessdf[sessdf.task == 'dls'].animal.unique():
    ax = plt.subplot(3, 3, pn)
    raltzdf = sessdf[(sessdf.task == 'dls') & (sessdf.animal == rat) & (sessdf.shift_t0 == 1)]
    raltzdf = raltzdf.groupby('session').filter(lambda x: x.reward.size >= trialsinsess).groupby('session').head(trialsinsess)
    if raltzdf.empty==True:
        continue
    ax.plot(rr_mean_dict[(rat, 'unstr')], 'k', label = 'all')
    # ax.set_xlim(trialsinsess)
    # on this plot, plot mean RR curves for different highest rewprob along with transition matrix on right in smol
    # filter by sessions>= length
    for n in range(1,5):
        filtered = (sessdf[(sessdf.animal == rat) & (sessdf.high_port == n) & (sessdf.task == 'dls')]
                    .groupby('session')).filter(lambda x: x.reward.size >= trialsinsess)
        
        # take only those sessions
        filtered = filtered.groupby('session').head(trialsinsess)
        if filtered.empty==True:
            continue
            
        # convert to numpy array
        g = filtered.groupby('session').cumcount()
        L = np.array(filtered.set_index(['session', g])
               .unstack(fill_value=0)
               .stack().groupby(level=0)
               .apply(lambda x: x.rr.values.tolist())
               .tolist())
        rr_port_dict[n] = np.mean(L, axis = 0)
        ax.plot(rr_port_dict[n], label = n)
        ax.set_title(rat)
    ins = ax.inset_axes([0.7,0.7,0.25,0.25])
    
    sns.heatmap(pd.crosstab(raltzdf.port, raltzdf.choice_t1, normalize = 'index', dropna = False), 
                ax = ins,
                cmap = 'YlGnBu',
                annot = False,
                fmt = '.2f',
                vmin = 0.0,
                vmax = 0.7,
                square = True,
                mask = np.eye(4),
                xticklabels = np.arange(1,5),
                yticklabels = np.arange(1,5))
    ax.patch.set_facecolor('white')
    sns.despine()
    ax.legend()
    pn+=1

NameError: name 'rr_mean_dict' is not defined

# Early v/s late sessions - weekly analysis

In [ ]:
# assign week numbers to each session based on the date
from utils.dfLoading import subset_trials, add_shift_info, add_block_groups

# sessdf = add_block_groups(sessdf)
# sessdf['datetime'] = pd.to_datetime(sessdf['datetime'])

# # ## week relative to each animal-task first date
# sessdf['week_task'] = sessdf.groupby(['animal', 'task'])['datetime'].transform(
#     lambda x: ((x - x.min()).dt.days // 7) + 1
# )

# hsb_data_sorted = hsb_data.sort_values(['animal','eptime', 'session'])
# hsb_data_sorted = add_block_groups(hsb_data_sorted)
hsb_data_sorted = add_shift_info(hsb_data)
window = 5
hsb_data['rr'] = hsb_data.groupby(['animal', 'session'], as_index = False).reward.rolling(window, center=True).mean().reward
hsb_data['entropy'] = (hsb_data.groupby(['animal', 'session'], as_index = False)
                     .port
                     .rolling(window, center=True)
                     .apply(lambda x: entropy(calc_prob(x), base = 2))
                     .port)
# hsb_data_sorted['datetime'] = pd.to_datetime(hsb_data_sorted['datetime'])

# ## week relative to each animal-task first date
# hsb_data_sorted['week_task'] = hsb_data_sorted.groupby(['animal', 'task'])['datetime'].transform(
#     lambda x: ((x - x.min()).dt.days // 7) + 1
# )

### otherwise, 
# bin week numbers for each animal into 10 
# sessdf['week_bin'] = sessdf.groupby('animal')['week'].transform(lambda x: pd.qcut(x, 7, labels = False, duplicates='drop'))
# Mark each session number with a bin number. Each bin has 50 sessions.
# bin_size = 50
# sessdf['sess_bin'] = sessdf.groupby(['animal', 'task'])['session'].transform(lambda x: pd.cut(x, bins=range(0, x.max() + bin_size, bin_size), labels=False, right=False)+1)
trialsinsess = 100
# sessdf['entropy'] = (sessdf.groupby(['animal', 'session'], as_index = False)
#                      .port
#                      .rolling(window, center=True)
#                      .apply(lambda x: entropy(calc_prob(x), base = 2))
#                      .port)
print('done now')

# set subset_dict for extracting weeks/sess_bins

# subset_df_sessbin_dict = {}
# for sess_bin in sessdf.sess_bin.unique():
#     subset_df = sessdf[sessdf.task == 'unstr']
#     mask = (subset_df.sess_bin == sess_bin) & (subset_df.animal.isin(an_dict['unstr'])) #& (subset_df.animal.isin(block_an_dict['cued']+block_an_dict['uncued']))
    
#     subset_df = add_shift_info(subset_df[mask])
#     subset_df_sessbin_dict[sess_bin] = subset_trials(subset_df, trialsinsess=trialsinsess, head_trials=trialsinsess)


subset_df_week_dict = {}
for weeks in sessdf.week_task.unique():
    subset_df = sessdf[sessdf.task == 'unstr']

    mask = (subset_df.week_task == weeks) & (subset_df.animal.isin(block_an_dict['cued']+block_an_dict['uncued']))
    subset_df = add_shift_info(subset_df[mask])
    subset_df = subset_df[subset_df.block_group == 1]
    subset_df_week_dict[weeks] = subset_trials(subset_df, trialsinsess=trialsinsess, head_trials=trialsinsess)

# subset_df_week_dict_hsb = {}

# for weeks in hsb_data.week_task.unique():
#     subset_df = hsb_data[hsb_data.task == 'str']

#     mask = (subset_df.week_task == weeks) & (subset_df.animal.isin(an_dict['hsb_str']))
#     subset_df = add_shift_info(subset_df[mask])
#     subset_df = subset_df[subset_df.block_group == 1]

#     subset_df_week_dict_hsb[weeks] = subset_trials(subset_df, trialsinsess=trialsinsess, head_trials=trialsinsess)

c:\Users\dlab\banditAnalysis\utils\dfLoading.py:177: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['shift_t0'] = (df['choice_t1']==df['port']).replace({True: 0, False: 1})
c:\Users\dlab\banditAnalysis\utils\dfLoading.py:178: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['shift_t1'] = (df['choice_t2']==df['port']).replace({True: 0, False: 1})


done now


c:\Users\dlab\banditAnalysis\utils\dfLoading.py:171: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['choice_t1'] = df.groupby(['animal','block_num']).port.shift(-1)
c:\Users\dlab\banditAnalysis\utils\dfLoading.py:172: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['choice_t2'] = df.groupby(['animal','block_num']).port.shift(-2)
c:\Users\dlab\banditAnalysis\utils\dfLoading.py:175: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_in

In [122]:
### calculate some metric for given data
from utils.dfAnalysisHelpers import get_sessionwise_averages, calc_dist_metric
metric = 'rr'
metric_dict = {}
chance_dict = {}
for key, data in subset_df_week_dict.items():
    for animal, group in data.groupby('animal'):
        metric_dict[(animal, key)] = np.array(list(get_sessionwise_averages(group, metric= metric, tasks = ['unstr']).values())).flatten()
    if data.empty:
        continue
#     mask = (data.shift_t0 == 1)
#     dist_metric = calc_dist_metric(data, mask)
#     metric_dict[key] = dist_metric.groupby('animal').d_value.mean()
#     chance_dict[key] = dist_metric.groupby('animal').d_chance.mean()

# for key, data in subset_df_week_dict_hsb.items():
#     for animal, group in data.groupby('animal'):
#         metric_dict[(animal, key)] = np.array(list(get_sessionwise_averages(group, metric= metric, tasks = ['str']).values())).flatten()
#     if data.empty:
#         continue
    # mask = (data.shift_t0 == 1)
    # dist_metric = calc_dist_metric(data, mask)
    # metric_dict[key] = dist_metric.groupby('animal').d_value.mean()
    # chance_dict[key] = dist_metric.groupby('animal').d_chance.mean()

c:\Users\dlab\banditAnalysis\utils\dfAnalysisHelpers.py:18: RuntimeWarning: Mean of empty slice
  metric_dict[an, task] = np.nanmean(L, axis = 0)
c:\Users\dlab\banditAnalysis\utils\dfAnalysisHelpers.py:18: RuntimeWarning: Mean of empty slice
  metric_dict[an, task] = np.nanmean(L, axis = 0)
c:\Users\dlab\banditAnalysis\utils\dfAnalysisHelpers.py:18: RuntimeWarning: Mean of empty slice
  metric_dict[an, task] = np.nanmean(L, axis = 0)
c:\Users\dlab\banditAnalysis\utils\dfAnalysisHelpers.py:18: RuntimeWarning: Mean of empty slice
  metric_dict[an, task] = np.nanmean(L, axis = 0)
c:\Users\dlab\banditAnalysis\utils\dfAnalysisHelpers.py:18: RuntimeWarning: Mean of empty slice
  metric_dict[an, task] = np.nanmean(L, axis = 0)
c:\Users\dlab\banditAnalysis\utils\dfAnalysisHelpers.py:18: RuntimeWarning: Mean of empty slice
  metric_dict[an, task] = np.nanmean(L, axis = 0)
c:\Users\dlab\banditAnalysis\utils\dfAnalysisHelpers.py:18: RuntimeWarning: Mean of empty slice
  metric_dict[an, task] = np

In [103]:
# distance metric over weeks
%matplotlib qt
from scipy.stats import sem
from utils.plotSettings import *

######################## plotting ##################
fig = plt.figure(figsize = (5,4.5))
smooth = True
max_bin = 8
colors = plt.cm.cool_r(np.linspace(0, 1, max_bin))

for sess_bin in range(1,max_bin):
    temp_unstr = []
    chance_unstr = []
    temp_unstr.append(metric_dict[sess_bin])
    chance_unstr.append(chance_dict[sess_bin])
    temp_unstr = np.array(temp_unstr)
    mean_unstr = np.nanmean(temp_unstr)
    chance_unstr = np.nanmean(chance_unstr)
    plt.bar(sess_bin, chance_unstr, color = 'xkcd:grey', alpha = 0.3)
    plt.bar(sess_bin, mean_unstr, yerr = sem(temp_unstr.flatten(), nan_policy='omit'), color = colors[sess_bin])
plt.xlabel('Weeks in task')
plt.ylim(1)
plt.yticks([1, 1.2, 1.4, 1.6], ['1.0', '', '', 1.6], fontsize = 'large', color = 'grey')
plt.xticks(np.arange(1,max_bin), np.arange(1,max_bin))
plt.ylabel('Switch distance')
plt.tight_layout()

KeyError: 1

In [123]:
fig = plt.figure(figsize = (5,4.5))
smooth = True
max_bin = 19
ax= plt.subplot(111)
ylabel_dict = {'shift_t0':'Switch rate', 'rr':'Reward rate', 'entropy': 'Entropy (bits)'}
# colors = plt.cm.cool_r(np.linspace(0, 1, len(sessdf.sess_bin.unique())+1))
colors = plt.cm.cool_r(np.linspace(0, 1, max_bin))

temp_unstr = []
for sess_bin in range(1,max_bin):
    temp_unstr = []
    for animal in np.unique((np.array(list(metric_dict.keys()))[:, 0])):
        try:
            temp_unstr.append(metric_dict[(animal, sess_bin)])
        except KeyError:
            pass
    temp_unstr = np.array(temp_unstr)
    mean_unstr = np.nanmean(temp_unstr)
    ybars = ax.bar(sess_bin, mean_unstr, yerr= sem(temp_unstr.flatten(), nan_policy = 'omit'), color = colors[sess_bin])
    ax.bar_label(ybars, labels = [temp_unstr.shape[0]], label_type = 'center')

C:\Users\dlab\AppData\Local\Temp\ipykernel_30164\2584044715.py:18: RuntimeWarning: Mean of empty slice
  mean_unstr = np.nanmean(temp_unstr)
C:\Users\dlab\AppData\Local\Temp\ipykernel_30164\2584044715.py:19: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  ybars = ax.bar(sess_bin, mean_unstr, yerr= sem(temp_unstr.flatten(), nan_policy = 'omit'), color = colors[sess_bin])


In [104]:
# generic metric over weeks
%matplotlib qt
from scipy.stats import sem
from utils.plotSettings import *

######################## plotting ##################
fig = plt.figure(figsize = (5,4.5))
smooth = True
max_bin = 8
ylabel_dict = {'shift_t0':'Switch rate', 'rr':'Reward rate', 'entropy': 'Entropy (bits)'}
# colors = plt.cm.cool_r(np.linspace(0, 1, len(sessdf.sess_bin.unique())+1))
colors = plt.cm.cool_r(np.linspace(0, 1, max_bin))

temp_unstr = []
for sess_bin in range(1,max_bin):
    temp_unstr = []
    for animal in np.unique((np.array(list(metric_dict.keys()))[:, 0])):
        try:
            temp_unstr.append(metric_dict[(animal, sess_bin)])
        except KeyError:
            pass

    # calculate performance index
    # temp_unstr = (1-np.array(temp_unstr)/100)
    temp_unstr = np.array(temp_unstr)

    # smoothing details
    if smooth:
        for i, row in enumerate(temp_unstr):
            temp_unstr[i] = np.convolve(row, np.ones(window)/window, 'same')
        
        # works only for odd windows
        temp_unstr[:, 0:(window//2)] = np.nan
        temp_unstr[:, (trialsinsess-window//2):trialsinsess] = np.nan

    # temp_unstr = [temp_unstr[i] for i in [0, 2, 6]]


    # take mean of smoothed data   
    mean_unstr = (np.nanmean(np.array(temp_unstr), axis = 0))

    # plot line 1
    plt.plot(np.arange(trialsinsess), mean_unstr, color = colors[sess_bin], label = f'wk {sess_bin}')#, n {temp_unstr.shape[0]}')#, marker = '.') 
    plt.fill_between(np.arange(trialsinsess), mean_unstr+sem(temp_unstr, nan_policy = 'omit'),
                    mean_unstr-sem(temp_unstr, nan_policy = 'omit'), alpha = 0.2, color = colors[sess_bin])
    # ci+=1
plt.xlabel('Trials in session', fontsize = 'x-large')
plt.ylabel(f'{ylabel_dict[metric]}', fontsize = 'x-large')
plt.xlim(0, trialsinsess)
# plt.ylim(0.1, 0.6)
sns.despine()


plt.xticks([0,25, 50, 75, 100], [1, '','', '', 100], fontsize = 'large', color = 'grey')
# plt.ylim(0.35, 0.8)
# plt.yticks([0.6, 0.7,0.8, 0.9], [0.6, '', '', 0.9], fontsize = 'large', color = 'grey')
# plt.legend(['Naive','_fish_', 'Intermediate','_fish_', 'Expert'], loc = 'upper left')
leg = plt.legend(frameon=False)
for line, text in zip(leg.get_lines(), leg.get_texts()):
    text.set_color(line.get_color())
# plt.title(f'bin_size={bin_size}')
plt.tight_layout()

C:\Users\dlab\AppData\Local\Temp\ipykernel_30164\4195982791.py:40: RuntimeWarning: Mean of empty slice
  mean_unstr = (np.nanmean(np.array(temp_unstr), axis = 0))
C:\Users\dlab\AppData\Local\Temp\ipykernel_30164\4195982791.py:44: SmallSampleWarning: After omitting NaNs, one or more axis-slices of one or more sample arguments is too small; corresponding elements of returned arrays will be NaN. See documentation for sample size requirements.
  plt.fill_between(np.arange(trialsinsess), mean_unstr+sem(temp_unstr, nan_policy = 'omit'),
C:\Users\dlab\AppData\Local\Temp\ipykernel_30164\4195982791.py:45: SmallSampleWarning: After omitting NaNs, one or more axis-slices of one or more sample arguments is too small; corresponding elements of returned arrays will be NaN. See documentation for sample size requirements.
  mean_unstr-sem(temp_unstr, nan_policy = 'omit'), alpha = 0.2, color = colors[sess_bin])


In [ ]:
# transition matrices for first n bins of sessions 
ind = 1
trialsinsess = 100
fig = plt.figure(figsize = (10, 4))
tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
# tempdf = tempdf[~(tempdf.animal.isin(['Kakuna', 'Finneon']))]
an_list = tempdf.animal.unique()
mat = np.zeros((len(an_list), 4, 4))


for sess_bin in range(1,9):
    ax = plt.subplot(2, 4, sess_bin)
    temperdf = tempdf[(tempdf.sess_bin == sess_bin) & (tempdf.task == 'unstr')]
    sns.heatmap(pd.crosstab(temperdf.port, temperdf.choice_t1, normalize = 'index', dropna = False),
                cmap = parula_map,
                annot = False,
                fmt = '.2f',
                vmin = 0.0,
                vmax = 0.12,
                square = True,
                # mask = np.eye(4),
                xticklabels = np.arange(1,5),
                yticklabels = np.arange(1,5),
                ax = ax)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_title(f'Sessions {sess_bin*bin_size-49} to {sess_bin*bin_size}')
fig.supxlabel('Choice at trial t+1')
fig.supylabel('Choice at trial t')
plt.tight_layout()

In [ ]:
# difference transition matrices for first n bins of sessions 
ind = 1
trialsinsess = 100

fig = plt.figure(figsize = (10, 4))
tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
# tempdf = tempdf[~(tempdf.animal.isin(['Kakuna', 'Finneon']))]
an_list = tempdf.animal.unique()
mat = np.zeros((len(an_list), 4, 4))


for sess_bin in range(1,9):
    ax = plt.subplot(2, 4, sess_bin)
    temperdf = tempdf[(tempdf.sess_bin == sess_bin) & (tempdf.task == 'unstr')]
    rew_mat = pd.crosstab(temperdf[temperdf.reward==1].port, temperdf[temperdf.reward==1].choice_t1, normalize = 'index', dropna = False)
    unrew_mat = pd.crosstab(temperdf[temperdf.reward==0].port, temperdf[temperdf.reward==0].choice_t1, normalize = 'index', dropna = False)
    sns.heatmap(rew_mat - unrew_mat,
                cmap = "seismic",
                annot = False,
                fmt = '.2f',
                vmin = -0.3,
                vmax = 0.3,
                square = True,
                # mask = np.eye(4),
                xticklabels = np.arange(1,5),
                yticklabels = np.arange(1,5),
                ax = ax)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_title(f'Sessions {sess_bin*bin_size-49} to {sess_bin*bin_size}')
fig.supxlabel('Choice at trial t+1')
fig.supylabel('Choice at trial t')
plt.tight_layout()

c:\Users\dlab\miniforge3\envs\rishika\lib\site-packages\ipykernel\eventloops.py:145: UserWarning: The figure layout has changed to tight
  el.exec() if hasattr(el, "exec") else el.exec_()


In [ ]:
# distance moved for first n bins of sessions 
ind = 1
trialsinsess = 100
fig = plt.figure(figsize = (9, 5))
# tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
# tempdf = tempdf[~(tempdf.animal.isin(['Kakuna', 'Finneon']))]
tempdf = 
an_list = tempdf.animal.unique()
colors = plt.cm.cool_r(np.linspace(0, 1, 8))


for sess_bin in range(1,9):
    ax = plt.subplot(2, 4, sess_bin)
    temperdf = tempdf[(tempdf.sess_bin == sess_bin) & (tempdf.shift_t0 == 1)]
    # calculate for each animal, mean and sem
    x = (temperdf.groupby('animal', as_index = False)
                          .dist_abs
                          .value_counts(normalize = True))
    x.set_index('dist_abs', inplace = True)
    x['divisor']=pd.Series(randdisp).value_counts(normalize = True).sort_index()
    x['ratio']= x.proportion/x.divisor
    x['log_odds']= x.ratio.apply(np.log2)
    x['difference']=x.proportion-x.divisor
    
    mean_calc = x.reset_index().groupby('dist_abs').difference.mean()
    sem_calc = x.reset_index().groupby('dist_abs').difference.sem()

    # plot average statistic - although could have calculated this in a much much simpler way(?)
    # statmat = x.sort_values(['dist_abs', 'animal']).reset_index()[['dist_abs', 'difference']].to_numpy().reshape(3, -1, 2)
    # avg_grouped_statmat =  np.mean(statmat, axis = 1)

    # plot here
    ax.bar(np.array([1,2,3]), mean_calc, yerr = sem_calc, color = colors[sess_bin-1], width = 0.7)
    ax.set_xticks([1,2,3], ['1', '2', '3'], fontsize = 'large', color = 'grey')
    ax.axhline(0, color='k')
    ax.set_yticks([-0.2, 0, 0.2, 0.4], ['-0.2', '0.0', '0.2', '0.4'], fontsize = 'large', color = 'grey')
    sns.despine()
    ax.set_xlabel('')
    ax.set_ylabel('')
    # ax.set_title(f'Sessions {sess_bin*bin_size-49} to {sess_bin*bin_size}')
fig.supxlabel('Distance')
fig.supylabel("$\Delta$ probability of switching")
plt.tight_layout()

AttributeError: 'DataFrameGroupBy' object has no attribute 'dist_abs'

# Bias by port proportion

In [ ]:
# get most sampled to least sampled port for each animal in each session bin (L to R)
tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
tempdf = tempdf[~(tempdf.animal.isin(['Kakuna', 'Finneon']))]

# take port proportion and arrange by highest to lowest. then average highest with highest, second highest with second highest etc
port_prop = tempdf[(tempdf.task == 'unstr') & (tempdf.sess_bin <= 8)].groupby(['animal', 'sess_bin'], as_index = False).port.value_counts(normalize = True, sort = True)
port_prop['highness'] = port_prop.groupby(['animal', 'sess_bin']).cumcount()+1
bars = port_prop.groupby(['sess_bin', 'highness']).proportion.mean().unstack().to_numpy()

err = port_prop.groupby(['sess_bin', 'highness']).proportion.sem().unstack().to_numpy()

colors = plt.cm.cool_r(np.linspace(0, 1, 8))
plt.figure(figsize = (17, 3.5))
for i in range(1, 9):
    ax = plt.subplot(1, 8, i)
    ax.bar(np.arange(1, 5), bars[i-1], color = colors[i-1], yerr = err[i-1])
    ax.set_title(f'sess bin {i}')
    ax.set_xticks([1, 4], ['Most \n sampled',  'Least \n sampled'], fontsize = 'large', color = 'grey')
    ax.set_ylim([0, 0.6])
    ax.set_yticks([0, 0.3, 0.6], ['0.0', '', '0.6'], fontsize = 'large', color = 'grey')
sns.despine()
plt.tight_layout()

In [ ]:
# for each port, calculate the proportion of choices in each session bin. then calculate entropy in that session bin 
ys = port_prop.groupby(['animal', 'sess_bin']).proportion.apply(lambda x: entropy(x, base = 2)).groupby('sess_bin').mean().to_list()
# ys = [ys[0], ys[2], ys[7]]
yerrs = port_prop.groupby(['animal', 'sess_bin']).proportion.apply(lambda x: entropy(x, base = 2)).groupby('sess_bin').sem().to_list()
# yerrs = [yerrs[0], yerrs[2], yerrs[7]]
x = 1
colors = plt.cm.cool_r(np.linspace(0, 1, 8))
# colors = colors[[0, 3, 7]]
x = 1
for y, c in zip(ys, colors):
    plt.bar(x, y, color=c, yerr = yerrs[ys.index(y)])
    x +=1
plt.ylim(0.8, 1.6)
plt.yticks([0.8, 1.2, 1.6], [0.8, '', 1.6], fontsize = 'large', color = 'grey')
plt.xticks(np.arange(1, 9), np.arange(1, 9), fontsize = 'large', color = 'grey')
plt.xlabel('Session bin', fontsize = 'x-large')
plt.ylabel('Entropy (bits)', fontsize = 'x-large')
sns.despine()
plt.tight_layout()

In [ ]:
# DLS group
# get most sampled to least sampled port for each animal in each session bin (L to R)
tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
tempdf = tempdf[~(tempdf.animal.isin(['Kakuna', 'Finneon']))]

def calc_bias(tempdf, mask):
# take port proportion and arrange by highest to lowest. then average highest with highest, second highest with second highest etc
    port_prop = tempdf[mask].groupby('animal', as_index = False).port.value_counts(normalize = True, sort = True)
    port_prop['highness'] = port_prop.groupby('animal').cumcount()+1
    y = port_prop.groupby(['animal', 'highness']).proportion.mean().unstack().T.apply(lambda x: entropy(x, base = 2))
    return y
fish = ['test05022023', 'Blissey', 'Chikorita', 'Darkrai', 'Eevee']


mask = (tempdf.task == 'unstr') & (tempdf.animal.isin(dls_an)) & (tempdf.sess_bin>=4)
y_pre = calc_bias(tempdf, mask)
y_pre_sem = y_pre.sem()

mask = (tempdf.task == 'dls') & (tempdf.animal.isin(dls_an))
y_les = calc_bias(tempdf, mask)
y_les_sem = y_les.sem()

plt.figure(figsize = (3.5, 4.5))
plt.bar([0, 1], [y_pre.mean(), y_les.mean()], yerr = [y_pre_sem, y_les_sem], color = ['xkcd:azure', 'xkcd:slate grey'], width = 0.6)
plt.xticks([0, 1], ['Pre-lesion', 'Post-lesion'], fontsize = 'large', color = 'grey')
plt.ylim(1.25, 2)
plt.yticks(fontsize = 'large', color = 'grey')
plt.title(round(ttest_rel(y_pre, y_les)[1], 4))
sns.despine()
plt.tight_layout()

In [ ]:
plt.scatter(np.zeros(len(y_pre)), y_pre, color = 'xkcd:azure', alpha = 0.5)
plt.scatter(np.ones(len(y_les)), y_les, color = 'xkcd:slate grey', alpha = 0.5)
for row in y_pre.index:
    plt.plot([0, 1], [y_pre[row], y_les[row]], color = 'grey', alpha = 0.2)

plt.xticks([0, 1], ['Pre-lesion', 'Post-lesion'], fontsize = 'large', color = 'grey')
plt.ylim(0.0, 2)
plt.yticks(fontsize = 'large', color = 'grey')
plt.title(round(ttest_rel(y_pre, y_les)[1], 4))
sns.despine()
plt.tight_layout()

In [ ]:
1- (tempdf[mask].groupby('animal').regret.mean()/100)

animal
Blissey    0.724974
Darkrai    0.728391
Inkay      0.835456
Mesprit    0.744715
Quilava    0.793589
Raltz      0.755387
Zacian     0.830155
Name: regret, dtype: float64

# Entropy in a trial across sessions

In [ ]:
#################################### moving entropy ######################################
# set_cwd('/home/rishika/sim/')
import datetime
%matplotlib qt
window = 5
trialsinsess = 100
# fig = plt.figure(figsize=(20,10))
df = sessdf
cond = 'unstr'
from utils.dfLoading import subset_trials, add_block_groups, add_shift_info
subset_df_nohead = add_block_groups(df[(df.task.isin([cond])) & (df.animal.isin(block_an_dict['uncued']+block_an_dict['cued'])) & (df.sess_bin>=4)])
subset_df_nohead = add_shift_info(subset_df_nohead)
subset_df_nohead = subset_trials(subset_df_nohead, trialsinsess=trialsinsess, head_trials=trialsinsess)


task = cond
# block_2_exists = (subset_df_nohead[(subset_df_nohead.block_group>=2) & (subset_df_nohead.task == task)]
#                   .groupby(['animal', 'sess_block'])
#                   .size()
#                   .reset_index()[['animal', 'sess_block']])

# # remove animals with less than 10
# block_2_exists = block_2_exists[block_2_exists.animal.isin(block_2_exists.animal.value_counts()[block_2_exists.animal.value_counts()>10].index)]

# find first blocks for these sessions
all_blocked_groups = subset_df_nohead[subset_df_nohead.task == task]#.merge(block_2_exists, on = ['animal', 'sess_block'])
first_blocks = all_blocked_groups[all_blocked_groups.block_group == 1]
second_blocks = all_blocked_groups[all_blocked_groups.block_group == 2]

other_blocks = all_blocked_groups[all_blocked_groups.block_group > 1]

entropy_mean_sess_dict = {}
i = 'unstr'
colors ={'str':'#408ec6', 'unstr':'#7a2048', 'dls':'xkcd:gold', 'dms':'xkcd:red', 'sham':'b', 'ds':'xkcd:orange'} 
for i in ['unstr', 'str', 'dms', 'dls', 'sham']:
    for ind, (animal, group) in enumerate(first_blocks.groupby('animal')):
        # ax = plt.subplot(5, 7, ind+1)

        if i == 'unstr':
            group = group[group.sess_bin>=4]
        
        # filter by sessions>= length
        filtered = (group[group.task.isin([i])]
                    .groupby('session')).filter(lambda x: x.reward.size >= trialsinsess)

        # take only those sessions
        filtered = filtered.groupby('session').head(trialsinsess)
        
        if animal=='Nidorina':
            filtered = filtered[filtered.datetime.astype('datetime64[ns]')<datetime.datetime(2024, 5, 8)]
        
        if filtered.empty==True:
            continue
            
        # convert to numpy array
        g = filtered.groupby('session').cumcount()
        L = np.array(filtered.set_index(['session',g])
                .unstack(fill_value=0)
                .stack(future_stack=True).groupby(level=0)
                .apply(lambda x: x.entropy.values.tolist())
                .tolist())
        # mean
        entropy_trial = np.mean(L, axis = 0)
        # entropy_trial = [entropy(calc_prob(L[:, col]), base = 2) for col in range(L.shape[1])]
        entropy_mean_sess_dict[animal, i] = entropy_trial
        # ax.plot(entropy_trial, label = i, c = colors[i])
        # ax.set_title(animal)
        # ax.legend()


# sns.despine()
# fig.supylabel('Entropy, averaged across sessions')
# plt.tight_layout()

c:\Users\dlab\banditAnalysis\utils\dfLoading.py:62: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sess_block'] = 0
c:\Users\dlab\banditAnalysis\utils\dfLoading.py:65: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sess_block'] = df.groupby(['animal', 'task']).sess_block.cumsum()
c:\Users\dlab\banditAnalysis\utils\dfLoading.py:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the d

AttributeError: 'DataFrame' object has no attribute 'entropy'

In [ ]:
# plot average reponse
# plt.figure(figsize = (5,4.5))
temp_unstr, temp_str = [], []
from scipy.stats import sem
# set properties
smooth = True
window = 3
label1, label2 = 'Pre-lesion', 'Post-lesion'
# label1, label2 = 'Unstructured', 'Structured'
color1, color2 = 'xkcd:cornflower', 'xkcd:slate grey'
# color1, color2 = 'xkcd:pumpkin orange', 'xkcd:emerald green'
# color1, color2 = '#7a2048', '#408ec6'

cond = 'unstr'
an_list = []
# for an in an_dict[cond]:
    # if an in (block_an_dict['unblocked']):
        # an_list.append(an)
# set data for plotting

for (animal, env) in entropy_mean_sess_dict.keys():
    if animal in an_list:
        if env == 'unstr':
            temp_unstr.append(entropy_mean_sess_dict[(animal, env)])
        # elif (env == cond):
        #     temp_str.append(entropy_mean_sess_dict[(animal, env)])

temp_unstr = np.array(temp_unstr)
# temp_str = np.array(temp_str)

if smooth:           
    for i, row in enumerate(temp_unstr):
        temp_unstr[i] = np.convolve(row, np.ones(3)/3, 'same')
        # temp_str[i] = np.convolve(temp_str[i], np.ones(3)/3, 'same')
    #     temp_dms[i] = np.convolve(temp_dms[i], np.ones(3)/3, 'same')

    temp_unstr = np.array(temp_unstr)
    # temp_str = np.array(temp_str)

    # works only for odd windows
    temp_unstr[:, 0:(window//2)] = np.nan
    temp_unstr[:, (trialsinsess-window//2):trialsinsess] = np.nan
    
    # temp_str[:, 0:(window//2)] = np.nan
    # temp_str[:, (trialsinsess-window//2):trialsinsess] = np.nan
# # 

mean_unstr = (np.mean(np.array(temp_unstr), axis = 0))
# mean_str = (np.mean(np.array(temp_str), axis = 0))
# temp_str = (np.mean(np.array(temp_str), axis = 0))

# [plt.plot(np.arange(trialsinsess), x, color = 'grey', alpha = 0.1) for x in temp_unstr]
plt.plot(np.arange(trialsinsess), mean_unstr, color1, label = label1) #7a2048
plt.fill_between(np.arange(trialsinsess), mean_unstr+sem(temp_unstr, nan_policy = 'omit'),
                 mean_unstr-sem(temp_unstr, nan_policy = 'omit'), color = color1, alpha = 0.2)

# plt.plot(np.arange(trialsinsess), mean_str, color2, label = label2)
# plt.fill_between(np.arange(trialsinsess),mean_str+sem(temp_str, nan_policy = 'omit'), 
#                  mean_str-sem(temp_str, nan_policy = 'omit'), color = color2, alpha = 0.2)

plt.xlabel('Trials in session', fontsize = 'x-large')
plt.ylabel('Entropy (bits)', fontsize = 'x-large')
plt.xlim(0, trialsinsess)
# plt.ylim(1.5, 2.0)
plt.xticks([0, 25, 50, 75, 100], [1, '', '', '', 100], color = 'grey', fontsize = 'large')
# plt.yticks([1.25, 1.5, 1.75, 2.0], [1.25, '', '', 2.0], color = 'grey', fontsize = 'large')
plt.title(f'n = {temp_unstr.shape[0]}')
# plt.ylim(0, 0.8)
plt.legend()
sns.despine()
# sns.set_context('talk')
plt.tight_layout()

C:\Users\dlab\AppData\Local\Temp\ipykernel_5496\971645155.py:54: SmallSampleWarning: After omitting NaNs, one or more axis-slices of one or more sample arguments is too small; corresponding elements of returned arrays will be NaN. See documentation for sample size requirements.
  plt.fill_between(np.arange(trialsinsess), mean_unstr+sem(temp_unstr, nan_policy = 'omit'),
C:\Users\dlab\AppData\Local\Temp\ipykernel_5496\971645155.py:55: SmallSampleWarning: After omitting NaNs, one or more axis-slices of one or more sample arguments is too small; corresponding elements of returned arrays will be NaN. See documentation for sample size requirements.
  mean_unstr-sem(temp_unstr, nan_policy = 'omit'), color = color1, alpha = 0.2)


# Switch distance comparisons

## weekwise 

In [ ]:
sns.displot(data = tempdf[(tempdf.task == 'unstr') & (tempdf.sess_bin <= 8)], x = 'disp', discrete = True, col = 'sess_bin', col_wrap = 4, hue = 'sess_bin', palette='cool_r', stat = 'probability', common_norm = False, alpha = 1)
plt.ylim(0., 0.08)

c:\Users\dlab\miniforge3\envs\rishika\lib\site-packages\seaborn\axisgrid.py:123: UserWarning: The figure layout has changed to tight
  self._figure.tight_layout(*args, **kwargs)


(0.0, 0.08)

In [164]:
from utils.dfLoading import add_shift_info
sessdf = add_shift_info(sessdf)
from utils.dfAnalysisHelpers import calc_dist_metric
tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
tempdf = calc_dist_metric(tempdf, (tempdf.shift_t0==1) & (tempdf.task == 'unstr') & (tempdf.week_task <= 8))
y = tempdf.groupby(['week_task','animal']).d_value.mean().unstack().mean(axis = 1)
# y = [y[1], y[3], y[7]]
yerr = tempdf.groupby(['week_task','animal']).d_value.mean().unstack().sem(axis = 1)
# yerr = [yerr[1], yerr[3], yerr[7]]
x = tempdf.groupby(['week_task','animal']).d_chance.mean().unstack().mean(axis = 1)
# x = [x[1], x[3], x[7]]
# colors = [plt.cm.cool_r(np.linspace(0, 1, 8)[0]), plt.cm.cool_r(np.linspace(0, 1, 8)[3]), plt.cm.cool_r(np.linspace(0, 1, 8)[7])]
plt.figure()
for i in y.index[:8]:    
    plt.bar(i, y[i], color = plt.cm.cool_r(np.linspace(0, 1, 8)[int(i)-1]), yerr = yerr[i])
plt.bar(np.arange(len(x)), x, color = 'grey', alpha = 0.2)
# plt.xticks(np.arange(1, 4), ['Naive', 'Intermediate', 'Expert'], fontsize = 'large', color = 'grey')
plt.xticks(np.arange(1, 9), np.arange(1, 9), fontsize = 'large', color = 'grey')
plt.xlabel('Week in task', fontsize = 'x-large')
plt.ylim(1, 1.65)
plt.yticks([1, 1.2, 1.4, 1.6], [1,'', '', 1.6], fontsize = 'large', color = 'grey')
plt.ylabel('Switch distance', fontsize = 'x-large')
sns.despine()
plt.tight_layout()

c:\Users\dlab\banditAnalysis\utils\dfLoading.py:141: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['shift_t0'] = (df['choice_t1']==df['port']).replace({True: 0, False: 1})
c:\Users\dlab\banditAnalysis\utils\dfLoading.py:142: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['shift_t1'] = (df['choice_t2']==df['port']).replace({True: 0, False: 1})


In [120]:
tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
mask = ((tempdf.week_task >= 4) & (tempdf.shift_t0 == 1) & (tempdf.task == 'unstr'))
dist_moved = calc_dist_metric(tempdf, mask).groupby('animal').d_value.mean()
chance_move = calc_dist_metric(tempdf, mask).groupby('animal').d_chance.mean()

plt.figure(figsize=(3.5,4))
plt.bar(0, chance_move.mean(), color = 'grey',  yerr = chance_move.sem(), width = 0.6)
plt.bar(1, dist_moved.mean(), color = 'xkcd:cornflower', yerr = dist_moved.sem(), width = 0.6)
plt.xlim(-0.5, 1.5)
plt.ylim(1,1.65)
plt.xticks([0, 1], ['Chance', 'Animal data'], fontsize = 'large', color = 'grey')
plt.yticks([1, 1.2, 1.4, 1.6], [1,'', '', 1.6], fontsize = 'large', color = 'grey')
plt.ylabel('Switch distance', fontsize = 'x-large')
sns.despine()
plt.tight_layout()

In [162]:
# for r and nr
trialsinsess = 100
# an_list = block_an_dict['uncued']+block_an_dict['cued']+block_an_dict['unblocked']
an_list = an_dict['hsb_str']
tempdf = hsb_data.groupby(['animal', 'session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
mask = (tempdf.reward == 0) & (tempdf.week_task >= 4) & (tempdf.shift_t0 == 1) & (tempdf.task == 'str') & (tempdf.block_group == 1) & (tempdf.animal.isin(an_list))
nr = calc_dist_metric(tempdf, mask)
ys_nr = nr.groupby('animal').d_value.mean()

mask = (tempdf.reward == 1) & (tempdf.week_task >= 4) & (tempdf.shift_t0 == 1) & (tempdf.task == 'str') & (tempdf.block_group == 1) & (tempdf.animal.isin(an_list))
r = calc_dist_metric(tempdf, mask)
ys_r = r.groupby('animal').d_value.mean()
plt.figure(figsize=(3.5,4))
plt.bar(1, nr.groupby('animal').d_chance.mean().mean(), color = 'grey', alpha = 0.2, width = 0.6)
plt.bar(1, ys_nr.mean(), color = 'blue', yerr = ys_nr.sem(), width = 0.6)
plt.hlines(y = nr.groupby('animal').d_chance.mean().mean(), xmin = -0.3, xmax = 0.3, color = 'grey', ls = '--', lw = 2)
plt.bar(0, r.groupby('animal').d_chance.mean().mean(), color = 'grey', alpha = 0.2, width = 0.6)
plt.bar(0, ys_r.mean(), color = 'red', yerr = ys_r.sem(), width = 0.6)
plt.hlines(y = r.groupby('animal').d_chance.mean().mean(), xmin = 1-0.3, xmax = 1+0.3, color = 'grey', ls = '--', lw = 2)

plt.xlim(-0.5, 1.5)
plt.ylim(1,1.7)
plt.xticks([0, 1], ['Reward', 'No reward'], fontsize = 'large', color = 'grey')
plt.yticks([1, 1.2, 1.4, 1.6], [1,'', '', 1.6], fontsize = 'large', color = 'grey')
plt.ylabel('Switch distance', fontsize = 'x-large')
sns.despine()
plt.tight_layout()

In [123]:
ttest_rel(ys_nr, ys_r, nan_policy='omit')

TtestResult(statistic=np.float64(4.607160643037783), pvalue=np.float64(0.0009694355380706622), df=np.int64(10))

##  changes with trials

In [ ]:
#################################### distance moved ######################################
from utils.dfLoading import subset_trials, add_block_groups
subset_df = add_block_groups(sessdf)
subset_df = subset_df[(subset_df.week_task>=4) & (subset_df.block_group == 1) & (subset_df.animal.isin(block_an_dict['uncued']+block_an_dict['cued']))]

subset_df_unbl = add_block_groups(sessdf)
subset_df_unbl = subset_df_unbl[(subset_df_unbl.week_task>=4) & (subset_df_unbl.block_group == 1) & (subset_df_unbl.animal.isin(block_an_dict['unblocked']))]

subset_df_hsb = add_block_groups(hsb_data)
subset_df_hsb = subset_df_hsb[(subset_df_hsb.block_group == 1)& (subset_df_hsb.week_task>=4)]

trialsinsess = 100
subset_df = subset_trials(subset_df, trialsinsess=trialsinsess, head_trials = trialsinsess)
subset_df_hsb = subset_trials(subset_df_hsb, trialsinsess=trialsinsess, head_trials = trialsinsess)
subset_df_unbl = subset_trials(subset_df_unbl, trialsinsess=trialsinsess, head_trials = trialsinsess)


from utils.dfAnalysisHelpers import get_sessionwise_metric, get_switch_distance_from_ports
port_mean_dict = get_sessionwise_metric(subset_df, metric= 'port', tasks = ['unstr', 'dms', 'dls'])
port_mean_dict_hsb = get_sessionwise_metric(subset_df_hsb, metric = 'port', tasks = ['str'])

port_mean_dict_hsb_unstr = get_sessionwise_metric(subset_df_hsb[subset_df_hsb.env == 'unstr'], metric= 'port', tasks = ['str'])
port_mean_dict_hsb_str = get_sessionwise_metric(subset_df_hsb[subset_df_hsb.env == 'str'], metric= 'port', tasks = ['str'])

port_mean_dict_pure_unstr = get_sessionwise_metric(subset_df[subset_df.env == 'unstr'], metric= 'port', tasks = ['unstr'])
port_mean_dict_str_in_unstr = get_sessionwise_metric(subset_df[subset_df.env == 'str'], metric= 'port', tasks = ['unstr'])

port_mean_dict_unbl = get_sessionwise_metric(subset_df_unbl, metric = 'port', tasks = ['unstr'])

In [115]:
from scipy.optimize import curve_fit
from scipy.stats import sem
from itertools import permutations
# plot average reponse
from utils.plotSettings import *
plt.figure(figsize = (5,4.5))
temp_unstr, temp_str = [], []

# set properties
smooth = False
window = 5
trialsinsess = 99
line2 = True
legend = True
# label1, label2 = 'Pre-lesion', 'Post-lesion'
# label1, label2 = 'Structured in unstructured', 'Structured'
# label1, label2 = 'Uncued', 'Cued'
# label1, label2 = 'Block 1', 'Block 2'
label1, label2 = 'Blocked', 'Unblocked'
# color1, color2 =  sns.color_palette('crest')[0],  sns.color_palette('crest')[4]
color1, color2 = 'xkcd:cornflower', 'xkcd:pale purple'
# color1, color2 = 'xkcd:cornflower', 'xkcd:slate grey'
# color1, color2 = 'xkcd:olive green', 'xkcd:emerald green'
# color1, color2 = 'xkcd:pumpkin orange', 'xkcd:dark orange'


# set data for plotting
for (animal, env) in port_mean_dict.keys():
    if env == 'unstr':
        ports_sess = port_mean_dict[(animal, env)]
        switch_arr = get_switch_distance_from_ports(ports_sess)
        temp_unstr.append(np.nanmean(switch_arr, axis = 0))

for (animal, env) in port_mean_dict_unbl.keys():
    if env == 'unstr':
        ports_sess = port_mean_dict_unbl[(animal, env)]
        switch_arr = get_switch_distance_from_ports(ports_sess)
        temp_str.append(np.nanmean(switch_arr, axis = 0))


temp_unstr = np.array(temp_unstr)
# temp_unstr[:, 99] = np.nan
temp_str = np.array(temp_str)
# temp_str[:, 99] = np.nan

# smoothing details
bigger_arr = temp_unstr if temp_unstr.shape[0]>temp_str.shape[0] else temp_str
if smooth:           
    for i, row in enumerate(bigger_arr):
        temp_unstr[i] = np.convolve(row, np.ones(window)/window, 'same')
        try:
            temp_str[i] = np.convolve(temp_str[i], np.ones(window)/window, 'same')
        except IndexError:
            pass
    
    temp_unstr = np.array(temp_unstr)
    temp_str = np.array(temp_str)
    # 
    # works only for odd windows
    temp_unstr[:, 0:(window//2)] = np.nan
    temp_unstr[:, (trialsinsess-window//2):trialsinsess] = np.nan
    
    temp_str[:, 0:(window//2)] = np.nan
    temp_str[:, (trialsinsess-window//2):trialsinsess] = np.nan


# take mean of smoothed data   
mean_unstr = (np.nanmean(np.array(temp_unstr), axis = 0))
mean_str = (np.nanmean(np.array(temp_str), axis = 0))

# plot line 1
plt.plot(np.arange(trialsinsess), mean_unstr, color = color1, label = label1)#, marker = '.') 
plt.fill_between(np.arange(trialsinsess), mean_unstr+sem(temp_unstr, nan_policy = 'omit'),
                mean_unstr-sem(temp_unstr, nan_policy = 'omit'), color = color1, alpha = 0.2)

# plot line 2 (or comment)
if line2:
    plt.plot(np.arange(trialsinsess), mean_str, color = color2, label = label2)
    plt.fill_between(np.arange(trialsinsess), mean_str+sem(temp_str, nan_policy = 'omit'), 
                    mean_str-sem(temp_str, nan_policy = 'omit'), color = color2, alpha = 0.2)

# plot significance
# from scipy.stats import ttest_ind
# pvals = ttest_ind(temp_str, temp_unstr)[1]
# pvals_ind = [ind for ind, i in enumerate(pvals) if i < 0.05]
# plt.scatter(pvals_ind, [0.45]*len(pvals_ind), marker = '.', color = 'r', alpha = 0.4) #(6, 2, 60) for asterisks

# chance calc
# plt.axhline(np.mean(np.unique(list(permutations([80, 46, 15, 10]))+list(permutations([46, 80, 46, 15])), axis = 0))/100,
#             color = color1, linestyle = '--')
# plt.axhline(np.mean([[80, 46, 15, 10], [46, 80, 46, 15], [15, 46, 80, 46], [10, 15, 46, 80]])/100, color = color2, linestyle = '--')

# plot settings
plt.xlabel('Trials in session', fontsize = 'x-large')
plt.ylabel('Switch distance', fontsize = 'x-large')
plt.xlim(0, trialsinsess)
plt.ylim(1., 1.6)
plt.xticks([0, 25, 50, 75, 100], [1, '', '', '', 100], color = 'grey', fontsize = 'large')
# plt.yticks([ 1.2, 1.4, 1.6], ['1.2',  '',  '1.6'], color = 'grey', fontsize = 'large')
plt.title(f'n = {temp_unstr.shape[0]}')

if legend:
    leg = plt.legend(frameon=False)
    for line, text in zip(leg.get_lines(), leg.get_texts()):
        text.set_color(line.get_color())
sns.despine()
# sns.set_context('talk')
plt.tight_layout()
# plt.savefig('C:/Users/dlab/OneDrive - Indian Institute of Science/Drawings/awp1/entropy_n_21.png', dpi = 300)


# if fitting exp curve is reqd
def logFunc(x, a, b):
    return a + b*np.log(x)

# popt_str, _ = curve_fit(logFunc, np.arange(150), temp_str, nan_policy = 'omit')
# popt_unstr, _ = curve_fit(logFunc, np.arange(trialsinsess), mean_unstr, nan_policy = 'omit')
# plt.plot(np.arange(150), logFunc(np.arange(150), *popt_str), '#408ec6', linestyle= 'dotted')
# plt.plot(np.arange(trialsinsess), logFunc(np.arange(trialsinsess), *popt_unstr), color1, linestyle= 'dotted')

C:\Users\dlab\AppData\Local\Temp\ipykernel_25620\1154667669.py:32: RuntimeWarning: Mean of empty slice
  temp_unstr.append(np.nanmean(switch_arr, axis = 0))
C:\Users\dlab\AppData\Local\Temp\ipykernel_25620\1154667669.py:38: RuntimeWarning: Mean of empty slice
  temp_str.append(np.nanmean(switch_arr, axis = 0))


In [131]:
# blockwise switch distance diff groups
import warnings
from scipy.stats import sem
from utils.plotSettings import *

# (block_group, 'first'/'last', n_trials) in x-axis order
SEGMENTS = [(1, 'first', 100), (1, 'last', 25),
            (2, 'first', 100)]
seg_lens = [n for _, _, n in SEGMENTS]
seg_edges = np.cumsum([0] + seg_lens)
n_trials = seg_edges[-1]   # 275
ALLOW_OVERLAP = False      # False: a block needs >= first_n + last_n trials so first/last don't share trials

# set properties
smooth = True
window = 5                # odd number so the window is centred
smooth_within_blocks = False   # True: don't smooth across segment boundaries
metric = 'raw'       # 'raw' | 'chance' | 'minus_chance' | 'ratio'

MEAN_DIST = {1: 2, 2: 4/3, 3: 4/3, 4: 2}   # chance |distance| for a random switch from each port (as in calc_dist_metric)

# minimum trials each block needs
need = {}
for bg, _, n in SEGMENTS:
    need[bg] = need.get(bg, 0) + n if not ALLOW_OVERLAP else max(need.get(bg, 0), n)

def add_switch_dist(df, metric='minus_chance'):
    """Per-trial switch distance on shift trials (NaN otherwise), relative to chance given the starting port."""
    df = df.copy()
    is_switch = (df.shift_t0 == 1) & df.disp.notna()
    d_value = df.disp.abs()
    d_chance = df.port.map(MEAN_DIST)
    val = {'raw': d_value,
           'chance': d_chance,
           'minus_chance': d_value - d_chance,
           'ratio': d_value / d_chance}[metric]
    df['switch_dist'] = val.where(is_switch)
    return df

def session_matrix(df, col, sess_col='session'):
    """Returns {animal: n_sessions x n_trials matrix of `col`}; only sessions where every block is long enough."""
    out = {}
    df = df.sort_values('datetime')
    for animal, adf in df.groupby('animal'):
        rows = []
        for _, sdf in adf.groupby(sess_col):
            blocks = {bg: sdf.loc[sdf.block_group == bg, col].to_numpy(dtype=float) for bg in need}
            if any(len(blocks[bg]) < need[bg] for bg in need):
                continue
            parts = [blocks[bg][:n] if which == 'first' else blocks[bg][-n:] for bg, which, n in SEGMENTS]
            rows.append(np.concatenate(parts))
        if rows:
            out[animal] = np.vstack(rows)
    return out

def smooth_rows(mat, window, within_blocks=True):
    """NaN-aware moving average; near segment edges/NaNs, averages only over available trials."""
    out = np.full_like(mat, np.nan)
    edges = seg_edges if within_blocks else [0, mat.shape[1]]
    k = np.ones(window)
    for s, e in zip(edges[:-1], edges[1:]):
        seg = mat[:, s:e]
        valid = ~np.isnan(seg)
        num = np.array([np.convolve(r, k, 'same') for r in np.where(valid, seg, 0)])
        den = np.array([np.convolve(r, k, 'same') for r in valid.astype(float)])
        with np.errstate(invalid='ignore', divide='ignore'):
            out[:, s:e] = num / den
    return out

# data
unstr_df = sessdf[(sessdf.day_task > 30) & (sessdf.task == 'unstr') & (sessdf.animal.isin(block_an_dict['cued']+block_an_dict['uncued']))]
str_df   = hsb_data[(hsb_data.day_task > 30) & (hsb_data.task == 'str') ]
unstr_mats = session_matrix(add_switch_dist(unstr_df, metric), 'switch_dist', sess_col='session')
str_mats   = session_matrix(add_switch_dist(str_df, metric), 'switch_dist', sess_col='session')

# average over sessions (rows) per animal -> animals x n_trials; NaN where an animal never switched at that trial
with warnings.catch_warnings():
    warnings.simplefilter('ignore', category=RuntimeWarning)
    temp_unstr = np.array([np.nanmean(m, axis=0) for m in unstr_mats.values()])
    temp_str   = np.array([np.nanmean(m, axis=0) for m in str_mats.values()])

    if smooth:
        temp_unstr = smooth_rows(temp_unstr, window, smooth_within_blocks)
        temp_str = smooth_rows(temp_str, window, smooth_within_blocks)

    mean_unstr = np.nanmean(temp_unstr, axis=0)
    mean_str = np.nanmean(temp_str, axis=0)

# plot
plt.figure(figsize=(9, 4.5))
ax = plt.gca()
color1, color2 = 'xkcd:barney purple', 'xkcd:emerald green'
label1, label2 = 'Uncorrelated', 'Correlated'

# split lines only at axis breaks (first 100 -> last 25), not at block switches
break_pos = [b for (_, which, _), b in zip(SEGMENTS[1:], seg_edges[1:-1]) if which == 'last']
switch_pos = [b for (_, which, _), b in zip(SEGMENTS[1:], seg_edges[1:-1]) if which == 'first']
runs = list(zip([0] + break_pos, break_pos + [n_trials]))

# shift x after each axis break so the data has a visible gap there
gap = 2.5   # half-width of the break, in trials
def xmap(t):
    return np.asarray(t) + 2 * gap * np.searchsorted(break_pos, t, side='right')
x = xmap(np.arange(n_trials))

for i, (s, e) in enumerate(runs):
    for mean, temp, c, lab in [(mean_unstr, temp_unstr, color1, label1), (mean_str, temp_str, color2, label2)]:
        # for row in temp:   # individual animals
        #     plt.plot(x[s:e], row[s:e], color=c, linewidth=0.5, alpha=0.4)
        plt.plot(x[s:e], mean[s:e], color=c, linewidth=2, label=lab if i == 0 else None)
        err = sem(temp[:, s:e], nan_policy='omit')
        plt.fill_between(x[s:e], mean[s:e] + err, mean[s:e] - err, color=c, alpha=0.2)

# label animals whose switch distance exceeds LABEL_THRESH anywhere, at their peak
LABEL_THRESH = 2
for mats, temp, c in [(unstr_mats, temp_unstr, color1), (str_mats, temp_str, color2)]:
    for animal, row in zip(mats.keys(), temp):
        if np.nanmax(row) > LABEL_THRESH:
            t = np.nanargmax(row)
            plt.annotate(animal, (x[t], row[t]), xytext=(0, 4), textcoords='offset points',
                         ha='center', va='bottom', fontsize='small', color=c)

# block switches: dashed vertical line
for b in switch_pos:
    plt.axvline(xmap(b), color='grey', linestyle='--', linewidth=0.75)

# chance reference
if metric == 'minus_chance':
    plt.axhline(0, color='grey', linewidth=0.75)
elif metric == 'ratio':
    plt.axhline(1, color='grey', linewidth=0.75)

# axis break where the middle of a block is skipped
trans = ax.get_xaxis_transform()   # x in data units, y in axes units
for b in xmap(break_pos) - gap - 0.5:   # centre of each gap in the data
    ax.add_patch(plt.Rectangle((b - gap, -0.02), 2 * gap, 0.04, transform=trans,
                               facecolor=plt.gcf().get_facecolor(), edgecolor='none',
                               clip_on=False, zorder=3))
    ax.plot([b - gap, b + gap], [0, 0], transform=trans, linestyle='none',
            marker=[(-1, -1.5), (1, 1.5)], markersize=8, markeredgewidth=1,
            color='k', clip_on=False, zorder=4)

ylabels = {'raw': 'Switch distance', 'chance': 'Chance switch distance (ports)',
           'minus_chance': 'Switch distance − chance', 'ratio': 'Switch distance / chance'}
plt.xlabel('Trials in session', fontsize='x-large')
plt.ylabel(ylabels[metric], fontsize='x-large')
plt.xlim(0, xmap(n_trials - 1) + 1)
ticks = np.arange(0, n_trials + 1, 25)
plt.xticks(xmap(ticks), ticks, color='grey', fontsize='large')
leg = plt.legend(frameon=False, loc='upper left')
for line, text in zip(leg.get_lines(), leg.get_texts()):
    text.set_color(line.get_color())
plt.tight_layout()


In [31]:
for name, mats in [('Uncorrelated', unstr_mats), ('Correlated', str_mats)]:
    n_sess = {a: m.shape[0] for a, m in mats.items()}
    n_sw = np.array([np.sum(~np.isnan(m), axis=0) for m in mats.values()])   # animals x trials: switches per trial
    temp = np.array([np.nanmean(m, axis=0) for m in mats.values()])
    print(f'{name}: {len(mats)} animals, sessions/animal {n_sess}')
    print(f'  median switches behind each animal-trial value: {np.median(n_sw):.1f}  '
          f'(fraction of animal-trials with <=2 switches: {(n_sw <= 2).mean():.2f})')
    print(f'  animals contributing per trial (median): {np.median((n_sw > 0).sum(0)):.1f}')
    print(f'  mean SD across animals: {np.nanmean(np.nanstd(temp, axis=0)):.3f}')
    print(f'  mean SD of each animal\'s own distances: '
          f'{np.mean([np.nanstd(m) for m in mats.values()]):.3f}')


Uncorrelated: 16 animals, sessions/animal {'Azelf': 4, 'Bidoof': 7, 'Emolga': 121, 'Giratina': 168, 'Ivysaur': 143, 'Jigglypuff': 114, 'Lugia': 95, 'Pichu': 61, 'Quaxly': 102, 'Sableye': 137, 'Torchic': 158, 'Uxie': 100, 'Vanillish': 151, 'Whismur': 112, 'Yamper': 102, 'Zorua': 137}
  median switches behind each animal-trial value: 11.0  (fraction of animal-trials with <=2 switches: 0.14)
  animals contributing per trial (median): 15.0
  mean SD across animals: 0.177
  mean SD of each animal's own distances: 0.496
Correlated: 9 animals, sessions/animal {'HSB_F_006': 158, 'HSB_F_007': 157, 'HSB_F_008': 154, 'HSB_F_009': 165, 'HSB_F_010': 157, 'HSB_F_011': 144, 'HSB_F_012': 176, 'HSB_F_013': 219, 'HSB_F_014': 249}
  median switches behind each animal-trial value: 15.0  (fraction of animal-trials with <=2 switches: 0.02)
  animals contributing per trial (median): 9.0
  mean SD across animals: 0.271
  mean SD of each animal's own distances: 0.475


C:\Users\dlab\AppData\Local\Temp\ipykernel_22024\1382674020.py:4: RuntimeWarning: Mean of empty slice
  temp = np.array([np.nanmean(m, axis=0) for m in mats.values()])


## pairwise condition comparisons

In [132]:
# compare unblocked with blocked animals' data
trialsinsess = 100
an_list_unbl = block_an_dict['unblocked']
an_list_bl = block_an_dict['cued']+block_an_dict['uncued']
tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)

struc = calc_dist_metric(tempdf, (tempdf.task == 'unstr') & (tempdf.animal.isin(an_list_bl)) & (tempdf.shift_t0 == 1) & (tempdf.week_task >= 4) & (tempdf.block_group == 1))
unstruc = calc_dist_metric(tempdf, (tempdf.task == 'unstr') & (tempdf.animal.isin(an_list_unbl)) & (tempdf.shift_t0 == 1) & (tempdf.week_task >= 4) & (tempdf.block_group == 1))

y_str = struc[struc.animal.isin(an_list_bl)].groupby('animal').d_value.mean()
y_unstr = unstruc[unstruc.animal.isin(an_list_unbl)].groupby('animal').d_value.mean()

width = 0.6
plt.figure(figsize=(3.5,4))
plt.bar(0, unstruc[unstruc.animal.isin(an_list_unbl)].groupby('animal').d_chance.mean().mean(), color = 'grey', alpha = 0.2, width = 0.6)
plt.bar(0, y_unstr.dropna().mean(), color = 'xkcd:pale purple', yerr = y_unstr.dropna().sem(), width = 0.6)
plt.bar(1, struc[struc.animal.isin(an_list_bl)].groupby('animal').d_chance.mean().mean(), color = 'grey', alpha = 0.2, width = 0.6)
plt.bar(1, y_str.dropna().mean(), color = 'xkcd:cornflower', yerr = y_str.dropna().sem(), width = 0.6)
plt.xlim(-0.5, 1.5)
plt.ylim(1,1.9)
plt.xticks([0, 1], ['Pre-lesion', 'Post-lesion'], fontsize = 'large', color = 'grey')
plt.yticks([1, 1.2, 1.4, 1.6, 1.8], [1,'', '', '',1.8], fontsize = 'large', color = 'grey')
plt.ylabel('Distance metric', fontsize = 'x-large')
sns.despine()
plt.tight_layout()
# print(ttest_rel(y_unstr, y_str))
# print(ttest_rel(y_unstr, unstruc[unstruc.animal.isin(an_list_unbl)].groupby('animal').d_chance.mean()))
# print(ttest_rel(y_str, struc[struc.animal.isin(an_list_bl)].groupby('animal').d_chance.mean()))
# Statistical annotations using matplotlib
from scipy.stats import ttest_rel
annot = False
if annot:
    # Perform t-tests
    t_stat1, p_val1 = ttest_rel(y_unstr, y_str)
    t_stat2, p_val2 = ttest_rel(y_unstr, unstruc[unstruc.animal.isin(dms_an)].groupby('animal').d_chance.mean())
    t_stat3, p_val3 = ttest_rel(y_str, struc[struc.animal.isin(dms_an)].groupby('animal').d_chance.mean())

    # Add annotations
    max_y = (unstruc[unstruc.animal.isin(dms_an)].groupby('animal').d_chance.mean()).mean()

    if p_val1:
        plt.plot([0, 1], [max_y + 0.02, max_y + 0.02], color='black', linewidth=1)
        plt.text(0.5, max_y + 0.03, f'p={p_val1:.3f}', ha='center', fontsize=10)

    if p_val2:
        plt.plot([width/2 +0.05, width/2 + 0.05], [(y_unstr.mean()+1)/2, (max_y - y_unstr.mean())/2 + y_unstr.mean()], color='black', linewidth=1)
        plt.text(width/2 +0.1, y_unstr.mean() - 0.15, f'p={p_val2:.3f}', ha='center', fontsize=10, rotation = 270)

    if p_val3:
        plt.plot([width/2+1.05, width/2+1.05], [(y_str.mean()+1)/2, (max_y - y_str.mean())/2 + y_str.mean()], color='black', linewidth=1)
        plt.text(width/2+1.1, y_str.mean() - 0.15, f'p={p_val3:.3f}', ha='center', fontsize=10, rotation = 270)
plt.tight_layout()

In [ ]:
# sessdf.drop(columns = ['d_value', 'd_chance'], inplace = True)
trialsinsess = 100
an_list = an_dict['unstr']
# tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
# struc = calc_dist_metric(tempdf, (tempdf.task == 'unstr')  & (tempdf.env == 'str') & (tempdf.animal.isin(an_dict['unstr'])) & (tempdf.shift_t0 == 1) & (tempdf.sess_bin >= 4))

# comparison with str trained data
tempdf = hsb_data.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
struc = calc_dist_metric(tempdf, (tempdf.task == 'str') & (tempdf.env == 'str') & (tempdf.shift_t0 == 1))

tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
unstruc = calc_dist_metric(tempdf, (tempdf.task == 'unstr') & (tempdf.env == 'str') & (tempdf.animal.isin(an_dict['unstr'])) & (tempdf.shift_t0 == 1) & (tempdf.sess_bin >= 4))

# y_str = struc[struc.animal.isin(an_list)].groupby('animal').d_value.mean()
y_str = struc.groupby('animal').d_value.mean()
y_unstr = unstruc[unstruc.animal.isin(an_list)].groupby('animal').d_value.mean()

plt.figure(figsize=(3.5,4))
plt.bar(0, unstruc[unstruc.animal.isin(an_list)].groupby('animal').d_chance.mean().mean(), color = 'grey', alpha = 0.2, width = 0.6)
plt.bar(0, y_unstr.mean(), color = 'xkcd:olive green', yerr = y_unstr.sem(), width = 0.6)
# plt.bar(1, struc[struc.animal.isin(an_list)].groupby('animal').d_chance.mean().mean(), color = 'grey', alpha = 0.2, width = 0.6)
plt.bar(1, struc.groupby('animal').d_chance.mean().mean(), color = 'grey', alpha = 0.2, width = 0.6)
plt.bar(1, y_str.mean(), color = 'xkcd:emerald green', yerr = y_str.sem(), width = 0.6)
plt.xlim(-0.5, 1.5)
plt.ylim(1,1.7)
plt.xticks([0, 1], ['Unstr in Str', 'Structured'], fontsize = 'large', color = 'grey')
plt.yticks([1, 1.2, 1.4, 1.6], [1,'', '', 1.6], fontsize = 'large', color = 'grey')
plt.ylabel('Distance metric', fontsize = 'x-large')
sns.despine()
plt.tight_layout()
# print(ttest_rel(y_unstr, y_str))

c:\Users\dlab\banditAnalysis\utils\dfLoading.py:138: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['shift_t0'] = (df['choice_t1']==df['port']).replace({True: 0, False: 1})
c:\Users\dlab\banditAnalysis\utils\dfLoading.py:139: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['shift_t1'] = (df['choice_t2']==df['port']).replace({True: 0, False: 1})
C:\Users\dlab\AppData\Local\Temp\ipykernel_26568\243829769.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a futur

In [ ]:
tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
struc = calc_dist_metric(tempdf, (tempdf.task == 'dms') & (tempdf.animal.isin(dms_an)) & (tempdf.shift_t0 == 1))
tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
unstruc = calc_dist_metric(tempdf, (tempdf.task == 'unstr') & (tempdf.animal.isin(dms_an)) & (tempdf.shift_t0 == 1) & (tempdf.sess_bin >= 3))
y_str = struc[struc.animal.isin(dms_an)].groupby('animal').d_value.mean()
y_unstr = unstruc[unstruc.animal.isin(dms_an)].groupby('animal').d_value.mean()
width = 0.6
plt.figure(figsize=(3.5,4))
plt.bar(0, unstruc[unstruc.animal.isin(dms_an)].groupby('animal').d_chance.mean().mean(), color = 'grey', alpha = 0.2, width = 0.6)
plt.bar(0, y_unstr.mean(), color = 'xkcd:cornflower', yerr = y_unstr.sem(), width = 0.6)
plt.bar(1, struc[struc.animal.isin(dms_an)].groupby('animal').d_chance.mean().mean(), color = 'grey', alpha = 0.2, width = 0.6)
plt.bar(1, y_str.mean(), color = 'xkcd:slate grey', yerr = y_str.sem(), width = 0.6)
plt.xlim(-0.5, 1.5)
plt.ylim(1,1.9)
plt.xticks([0, 1], ['Pre-lesion', 'Post-lesion'], fontsize = 'large', color = 'grey')
plt.yticks([1, 1.2, 1.4, 1.6, 1.8], [1,'', '', '',1.8], fontsize = 'large', color = 'grey')
plt.ylabel('Distance metric', fontsize = 'x-large')
sns.despine()
plt.tight_layout()
print(ttest_rel(y_unstr, y_str))
print(ttest_rel(y_unstr, unstruc[unstruc.animal.isin(dms_an)].groupby('animal').d_chance.mean()))
print(ttest_rel(y_str, struc[struc.animal.isin(dms_an)].groupby('animal').d_chance.mean()))
# Statistical annotations using matplotlib
from scipy.stats import ttest_rel
annot = True
if annot:
    # Perform t-tests
    t_stat1, p_val1 = ttest_rel(y_unstr, y_str)
    t_stat2, p_val2 = ttest_rel(y_unstr, unstruc[unstruc.animal.isin(dms_an)].groupby('animal').d_chance.mean())
    t_stat3, p_val3 = ttest_rel(y_str, struc[struc.animal.isin(dms_an)].groupby('animal').d_chance.mean())

    # Add annotations
    max_y = (unstruc[unstruc.animal.isin(dms_an)].groupby('animal').d_chance.mean()).mean()

    # if p_val1:
    #     plt.plot([0, 1], [max_y + 0.02, max_y + 0.02], color='black', linewidth=1)
    #     plt.text(0.5, max_y + 0.03, f'p={p_val1:.3f}', ha='center', fontsize=10)

    # if p_val2:
    #     plt.plot([width/2 +0.05, width/2 + 0.05], [(y_unstr.mean()+1)/2, (max_y - y_unstr.mean())/2 + y_unstr.mean()], color='black', linewidth=1)
    #     plt.text(width/2 +0.1, y_unstr.mean() - 0.15, f'p={p_val2:.3f}', ha='center', fontsize=10, rotation = 270)

    # if p_val3:
    #     plt.plot([width/2+1.05, width/2+1.05], [(y_str.mean()+1)/2, (max_y - y_str.mean())/2 + y_str.mean()], color='black', linewidth=1)
    #     plt.text(width/2+1.1, y_str.mean() - 0.15, f'p={p_val3:.3f}', ha='center', fontsize=10, rotation = 270)
plt.tight_layout()

C:\Users\dlab\AppData\Local\Temp\ipykernel_18528\1447638611.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  d_chances = filtered.groupby(['animal', 'session']).apply(
C:\Users\dlab\AppData\Local\Temp\ipykernel_18528\1447638611.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  d_chances = filtered.groupby(['animal', 'session']).apply(


TtestResult(statistic=np.float64(0.6129802803606876), pvalue=np.float64(0.5535738618503461), df=np.int64(10))
TtestResult(statistic=np.float64(-2.2278503745357123), pvalue=np.float64(0.05002446000250696), df=np.int64(10))
TtestResult(statistic=np.float64(-2.480195902723984), pvalue=np.float64(0.03253190871068822), df=np.int64(10))


In [ ]:
tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
struc = calc_dist_metric(tempdf, (tempdf.task == 'dls') & (tempdf.animal.isin(dls_an)) & (tempdf.shift_t0 == 1))
tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
unstruc = calc_dist_metric(tempdf, (tempdf.task == 'unstr') & (tempdf.animal.isin(dls_an)) & (tempdf.shift_t0 == 1) & (tempdf.sess_bin >= 3))
y_str = struc[struc.animal.isin(dls_an)].groupby('animal').d_value.mean()
y_unstr = unstruc[unstruc.animal.isin(dls_an)].groupby('animal').d_value.mean()
width = 0.6
plt.figure(figsize=(3.5,4))
plt.bar(0, unstruc[unstruc.animal.isin(dls_an)].groupby('animal').d_chance.mean().mean(), color = 'grey', alpha = 0.2, width = 0.6)
plt.bar(0, y_unstr.mean(), color = 'xkcd:cornflower', yerr = y_unstr.sem(), width = 0.6)
plt.bar(1, struc[struc.animal.isin(dls_an)].groupby('animal').d_chance.mean().mean(), color = 'grey', alpha = 0.2, width = 0.6)
plt.bar(1, y_str.mean(), color = 'xkcd:slate grey', yerr = y_str.sem(), width = 0.6)
plt.xlim(-0.5, 1.5)
plt.ylim(1,1.9)
plt.xticks([0, 1], ['Pre-lesion', 'Post-lesion'], fontsize = 'large', color = 'grey')
plt.yticks([1, 1.2, 1.4, 1.6, 1.8], [1,'', '', '', 1.8], fontsize = 'large', color = 'grey')
plt.ylabel('Distance metric', fontsize = 'x-large')
sns.despine()
plt.tight_layout()
print(ttest_rel(y_unstr, y_str))
print(ttest_rel(y_unstr, unstruc[unstruc.animal.isin(dls_an)].groupby('animal').d_chance.mean()))
print(ttest_rel(y_str, struc[struc.animal.isin(dls_an)].groupby('animal').d_chance.mean()))
# Statistical annotations using matplotlib
from scipy.stats import ttest_rel
annot = False
if annot:
    # Perform t-tests
    t_stat1, p_val1 = ttest_rel(y_unstr, y_str)
    t_stat2, p_val2 = ttest_rel(y_unstr, unstruc[unstruc.animal.isin(dls_an)].groupby('animal').d_chance.mean())
    t_stat3, p_val3 = ttest_rel(y_str, struc[struc.animal.isin(dls_an)].groupby('animal').d_chance.mean())

    # Add annotations
    max_y = (unstruc[unstruc.animal.isin(dls_an)].groupby('animal').d_chance.mean()).mean()

    if p_val1:
        plt.plot([0, 1], [max_y + 0.02, max_y + 0.02], color='black', linewidth=1)
        plt.text(0.5, max_y + 0.03, f'p={p_val1:.3f}', ha='center', fontsize=10)

    if p_val2:
        plt.plot([width/2 +0.05, width/2 + 0.05], [(y_unstr.mean()+1)/2, (max_y - y_unstr.mean())/2 + y_unstr.mean()], color='black', linewidth=1)
        plt.text(width/2 +0.1, y_unstr.mean() - 0.15, f'p={p_val2:.3f}', ha='center', fontsize=10, rotation = 270)

    if p_val3:
        plt.plot([width/2+1.05, width/2+1.05], [(y_str.mean()+1)/2, (max_y - y_str.mean())/2 + y_str.mean()], color='black', linewidth=1)
        plt.text(width/2+1.1, y_str.mean() - 0.15, f'p={p_val3:.3f}', ha='center', fontsize=10, rotation = 270, color = 'red')
plt.tight_layout()

C:\Users\dlab\AppData\Local\Temp\ipykernel_18528\1447638611.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  d_chances = filtered.groupby(['animal', 'session']).apply(
C:\Users\dlab\AppData\Local\Temp\ipykernel_18528\1447638611.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  d_chances = filtered.groupby(['animal', 'session']).apply(


TtestResult(statistic=np.float64(-1.06874059236244), pvalue=np.float64(0.30621441466674804), df=np.int64(12))
TtestResult(statistic=np.float64(-4.847211786450345), pvalue=np.float64(0.00040012835586105127), df=np.int64(12))
TtestResult(statistic=np.float64(-1.4792296468718562), pvalue=np.float64(0.1648450968697575), df=np.int64(12))


In [ ]:
tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
struc = calc_dist_metric(tempdf, (tempdf.task == 'sham') & (tempdf.animal.isin(sham_an)) & (tempdf.shift_t0 == 1))
tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
unstruc = calc_dist_metric(tempdf, (tempdf.task == 'unstr') & (tempdf.animal.isin(sham_an)) & (tempdf.shift_t0 == 1) & (tempdf.sess_bin >= 4))
y_str = struc[struc.animal.isin(sham_an)].groupby('animal').d_value.mean()
y_unstr = unstruc[unstruc.animal.isin(sham_an)].groupby('animal').d_value.mean()
width = 0.6
plt.figure(figsize=(3.5,4))
plt.bar(0, unstruc.groupby('animal').d_chance.mean().mean(), color = 'grey', alpha = 0.2, width = 0.6)
plt.bar(0, y_unstr.mean(), color = 'xkcd:cornflower', yerr = y_unstr.sem(), width = 0.6)
plt.bar(1, struc.groupby('animal').d_chance.mean().mean(), color = 'grey', alpha = 0.2, width = 0.6)
plt.bar(1, y_str.mean(), color = 'xkcd:slate grey', yerr = y_str.sem(), width = 0.6)
plt.xlim(-0.5, 1.5)
plt.ylim(1,1.9)
plt.xticks([0, 1], ['Pre-lesion', 'Post-lesion'], fontsize = 'large', color = 'grey')
plt.yticks([1, 1.2, 1.4, 1.6, 1.8], [1,'', '', '', 1.8], fontsize = 'large', color = 'grey')
plt.ylabel('Distance metric', fontsize = 'x-large')
sns.despine()
plt.tight_layout()
print(ttest_rel(y_unstr, y_str))
print(ttest_rel(y_unstr, unstruc[unstruc.animal.isin(sham_an)].groupby('animal').d_chance.mean()))
print(ttest_rel(y_str, struc[struc.animal.isin(sham_an)].groupby('animal').d_chance.mean()))
# Statistical annotations using matplotlib
from scipy.stats import ttest_rel

# Perform t-tests
t_stat1, p_val1 = ttest_rel(y_unstr, y_str)
t_stat2, p_val2 = ttest_rel(y_unstr, unstruc[unstruc.animal.isin(sham_an)].groupby('animal').d_chance.mean())
t_stat3, p_val3 = ttest_rel(y_str, struc[struc.animal.isin(sham_an)].groupby('animal').d_chance.mean())

# Add annotations
max_y = (unstruc[unstruc.animal.isin(sham_an)].groupby('animal').d_chance.mean()).mean()

if p_val1:
    plt.plot([0, 1], [max_y + 0.02, max_y + 0.02], color='black', linewidth=1)
    plt.text(0.5, max_y + 0.03, f'p={p_val1:.3f}', ha='center', fontsize=10)

if p_val2:
    plt.plot([width/2 +0.05, width/2 + 0.05], [(y_unstr.mean()+1)/2, (max_y - y_unstr.mean())/2 + y_unstr.mean()], color='black', linewidth=1)
    plt.text(width/2 +0.1, y_unstr.mean() - 0.15, f'p={p_val2:.3f}', ha='center', fontsize=10, rotation = 270)

if p_val3:
    plt.plot([width/2+1.05, width/2+1.05], [(y_str.mean()+1)/2, (max_y - y_str.mean())/2 + y_str.mean()], color='black', linewidth=1)
    plt.text(width/2+1.1, y_str.mean() - 0.15, f'p={p_val3:.3f}', ha='center', fontsize=10, rotation = 270)
plt.tight_layout()

C:\Users\dlab\AppData\Local\Temp\ipykernel_27724\2368743184.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  d_chances = filtered.groupby(['animal', 'session']).apply(
C:\Users\dlab\AppData\Local\Temp\ipykernel_27724\2368743184.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  d_chances = filtered.groupby(['animal', 'session']).apply(


TtestResult(statistic=np.float64(-0.015446702498175658), pvalue=np.float64(0.9880540849631145), df=np.int64(8))
TtestResult(statistic=np.float64(-3.4781724611786644), pvalue=np.float64(0.008341986294539198), df=np.int64(8))
TtestResult(statistic=np.float64(-5.779804117773085), pvalue=np.float64(0.0004146098651563682), df=np.int64(8))


In [ ]:
tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
struc = calc_dist_metric(tempdf, (tempdf.task == 'sham') & (tempdf.animal.isin(sham_an)) & (tempdf.shift_t0 == 1))
tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
unstruc = calc_dist_metric(tempdf, (tempdf.task == 'unstr') & (tempdf.animal.isin(sham_an)) & (tempdf.shift_t0 == 1) & (tempdf.sess_bin >= 4))
y_str = struc[struc.animal.isin(sham_an)].groupby('animal').d_value.mean()
y_unstr = unstruc[unstruc.animal.isin(sham_an)].groupby('animal').d_value.mean()
width = 0.6
plt.figure(figsize=(3.5,4))
plt.bar(0, unstruc.groupby('animal').d_chance.mean().mean(), color = 'grey', alpha = 0.2, width = 0.6)
plt.bar(0, y_unstr.mean(), color = 'xkcd:cornflower', yerr = y_unstr.sem(), width = 0.6)
plt.bar(1, struc.groupby('animal').d_chance.mean().mean(), color = 'grey', alpha = 0.2, width = 0.6)
plt.bar(1, y_str.mean(), color = 'xkcd:slate grey', yerr = y_str.sem(), width = 0.6)
plt.xlim(-0.5, 1.5)
plt.ylim(1,1.9)
plt.xticks([0, 1], ['Pre-lesion', 'Post-lesion'], fontsize = 'large', color = 'grey')
plt.yticks([1, 1.2, 1.4, 1.6, 1.8], [1,'', '', '', 1.8], fontsize = 'large', color = 'grey')
plt.ylabel('Distance metric', fontsize = 'x-large')
sns.despine()
plt.tight_layout()
print(ttest_rel(y_unstr, y_str))
print(ttest_rel(y_unstr, unstruc[unstruc.animal.isin(sham_an)].groupby('animal').d_chance.mean()))
print(ttest_rel(y_str, struc[struc.animal.isin(sham_an)].groupby('animal').d_chance.mean()))

# Statistical annotations using matplotlib
from scipy.stats import ttest_rel

# Perform t-tests
t_stat1, p_val1 = ttest_rel(y_unstr, y_str)
t_stat2, p_val2 = ttest_rel(y_unstr, unstruc[unstruc.animal.isin(sham_an)].groupby('animal').d_chance.mean())
t_stat3, p_val3 = ttest_rel(y_str, struc[struc.animal.isin(sham_an)].groupby('animal').d_chance.mean())

# Add annotations
max_y = (unstruc[unstruc.animal.isin(sham_an)].groupby('animal').d_chance.mean()).mean()

if p_val1:
    plt.plot([0, 1], [max_y + 0.02, max_y + 0.02], color='black', linewidth=1)
    plt.text(0.5, max_y + 0.03, f'p={p_val1:.3f}', ha='center', fontsize=10)

if p_val2:
    plt.plot([width/2 +0.05, width/2 + 0.05], [(y_unstr.mean()+1)/2, (max_y - y_unstr.mean())/2 + y_unstr.mean()], color='black', linewidth=1)
    plt.text(width/2 +0.1, y_unstr.mean() - 0.15, f'p={p_val2:.3f}', ha='center', fontsize=10, rotation = 270)

if p_val3:
    plt.plot([width/2+1.05, width/2+1.05], [(y_str.mean()+1)/2, (max_y - y_str.mean())/2 + y_str.mean()], color='black', linewidth=1)
    plt.text(width/2+1.1, y_str.mean() - 0.15, f'p={p_val3:.3f}', ha='center', fontsize=10, rotation = 270)
plt.tight_layout()

C:\Users\dlab\AppData\Local\Temp\ipykernel_22488\2368743184.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  d_chances = filtered.groupby(['animal', 'session']).apply(
C:\Users\dlab\AppData\Local\Temp\ipykernel_22488\2368743184.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  d_chances = filtered.groupby(['animal', 'session']).apply(


TtestResult(statistic=np.float64(-0.015446702498175658), pvalue=np.float64(0.9880540849631145), df=np.int64(8))
TtestResult(statistic=np.float64(-3.4781724611786644), pvalue=np.float64(0.008341986294539198), df=np.int64(8))
TtestResult(statistic=np.float64(-5.779804117773085), pvalue=np.float64(0.0004146098651563682), df=np.int64(8))


## split by outcome

In [22]:
from utils.dfAnalysisHelpers import calc_dist_metric
hsb_data = add_shift_info(hsb_data)
tempdf = hsb_data.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
les_r = calc_dist_metric(tempdf, (tempdf.task == 'str') & (tempdf.shift_t0 == 1) & (tempdf.reward == 1) & (tempdf.day_task>30))

tempdf = hsb_data.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
les_nr = calc_dist_metric(tempdf, (tempdf.task == 'str') & (tempdf.shift_t0 == 1) & (tempdf.reward == 0) & (tempdf.day_task>30))

sessdf = add_shift_info(sessdf)
tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
pre_r = calc_dist_metric(tempdf, (tempdf.task == 'unstr') & (tempdf.animal.isin(block_an_dict['cued']+block_an_dict['uncued'])) & (tempdf.shift_t0 == 1) & (tempdf.day_task>30) & (tempdf.reward == 1))

tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
pre_nr = calc_dist_metric(tempdf, (tempdf.task == 'unstr') & (tempdf.animal.isin(block_an_dict['cued']+block_an_dict['uncued'])) & (tempdf.shift_t0 == 1) & (tempdf.day_task>30) & (tempdf.reward == 0))

y_pre_r = pre_r[pre_r.animal.isin(block_an_dict['cued']+block_an_dict['uncued'])].groupby('animal').d_value.mean()
y_pre_nr = pre_nr[pre_nr.animal.isin(block_an_dict['cued']+block_an_dict['uncued'])].groupby('animal').d_value.mean()
y_pre_r_chance = pre_r[pre_r.animal.isin(block_an_dict['cued']+block_an_dict['uncued'])].groupby('animal').d_chance.mean()
y_pre_nr_chance = pre_nr[pre_nr.animal.isin(block_an_dict['cued']+block_an_dict['uncued'])].groupby('animal').d_chance.mean()

y_les_r = les_r[les_r.animal.isin(an_dict['hsb_str'])].groupby('animal').d_value.mean()
y_les_nr = les_nr[les_nr.animal.isin(an_dict['hsb_str'])].groupby('animal').d_value.mean()
y_les_r_chance = les_r[les_r.animal.isin(an_dict['hsb_str'])].groupby('animal').d_chance.mean()
y_les_nr_chance = les_nr[les_nr.animal.isin(an_dict['hsb_str'])].groupby('animal').d_chance.mean()

plt.figure(figsize=(3.5,4))
width = 0.4
# plt.bar(0-width/2, y_pre_r_chance.mean(), color = 'grey', alpha = 0.2, width = width)
# plt.bar(0+width/2, y_pre_nr_chance.mean(), color = 'grey', alpha = 0.2, width = width)

plt.bar(0-width/2, y_pre_r.mean(), color = 'red', yerr = y_pre_r.sem(), width = width)
plt.bar(0+width/2, y_pre_nr.mean(), color = 'blue', yerr = y_pre_nr.sem(), width = width)

plt.hlines(y_pre_r_chance.mean(), xmin = -width, xmax = 0, color = 'grey', ls = '--', lw = 2)
plt.hlines(y_pre_nr_chance.mean(), xmin = 0, xmax = width, color = 'grey', ls = '--', lw = 2)

# plt.bar(1-width/2, y_les_r_chance.mean(), color = 'grey', alpha = 0.2, width = width)
# plt.bar(1+width/2, y_les_nr_chance.mean(), color = 'grey', alpha = 0.2, width = width)

plt.bar(1-width/2, y_les_r.mean(), color = 'red', yerr = y_les_r.sem(), width = width)
plt.bar(1+width/2, y_les_nr.mean(), color = 'blue', yerr = y_les_nr.sem(), width = width)

plt.hlines(y_les_r_chance.mean(), xmin = 1-width, xmax = 1, color = 'grey', ls = '--', lw = 2)
plt.hlines(y_les_nr_chance.mean(), xmin = 1, xmax = width+1, color = 'grey', ls = '--', lw = 2)

plt.xlim(-0.5, 1.5)
plt.ylim(1,1.9)
plt.xticks([0, 1], ['Pre-lesion', 'Post-lesion'], fontsize = 'large', color = 'grey')
plt.yticks([1, 1.2, 1.4, 1.6, 1.8], [1,'', '', '', 1.8], fontsize = 'large', color = 'grey')
plt.ylabel('Distance metric', fontsize = 'x-large')
sns.despine()
plt.tight_layout()

c:\Users\dlab\banditAnalysis\utils\dfLoading.py:177: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['shift_t0'] = (df['choice_t1']==df['port']).replace({True: 0, False: 1})
c:\Users\dlab\banditAnalysis\utils\dfLoading.py:178: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['shift_t1'] = (df['choice_t2']==df['port']).replace({True: 0, False: 1})
c:\Users\dlab\banditAnalysis\utils\dfLoading.py:177: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, e

In [ ]:
tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
les_r = calc_dist_metric(tempdf, (tempdf.task == 'sham') & (tempdf.animal.isin(sham_an)) & (tempdf.shift_t0 == 1) & (tempdf.reward == 1))

tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
les_nr = calc_dist_metric(tempdf, (tempdf.task == 'sham') & (tempdf.animal.isin(sham_an)) & (tempdf.shift_t0 == 1) & (tempdf.reward == 0))

tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
pre_r = calc_dist_metric(tempdf, (tempdf.task == 'unstr') & (tempdf.animal.isin(sham_an)) & (tempdf.shift_t0 == 1) & (tempdf.sess_bin >= 4) & (tempdf.reward == 1))

tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
pre_nr = calc_dist_metric(tempdf, (tempdf.task == 'unstr') & (tempdf.animal.isin(sham_an)) & (tempdf.shift_t0 == 1) & (tempdf.sess_bin >= 4) & (tempdf.reward == 0))

y_pre_r = pre_r[pre_r.animal.isin(sham_an)].groupby('animal').d_value.mean()
y_pre_nr = pre_nr[pre_nr.animal.isin(sham_an)].groupby('animal').d_value.mean()
y_pre_r_chance = pre_r[pre_r.animal.isin(sham_an)].groupby('animal').d_chance.mean()
y_pre_nr_chance = pre_nr[pre_nr.animal.isin(sham_an)].groupby('animal').d_chance.mean()

y_les_r = les_r[les_r.animal.isin(sham_an)].groupby('animal').d_value.mean()
y_les_nr = les_nr[les_nr.animal.isin(sham_an)].groupby('animal').d_value.mean()
y_les_r_chance = les_r[les_r.animal.isin(sham_an)].groupby('animal').d_chance.mean()
y_les_nr_chance = les_nr[les_nr.animal.isin(sham_an)].groupby('animal').d_chance.mean()

plt.figure(figsize=(3.5,4))
width = 0.4
# plt.bar(0-width/2, y_pre_r_chance.mean(), color = 'grey', alpha = 0.2, width = width)
# plt.bar(0+width/2, y_pre_nr_chance.mean(), color = 'grey', alpha = 0.2, width = width)

plt.bar(0-width/2, y_pre_r.mean(), color = 'red', yerr = y_pre_r.sem(), width = width)
plt.bar(0+width/2, y_pre_nr.mean(), color = 'blue', yerr = y_pre_nr.sem(), width = width)

plt.hlines(y_pre_r_chance.mean(), xmin = -width, xmax = 0, color = 'grey', ls = '--', lw = 2)
plt.hlines(y_pre_nr_chance.mean(), xmin = 0, xmax = width, color = 'grey', ls = '--', lw = 2)

# plt.bar(1-width/2, y_les_r_chance.mean(), color = 'grey', alpha = 0.2, width = width)
# plt.bar(1+width/2, y_les_nr_chance.mean(), color = 'grey', alpha = 0.2, width = width)

plt.bar(1-width/2, y_les_r.mean(), color = 'red', yerr = y_les_r.sem(), width = width)
plt.bar(1+width/2, y_les_nr.mean(), color = 'blue', yerr = y_les_nr.sem(), width = width)

plt.hlines(y_les_r_chance.mean(), xmin = 1-width, xmax = 1, color = 'grey', ls = '--', lw = 2)
plt.hlines(y_les_nr_chance.mean(), xmin = 1, xmax = width+1, color = 'grey', ls = '--', lw = 2)

plt.xlim(-0.5, 1.5)
plt.ylim(1,1.9)
plt.xticks([0, 1], ['Pre-lesion', 'Post-lesion'], fontsize = 'large', color = 'grey')
plt.yticks([1, 1.2, 1.4, 1.6, 1.8], [1,'', '', '', 1.8], fontsize = 'large', color = 'grey')
plt.ylabel('Distance metric', fontsize = 'x-large')
sns.despine()
plt.tight_layout()

C:\Users\dlab\AppData\Local\Temp\ipykernel_8772\3827990971.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  d_chances = filtered.groupby(['animal', 'session']).apply(
C:\Users\dlab\AppData\Local\Temp\ipykernel_8772\3827990971.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  d_chances = filtered.groupby(['animal', 'session']).apply(
C:\Users\dlab\AppData\Local\Temp\ipykernel_8772\3827990971.py:11: De

In [ ]:
ttest_rel(y_pre_r, y_les_r), ttest_rel(y_pre_nr, y_les_nr)

(TtestResult(statistic=np.float64(-1.0516376713083069), pvalue=np.float64(0.33346442197301746), df=np.int64(6)),
 TtestResult(statistic=np.float64(-1.1102494601715123), pvalue=np.float64(0.30939408024261356), df=np.int64(6)))

In [ ]:
trialsinsess = 100
%matplotlib qt
dls_an = an_dict['dls']
tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
les_r = calc_dist_metric(tempdf, (tempdf.task == 'dls') & (tempdf.animal.isin(dls_an)) & (tempdf.shift_t0 == 1) & (tempdf.reward == 1))

tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
les_nr = calc_dist_metric(tempdf, (tempdf.task == 'dls') & (tempdf.animal.isin(dls_an)) & (tempdf.shift_t0 == 1) & (tempdf.reward == 0))

tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
pre_r = calc_dist_metric(tempdf, (tempdf.task == 'unstr') & (tempdf.animal.isin(dls_an)) & (tempdf.shift_t0 == 1) & (tempdf.sess_bin >= 3) & (tempdf.reward == 1))

tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
pre_nr = calc_dist_metric(tempdf, (tempdf.task == 'unstr') & (tempdf.animal.isin(dls_an)) & (tempdf.shift_t0 == 1) & (tempdf.sess_bin >= 3) & (tempdf.reward == 0))

y_pre_r = pre_r[pre_r.animal.isin(dls_an)].groupby('animal').d_value.mean()
y_pre_nr = pre_nr[pre_nr.animal.isin(dls_an)].groupby('animal').d_value.mean()
y_pre_r_chance = pre_r[pre_r.animal.isin(dls_an)].groupby('animal').d_chance.mean()
y_pre_nr_chance = pre_nr[pre_nr.animal.isin(dls_an)].groupby('animal').d_chance.mean()

y_les_r = les_r[les_r.animal.isin(dls_an)].groupby('animal').d_value.mean()
y_les_nr = les_nr[les_nr.animal.isin(dls_an)].groupby('animal').d_value.mean()
y_les_r_chance = les_r[les_r.animal.isin(dls_an)].groupby('animal').d_chance.mean()
y_les_nr_chance = les_nr[les_nr.animal.isin(dls_an)].groupby('animal').d_chance.mean()
from utils.plotSettings import *
plt.figure(figsize=(3.5,4))
width = 0.4
# plt.bar(0-width/2, y_pre_r_chance.mean(), color = 'grey', alpha = 0.2, width = width)
# plt.bar(0+width/2, y_pre_nr_chance.mean(), color = 'grey', alpha = 0.2, width = width)

plt.bar(0-width/2, y_pre_r.mean(), color = 'red', yerr = y_pre_r.sem(), width = width)
plt.bar(0+width/2, y_pre_nr.mean(), color = 'blue', yerr = y_pre_nr.sem(), width = width)

plt.hlines(y_pre_r_chance.mean(), xmin = -width, xmax = 0, color = 'grey', ls = '--', lw = 2)
plt.hlines(y_pre_nr_chance.mean(), xmin = 0, xmax = width, color = 'grey', ls = '--', lw = 2)

# plt.bar(1-width/2, y_les_r_chance.mean(), color = 'grey', alpha = 0.2, width = width)
# plt.bar(1+width/2, y_les_nr_chance.mean(), color = 'grey', alpha = 0.2, width = width)

plt.bar(1-width/2, y_les_r.mean(), color = 'red', yerr = y_les_r.sem(), width = width)
plt.bar(1+width/2, y_les_nr.mean(), color = 'blue', yerr = y_les_nr.sem(), width = width)

plt.hlines(y_les_r_chance.mean(), xmin = 1 - width, xmax = 1, color = 'grey', ls = '--', lw = 2)
plt.hlines(y_les_nr_chance.mean(), xmin = 1, xmax = 1+ width, color = 'grey', ls = '--', lw = 2)

plt.xlim(-0.5, 1.5)
plt.ylim(1,1.9)
plt.xticks([0, 1], ['Pre-lesion', 'Post-lesion'], fontsize = 'large', color = 'grey')
plt.yticks([1, 1.2, 1.4, 1.6, 1.8], [1,'', '', '', 1.8], fontsize = 'large', color = 'grey')
plt.ylabel('Distance metric', fontsize = 'x-large')
sns.despine()
plt.tight_layout()

C:\Users\dlab\AppData\Local\Temp\ipykernel_18528\1447638611.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  d_chances = filtered.groupby(['animal', 'session']).apply(
C:\Users\dlab\AppData\Local\Temp\ipykernel_18528\1447638611.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  d_chances = filtered.groupby(['animal', 'session']).apply(
C:\Users\dlab\AppData\Local\Temp\ipykernel_18528\1447638611.py:13:

In [ ]:
dms_an = an_dict['dms']
tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
les_r = calc_dist_metric(tempdf, (tempdf.task == 'dms') & (tempdf.animal.isin(dms_an)) & (tempdf.shift_t0 == 1) & (tempdf.reward == 1))

tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
les_nr = calc_dist_metric(tempdf, (tempdf.task == 'dms') & (tempdf.animal.isin(dms_an)) & (tempdf.shift_t0 == 1) & (tempdf.reward == 0))

tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
pre_r = calc_dist_metric(tempdf, (tempdf.task == 'unstr') & (tempdf.animal.isin(dms_an)) & (tempdf.shift_t0 == 1) & (tempdf.sess_bin >= 3) & (tempdf.reward == 1))

tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
pre_nr = calc_dist_metric(tempdf, (tempdf.task == 'unstr') & (tempdf.animal.isin(dms_an)) & (tempdf.shift_t0 == 1) & (tempdf.sess_bin >= 3) & (tempdf.reward == 0))

y_pre_r = pre_r[pre_r.animal.isin(dms_an)].groupby('animal').d_value.mean()
y_pre_nr = pre_nr[pre_nr.animal.isin(dms_an)].groupby('animal').d_value.mean()
y_pre_r_chance = pre_r[pre_r.animal.isin(dms_an)].groupby('animal').d_chance.mean()
y_pre_nr_chance = pre_nr[pre_nr.animal.isin(dms_an)].groupby('animal').d_chance.mean()

y_les_r = les_r[les_r.animal.isin(dms_an)].groupby('animal').d_value.mean()
y_les_nr = les_nr[les_nr.animal.isin(dms_an)].groupby('animal').d_value.mean()
y_les_r_chance = les_r[les_r.animal.isin(dms_an)].groupby('animal').d_chance.mean()
y_les_nr_chance = les_nr[les_nr.animal.isin(dms_an)].groupby('animal').d_chance.mean()

plt.figure(figsize=(3.5,4))
width = 0.4
# plt.bar(0-width/2, y_pre_r_chance.mean(), color = 'grey', alpha = 0.2, width = width)
# plt.bar(0+width/2, y_pre_nr_chance.mean(), color = 'grey', alpha = 0.2, width = width)

plt.bar(0-width/2, y_pre_r.mean(), color = 'red', yerr = y_pre_r.sem(), width = width)
plt.bar(0+width/2, y_pre_nr.mean(), color = 'blue', yerr = y_pre_nr.sem(), width = width)

plt.hlines(y_pre_r_chance.mean(), xmin = -width, xmax = 0, color = 'grey', ls = '--', lw = 2)
plt.hlines(y_pre_nr_chance.mean(), xmin = 0, xmax = width, color = 'grey', ls = '--', lw = 2)

# plt.bar(1-width/2, y_les_r_chance.mean(), color = 'grey', alpha = 0.2, width = width)
# plt.bar(1+width/2, y_les_nr_chance.mean(), color = 'grey', alpha = 0.2, width = width)

plt.bar(1-width/2, y_les_r.mean(), color = 'red', yerr = y_les_r.sem(), width = width)
plt.bar(1+width/2, y_les_nr.mean(), color = 'blue', yerr = y_les_nr.sem(), width = width)

plt.hlines(y_les_r_chance.mean(), xmin = 1-width, xmax = 1, color = 'grey', ls = '--', lw = 2)
plt.hlines(y_les_nr_chance.mean(), xmin = 1, xmax = 1+width, color = 'grey', ls = '--', lw = 2)

plt.xlim(-0.5, 1.5)
plt.ylim(1,1.9)
plt.xticks([0, 1], ['Pre-lesion', 'Post-lesion'], fontsize = 'large', color = 'grey')
plt.yticks([1, 1.2, 1.4, 1.6, 1.8], [1,'', '', '', 1.8], fontsize = 'large', color = 'grey')
plt.ylabel('Distance metric', fontsize = 'x-large')
sns.despine()
plt.tight_layout()

C:\Users\dlab\AppData\Local\Temp\ipykernel_18528\1447638611.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  d_chances = filtered.groupby(['animal', 'session']).apply(
C:\Users\dlab\AppData\Local\Temp\ipykernel_18528\1447638611.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  d_chances = filtered.groupby(['animal', 'session']).apply(
C:\Users\dlab\AppData\Local\Temp\ipykernel_18528\1447638611.py:13:

In [157]:
# hsb_data = add_shift_info(hsb_data)
les_an_list = block_an_dict['unblocked']
tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
les_r = calc_dist_metric(tempdf, (tempdf.task == 'unstr') & (tempdf.animal.isin(les_an_list)) & (tempdf.shift_t0 == 1) & (tempdf.reward == 1) & (tempdf.week_task >= 4) & (tempdf.block_group == 1))

tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
les_nr = calc_dist_metric(tempdf, (tempdf.task == 'unstr') & (tempdf.animal.isin(les_an_list)) & (tempdf.shift_t0 == 1) & (tempdf.reward == 0) & (tempdf.week_task >= 4) & (tempdf.block_group == 1))

pre_an_list = block_an_dict['uncued']+block_an_dict['cued']
tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
pre_r = calc_dist_metric(tempdf, (tempdf.task == 'unstr') & (tempdf.animal.isin(pre_an_list)) & (tempdf.shift_t0 == 1) & (tempdf.week_task >= 4) & (tempdf.reward == 1) & (tempdf.block_group == 1))

tempdf = sessdf.groupby(['animal','session']).filter(lambda x: x.reward.size >= trialsinsess).groupby(['animal','session']).head(trialsinsess)
pre_nr = calc_dist_metric(tempdf, (tempdf.task == 'unstr') & (tempdf.animal.isin(pre_an_list)) & (tempdf.shift_t0 == 1) & (tempdf.week_task >= 4) & (tempdf.reward == 0) & (tempdf.block_group == 1))

y_pre_r = pre_r[pre_r.animal.isin(pre_an_list)].groupby('animal').d_value.mean()
y_pre_nr = pre_nr[pre_nr.animal.isin(pre_an_list)].groupby('animal').d_value.mean()
y_pre_r_chance = pre_r[pre_r.animal.isin(pre_an_list)].groupby('animal').d_chance.mean()
y_pre_nr_chance = pre_nr[pre_nr.animal.isin(pre_an_list)].groupby('animal').d_chance.mean()

y_les_r = les_r[les_r.animal.isin(les_an_list)].groupby('animal').d_value.mean()
y_les_nr = les_nr[les_nr.animal.isin(les_an_list)].groupby('animal').d_value.mean()
y_les_r_chance = les_r[les_r.animal.isin(les_an_list)].groupby('animal').d_chance.mean()
y_les_nr_chance = les_nr[les_nr.animal.isin(les_an_list)].groupby('animal').d_chance.mean()

plt.figure(figsize=(3.5,4))
width = 0.4
plt.bar(0-width/2, y_pre_r_chance.mean(), color = 'grey', alpha = 0.2, width = width)
plt.bar(0+width/2, y_pre_nr_chance.mean(), color = 'grey', alpha = 0.2, width = width)

plt.bar(0-width/2, y_pre_r.mean(), color = 'red', yerr = y_pre_r.sem(), width = width, label = 'Rewarded')
plt.bar(0+width/2, y_pre_nr.mean(), color = 'blue', yerr = y_pre_nr.sem(), width = width, label = 'Unrewarded')

plt.bar(1-width/2, y_les_r_chance.mean(), color = 'grey', alpha = 0.2, width = width)
plt.bar(1+width/2, y_les_nr_chance.mean(), color = 'grey', alpha = 0.2, width = width)

plt.bar(1-width/2, y_les_r.mean(), color = 'red', yerr = y_les_r.sem(), width = width)
plt.bar(1+width/2, y_les_nr.mean(), color = 'blue', yerr = y_les_nr.sem(), width = width)

leg = plt.legend(frameon = False, labelcolor = 'linecolor')
plt.xlim(-0.5, 1.5)
plt.ylim(1,1.9)
plt.xticks([0, 1], ['Blocked', 'Unblocked'], fontsize = 'large', color = 'grey')
plt.yticks([1, 1.2, 1.4, 1.6, 1.8], [1,'', '', '', 1.8], fontsize = 'large', color = 'grey')
plt.ylabel('Switch distance', fontsize = 'x-large')
sns.despine()
plt.tight_layout()

## trialwise & split by reward

In [282]:
#################################### distance moved ######################################
from utils.dfLoading import subset_trials, add_block_groups
subset_df = add_block_groups(sessdf)
subset_df = subset_df[(subset_df.week_task>=4) & (subset_df.block_group == 1) & (subset_df.animal.isin(block_an_dict['uncued']+block_an_dict['cued']+block_an_dict['unblocked']))]

subset_df_unbl = add_block_groups(sessdf)
subset_df_unbl = subset_df_unbl[(subset_df_unbl.week_task>=4) & (subset_df_unbl.block_group == 1) & (subset_df_unbl.animal.isin(block_an_dict['unblocked']))]

subset_df_hsb = add_block_groups(hsb_data)
subset_df_hsb = subset_df_hsb[(subset_df_hsb.block_group == 1)& (subset_df_hsb.week_task>=4)]

trialsinsess = 100
subset_df = subset_trials(subset_df, trialsinsess=trialsinsess, head_trials = trialsinsess)
subset_df_hsb = subset_trials(subset_df_hsb, trialsinsess=trialsinsess, head_trials = trialsinsess)
subset_df_unbl = subset_trials(subset_df_unbl, trialsinsess=trialsinsess, head_trials = trialsinsess)


from utils.dfAnalysisHelpers import get_sessionwise_metric, get_switch_distance_from_ports
port_mean_dict = get_sessionwise_metric(subset_df, metric= 'port', tasks = ['unstr', 'dms', 'dls'])
port_mean_dict_hsb = get_sessionwise_metric(subset_df_hsb, metric = 'port', tasks = ['str'])
rew_mean_dict = get_sessionwise_metric(subset_df, metric = 'reward', tasks = ['unstr', 'dms', 'dls'])
rew_mean_dict_hsb = get_sessionwise_metric(subset_df_hsb, metric = 'reward', tasks = ['str'])

port_mean_dict_hsb_unstr = get_sessionwise_metric(subset_df_hsb[subset_df_hsb.env == 'unstr'], metric= 'port', tasks = ['str'])
port_mean_dict_hsb_str = get_sessionwise_metric(subset_df_hsb[subset_df_hsb.env == 'str'], metric= 'port', tasks = ['str'])
rew_mean_dict_hsb_unstr = get_sessionwise_metric(subset_df_hsb[subset_df_hsb.env == 'unstr'], metric= 'reward', tasks = ['str'])
rew_mean_dict_hsb_str = get_sessionwise_metric(subset_df_hsb[subset_df_hsb.env == 'str'], metric= 'reward', tasks = ['str'])

port_mean_dict_pure_unstr = get_sessionwise_metric(subset_df[subset_df.env == 'unstr'], metric= 'port', tasks = ['unstr'])
port_mean_dict_str_in_unstr = get_sessionwise_metric(subset_df[subset_df.env == 'str'], metric= 'port', tasks = ['unstr'])
rew_mean_dict_pure_unstr = get_sessionwise_metric(subset_df[subset_df.env == 'unstr'], metric= 'reward', tasks = ['unstr'])
rew_mean_dict_str_in_unstr = get_sessionwise_metric(subset_df[subset_df.env == 'str'], metric= 'reward', tasks = ['unstr'])

port_mean_dict_unbl = get_sessionwise_metric(subset_df_unbl, metric = 'port', tasks = ['unstr'])
rew_mean_dict_unbl = get_sessionwise_metric(subset_df_unbl, metric = 'reward', tasks = ['unstr'])

In [283]:
# for single cohort
port_dict = port_mean_dict
rew_dict = rew_mean_dict

avg_reward_switch_0 = np.ones((np.array(list(port_dict.keys())).shape[0], 99))*np.nan
avg_reward_switch_1 = np.ones((np.array(list(port_dict.keys())).shape[0], 99))*np.nan
for idx, (an, env) in enumerate(port_dict.keys()):
    actions = port_dict[(an, env)]
    rewards = rew_dict[(an, env)]
    # get switch trials
    sw_dist = get_switch_distance_from_ports(actions)
    # get trials on which reward was 1 before switch
    reward_switch_1 = np.where(rewards[:, :99]==1, sw_dist, np.nan)
    # get trials on which reward was 0 before switch
    reward_switch_0 = np.where(rewards[:, :99]==0, sw_dist, np.nan)
    # average across sessions
    avg_reward_switch_1[idx, :] = np.nanmean(reward_switch_1, axis=0)
    avg_reward_switch_0[idx, :] = np.nanmean(reward_switch_0, axis=0)

plt.figure(figsize = (5, 4.5))
ax = plt.subplot(111)
mean_r = np.nanmean(avg_reward_switch_1, axis = 0)
sem_r = sem(avg_reward_switch_1, axis = 0, nan_policy = 'omit')
ax.plot(np.arange(99)+0.5, mean_r, color='red', label = 'Rewarded')
ax.fill_between(x = np.arange(99)+0.5, y1 = mean_r - sem_r, y2 = mean_r + sem_r, color = 'red', alpha = 0.3)

mean_nr = np.nanmean(avg_reward_switch_0, axis = 0)
sem_nr = sem(avg_reward_switch_0, axis = 0, nan_policy = 'omit')
ax.plot(np.arange(99)+0.5, mean_nr, color = 'blue',label = 'Unrewarded')
ax.fill_between(x = np.arange(99)+0.5, y1 = mean_nr - sem_nr, y2 = mean_nr + sem_nr, color = 'blue', alpha = 0.3)
ax.set_ylabel('Switch distance')
ax.set_xlim(0,100)
ax.set_xticks(np.arange(0,101,25))
# ax.set_ylim(0, 0.35)
ax.legend(frameon=False, labelcolor = 'linecolor')
plt.tight_layout()

C:\Users\dlab\AppData\Local\Temp\ipykernel_25620\2809630777.py:17: RuntimeWarning: Mean of empty slice
  avg_reward_switch_1[idx, :] = np.nanmean(reward_switch_1, axis=0)
C:\Users\dlab\AppData\Local\Temp\ipykernel_25620\2809630777.py:18: RuntimeWarning: Mean of empty slice
  avg_reward_switch_0[idx, :] = np.nanmean(reward_switch_0, axis=0)


In [281]:
# pairwise comparison
label1, label2 = 'Uncued', 'Cued'
port_dict1 = port_mean_dict
rew_dict1 = rew_mean_dict
port_dict2 = port_mean_dict_unbl
rew_dict2 = rew_mean_dict_unbl

avg1_reward_switch_0 = np.ones((np.array(list(port_dict1.keys())).shape[0], 99))*np.nan
avg1_reward_switch_1 = np.ones((np.array(list(port_dict1.keys())).shape[0], 99))*np.nan
for idx, (an, env) in enumerate(port_dict1.keys()):
    actions = port_dict1[(an, env)]
    rewards = rew_dict1[(an, env)]
    # get switch trials
    sw_dist = get_switch_distance_from_ports(actions)
    # get trials on which reward was 1 before switch
    reward_switch_1 = np.where(rewards[:, :99]==1, sw_dist, np.nan)
    # get trials on which reward was 0 before switch
    reward_switch_0 = np.where(rewards[:, :99]==0, sw_dist, np.nan)
    # average across sessions
    avg1_reward_switch_1[idx, :] = np.nanmean(reward_switch_1, axis=0)
    avg1_reward_switch_0[idx, :] = np.nanmean(reward_switch_0, axis=0)

avg2_reward_switch_0 = np.ones((np.array(list(port_dict2.keys())).shape[0], 99))*np.nan
avg2_reward_switch_1 = np.ones((np.array(list(port_dict2.keys())).shape[0], 99))*np.nan
for idx, (an, env) in enumerate(port_dict2.keys()):
    actions = port_dict2[(an, env)]
    rewards = rew_dict2[(an, env)]
    # get switch trials
    sw_dist = get_switch_distance_from_ports(actions)
    # get trials on which reward was 1 before switch
    reward_switch_1 = np.where(rewards[:, :99]==1, sw_dist, np.nan)
    # get trials on which reward was 0 before switch
    reward_switch_0 = np.where(rewards[:, :99]==0, sw_dist, np.nan)
    # average across sessions
    avg2_reward_switch_1[idx, :] = np.nanmean(reward_switch_1, axis=0)
    avg2_reward_switch_0[idx, :] = np.nanmean(reward_switch_0, axis=0)

plt.figure(figsize = (10, 4.5))
ax = plt.subplot(121)
mean_r = np.nanmean(avg1_reward_switch_1, axis = 0)
sem_r = sem(avg1_reward_switch_1, axis = 0, nan_policy = 'omit')
ax.plot(np.arange(99)+0.5, mean_r, color='red', label = label1)
ax.fill_between(x = np.arange(99)+0.5, y1 = mean_r - sem_r, y2 = mean_r + sem_r, color = 'red', alpha = 0.3)
mean_r = np.nanmean(avg2_reward_switch_1, axis = 0)
sem_r = sem(avg2_reward_switch_1, axis = 0, nan_policy = 'omit')
ax.plot(np.arange(99)+0.5, mean_r, color='xkcd:dark red', label = label2)
ax.fill_between(x = np.arange(99)+0.5, y1 = mean_r - sem_r, y2 = mean_r + sem_r, color = 'xkcd:dark red', alpha = 0.3)
ax.set_ylabel('Switch distance')
ax.set_xlim(0,100)
ax.set_xticks(np.arange(0,101,25))
ax.set_title('Rewarded')
ax.set_ylim(1, 1.75)
ax.legend(frameon=False, labelcolor = 'linecolor')


ax = plt.subplot(122)
mean_nr = np.nanmean(avg1_reward_switch_0, axis = 0)
sem_nr = sem(avg1_reward_switch_0, axis = 0, nan_policy = 'omit')
ax.plot(np.arange(99)+0.5, mean_nr, color = 'blue',label = label1)
ax.fill_between(x = np.arange(99)+0.5, y1 = mean_nr - sem_nr, y2 = mean_nr + sem_nr, color = 'blue', alpha = 0.3)
mean_nr = np.nanmean(avg2_reward_switch_0, axis = 0)
sem_nr = sem(avg2_reward_switch_0, axis = 0, nan_policy = 'omit')
ax.plot(np.arange(99)+0.5, mean_nr, color = 'xkcd:dark blue',label = label2)
ax.fill_between(x = np.arange(99)+0.5, y1 = mean_nr - sem_nr, y2 = mean_nr + sem_nr, color = 'xkcd:dark blue', alpha = 0.3)
ax.set_ylabel('Switch distance')
ax.set_title('Unrewarded')
ax.set_xlim(0,100)
ax.set_xticks(np.arange(0,101,25))
ax.set_ylim(1, 1.75)
ax.legend(frameon=False, labelcolor = 'linecolor')
plt.tight_layout()

C:\Users\dlab\AppData\Local\Temp\ipykernel_25620\2208711243.py:20: RuntimeWarning: Mean of empty slice
  avg1_reward_switch_1[idx, :] = np.nanmean(reward_switch_1, axis=0)
C:\Users\dlab\AppData\Local\Temp\ipykernel_25620\2208711243.py:21: RuntimeWarning: Mean of empty slice
  avg1_reward_switch_0[idx, :] = np.nanmean(reward_switch_0, axis=0)
C:\Users\dlab\AppData\Local\Temp\ipykernel_25620\2208711243.py:35: RuntimeWarning: Mean of empty slice
  avg2_reward_switch_1[idx, :] = np.nanmean(reward_switch_1, axis=0)


# Switch rate

## pairwise comparison

In [407]:
trialsinsess = 100
from utils.dfLoading import subset_trials, add_shift_info
an_list1 = block_an_dict['uncued']#+block_an_dict['uncued']#+block_an_dict['unblocked']
an_list2 = block_an_dict['cued']#+block_an_dict['uncued']#+block_an_dict['unblocked']
# an_list1 = an_dict['hsb_str']
# an_list2 = an_dict['hsb_str']
data1, data2 = sessdf, sessdf
tempdf = subset_trials(data1, trialsinsess=trialsinsess, head_trials=trialsinsess)
tempdf = add_shift_info(tempdf)
mask1 = (tempdf.task == 'unstr') & (tempdf.animal.isin(an_list1)) & (tempdf.week_task>=4) & (tempdf.block_group == 1) #& (tempdf.env == 'str') #& (tempdf.reward == 1)
yr = tempdf[mask1].groupby('animal').shift_t0.mean().mean()
yr_err = tempdf[mask1].groupby('animal').shift_t0.mean().sem()

tempdf = subset_trials(data2, trialsinsess=trialsinsess, head_trials=trialsinsess)
tempdf = add_shift_info(tempdf)
mask2 = (tempdf.task == 'unstr') & (tempdf.animal.isin(an_list2)) & (tempdf.week_task>=4) & (tempdf.block_group == 1) #& (tempdf.env == 'str') #& (tempdf.reward == 0)
ynr = tempdf[mask2].groupby('animal').shift_t0.mean().mean()
ynr_err = tempdf[mask2].groupby('animal').shift_t0.mean().sem()

from utils.plotSettings import *
color1, color2 = 'xkcd:cornflower', 'xkcd:pink'
%matplotlib qt
plt.figure(figsize=(3.5,4))
plt.bar(0, yr, color = color1, yerr = yr_err, width = 0.6)
plt.bar(1, ynr, color = color2, yerr = ynr_err, width = 0.6)
plt.xticks([0, 1], ['Uncued', 'Cued'], fontsize = 'large', color = 'grey')
plt.ylabel('Switch rate', fontsize = 'x-large')
plt.yticks([0, 0.1, 0.2], ['0.0', 0.1, 0.2], fontsize = 'large', color = 'grey')
plt.ylim(0, 0.2)
sns.despine()
plt.tight_layout()

c:\Users\dlab\banditAnalysis\utils\dfLoading.py:141: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['shift_t0'] = (df['choice_t1']==df['port']).replace({True: 0, False: 1})
c:\Users\dlab\banditAnalysis\utils\dfLoading.py:142: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['shift_t1'] = (df['choice_t2']==df['port']).replace({True: 0, False: 1})
c:\Users\dlab\banditAnalysis\utils\dfLoading.py:141: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, e

In [393]:
trialsinsess = 100
from utils.dfLoading import subset_trials
an_list1 = an_dict['hsb_str']#block_an_dict['uncued']#+block_an_dict['cued']+
# an_list2 = block_an_dict['cued']#+block_an_dict['uncued']
an_list2 = an_dict['hsb_str']
data1, data2 = hsb_data, hsb_data
tempdf = subset_trials(data1, trialsinsess=trialsinsess, head_trials=trialsinsess)                                                                                 
mask1 = (tempdf.task == 'str') & (tempdf.block_group == 1) & (tempdf.week_task>=4) & tempdf.animal.isin(an_list1) #& (tempdf.env == 'str')

yr_unstr = tempdf[mask1 & (tempdf.reward == 1)].groupby('animal').shift_t0.mean().mean()
yr_unstr_err = tempdf[mask1 & (tempdf.reward == 1)].groupby('animal').shift_t0.mean().sem()
ynr_unstr = tempdf[mask1 & (tempdf.reward == 0)].groupby('animal').shift_t0.mean().mean()
ynr_unstr_err = tempdf[mask1 & (tempdf.reward == 0)].groupby('animal').shift_t0.mean().sem()

tempdf = subset_trials(data2, trialsinsess=trialsinsess, head_trials=trialsinsess)
mask2 = (tempdf.task == 'str') & (tempdf.block_group == 2) & (tempdf.week_task>=4) & tempdf.animal.isin(an_list2) #& (tempdf.env == 'str')
yr_str = tempdf[mask2 & (tempdf.reward == 1)].groupby('animal').shift_t0.mean().mean()
yr_str_err = tempdf[mask2 & (tempdf.reward == 1)].groupby('animal').shift_t0.mean().sem()
ynr_str = tempdf[mask2 & (tempdf.reward == 0)].groupby('animal').shift_t0.mean().mean()
ynr_str_err = tempdf[mask2 & (tempdf.reward == 0)].groupby('animal').shift_t0.mean().sem()


plt.figure(figsize=(3.5,4))
width = 0.4
plt.bar(0 - width/2, yr_unstr, color = 'red', yerr = yr_unstr_err, width = width, label = 'Rewarded')
plt.bar(0 + width/2, ynr_unstr, color = 'blue', yerr = yr_unstr_err, width = width, label = 'Unrewarded')

plt.bar(1 - width/2, yr_str, color = 'red', yerr = yr_str_err, width = width)
plt.bar(1 + width/2, ynr_str, color = 'blue', yerr = ynr_str_err, width = width)
plt.ylim(0, 0.4)
plt.xticks([0, 1], ['Block 1', 'Block 2'], fontsize = 'large', color = 'grey')
plt.ylabel('Switch rate', fontsize = 'x-large')
# plt.yticks([0, 0.2, 0.4], ['0.0', 0.2, 0.4], fontsize = 'large', color = 'grey')
plt.legend(frameon = False, labelcolor = 'linecolor')
sns.despine()
plt.tight_layout()

In [ ]:
from utils.dfLoading import subset_trials, add_shift_info
sessdf = add_shift_info(sessdf)
trialsinsess = 100
tempdf = subset_trials(sessdf, trialsinsess=trialsinsess, head_trials=trialsinsess)
dms_an = an_dict['dms']
yr_unstr = tempdf[(tempdf.task == 'unstr') & (tempdf.reward == 1) & (tempdf.sess_bin>=3) & tempdf.animal.isin(dms_an)].groupby('animal').shift_t0.mean().mean()
ynr_unstr = tempdf[(tempdf.task == 'unstr') & (tempdf.reward == 0) & (tempdf.sess_bin>=3) & tempdf.animal.isin(dms_an)].groupby('animal').shift_t0.mean().mean()

yr_unstr_err = tempdf[(tempdf.task == 'unstr') & (tempdf.reward == 1) & (tempdf.sess_bin>=3) & tempdf.animal.isin(dms_an)].groupby('animal').shift_t0.mean().sem()
ynr_unstr_err = tempdf[(tempdf.task == 'unstr') & (tempdf.reward == 0) & (tempdf.sess_bin>=3) & tempdf.animal.isin(dms_an)].groupby('animal').shift_t0.mean().sem()

yr_str = tempdf[(tempdf.task == 'dms') & (tempdf.reward == 1) & tempdf.animal.isin(dms_an)].groupby('animal').shift_t0.mean().mean()
ynr_str = tempdf[(tempdf.task == 'dms') & (tempdf.reward == 0) & tempdf.animal.isin(dms_an)].groupby('animal').shift_t0.mean().mean()

yr_str_err = tempdf[(tempdf.task == 'dms') & (tempdf.reward == 1) & tempdf.animal.isin(dms_an)].groupby('animal').shift_t0.mean().sem()
ynr_str_err = tempdf[(tempdf.task == 'dms') & (tempdf.reward == 0) & tempdf.animal.isin(dms_an)].groupby('animal').shift_t0.mean().sem()

plt.figure(figsize=(3.5,4))
width = 0.4
plt.bar(0 - width/2, yr_unstr, color = 'red', yerr = yr_unstr_err, width = width)
plt.bar(0 + width/2, ynr_unstr, color = 'blue', yerr = yr_unstr_err, width = width)

plt.bar(1 - width/2, yr_str, color = 'red', yerr = yr_str_err, width = width)
plt.bar(1 + width/2, ynr_str, color = 'blue', yerr = ynr_str_err, width = width)
plt.ylim(0, 0.45)
plt.xticks([0, 1], ['Pre-lesion', 'Post-lesion'], fontsize = 'large', color = 'grey')
plt.ylabel('Switch rate', fontsize = 'x-large')
plt.yticks([0, 0.2, 0.4], ['0.0', 0.2, 0.4], fontsize = 'large', color = 'grey')
sns.despine()

# Statistical annotations
from scipy.stats import ttest_rel

# Rewarded pre vs post
rewarded_pre = tempdf[(tempdf.task == 'unstr') & (tempdf.reward == 1) & (tempdf.sess_bin>=3) & tempdf.animal.isin(dms_an)].groupby('animal').shift_t0.mean()
rewarded_post = tempdf[(tempdf.task == 'dms') & (tempdf.reward == 1) & tempdf.animal.isin(dms_an)].groupby('animal').shift_t0.mean()
t_stat, p_val = ttest_rel(rewarded_pre, rewarded_post)

# Unrewarded pre vs post
unrewarded_pre = tempdf[(tempdf.task == 'unstr') & (tempdf.reward == 0) & (tempdf.sess_bin>=3) & tempdf.animal.isin(dms_an)].groupby('animal').shift_t0.mean()
unrewarded_post = tempdf[(tempdf.task == 'dms') & (tempdf.reward == 0) & tempdf.animal.isin(dms_an)].groupby('animal').shift_t0.mean()
t_stat2, p_val2 = ttest_rel(unrewarded_pre, unrewarded_post)

# Add annotations
max_nr = max(ynr_unstr + ynr_unstr_err, ynr_str + ynr_str_err)

# if p_val < 0.05:
#     plt.plot([0 - width/2, 1 - width/2], [max_nr + 0.05, max_nr + 0.05], color='black', linewidth=1)
#     plt.text(0.5, max_nr + 0.055, f'p={p_val:.3f}', ha='center', fontsize=10)
# else:
#     plt.plot([0 - width/2, 1 - width/2], [max_nr + 0.05, max_nr + 0.05], color='black', linewidth=1)
#     plt.text(0.5, max_nr + 0.055, 'ns', ha='center', fontsize=10)

# if p_val2 < 0.05:
#     plt.plot([0 + width/2, 1 + width/2], [max_nr + 0.1, max_nr + 0.1], color='black', linewidth=1)
#     plt.text(0.5, max_nr + 0.105, f'p={p_val2:.3f}', ha='center', fontsize=10)
# else:
#     plt.plot([0 + width/2, 1 + width/2], [max_nr + 0.1, max_nr + 0.1], color='black', linewidth=1)
#     plt.text(0.5, max_nr + 0.105, 'ns', ha='center', fontsize=10)
plt.tight_layout()

c:\Users\dlab\banditAnalysis\utils\dfLoading.py:82: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['shift_t0'] = (df['choice_t1']==df['port']).replace({True: 0, False: 1})
c:\Users\dlab\banditAnalysis\utils\dfLoading.py:83: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['shift_t1'] = (df['choice_t2']==df['port']).replace({True: 0, False: 1})


In [ ]:
dls_an = an_dict['dls']
yr_unstr = tempdf[(tempdf.task == 'unstr') & (tempdf.reward == 1) & (tempdf.sess_bin>=3) & tempdf.animal.isin(dls_an)].groupby('animal').shift_t0.mean().mean()
ynr_unstr = tempdf[(tempdf.task == 'unstr') & (tempdf.reward == 0) & (tempdf.sess_bin>=3) & tempdf.animal.isin(dls_an)].groupby('animal').shift_t0.mean().mean()

yr_unstr_err = tempdf[(tempdf.task == 'unstr') & (tempdf.reward == 1) & (tempdf.sess_bin>=3) & tempdf.animal.isin(dls_an)].groupby('animal').shift_t0.mean().sem()
ynr_unstr_err = tempdf[(tempdf.task == 'unstr') & (tempdf.reward == 0) & (tempdf.sess_bin>=3)& tempdf.animal.isin(dls_an)].groupby('animal').shift_t0.mean().sem()

yr_str = tempdf[(tempdf.task == 'dls') & (tempdf.reward == 1) & tempdf.animal.isin(dls_an)].groupby('animal').shift_t0.mean().mean()
ynr_str = tempdf[(tempdf.task == 'dls') & (tempdf.reward == 0) & tempdf.animal.isin(dls_an)].groupby('animal').shift_t0.mean().mean()

yr_str_err = tempdf[(tempdf.task == 'dls') & (tempdf.reward == 1) & tempdf.animal.isin(dls_an)].groupby('animal').shift_t0.mean().sem()
ynr_str_err = tempdf[(tempdf.task == 'dls') & (tempdf.reward == 0) & tempdf.animal.isin(dls_an)].groupby('animal').shift_t0.mean().sem()

plt.figure(figsize=(3.5,4))
width = 0.4
plt.bar(0 - width/2, yr_unstr, color = 'red', yerr = yr_unstr_err, width = width)
plt.bar(0 + width/2, ynr_unstr, color = 'blue', yerr = yr_unstr_err, width = width)

plt.bar(1 - width/2, yr_str, color = 'red', yerr = yr_str_err, width = width)
plt.bar(1 + width/2, ynr_str, color = 'blue', yerr = ynr_str_err, width = width)
plt.ylim(0, 0.45)
plt.xticks([0, 1], ['Pre-lesion', 'Post-lesion'], fontsize = 'large', color = 'grey')
plt.ylabel('Switch rate', fontsize = 'x-large')
plt.yticks([0, 0.2, 0.4], ['0.0', 0.2, 0.4], fontsize = 'large', color = 'grey')
sns.despine()
plt.tight_layout()
# Statistical annotations
from scipy.stats import ttest_rel

# Rewarded pre vs post
rewarded_pre = tempdf[(tempdf.task == 'unstr') & (tempdf.reward == 1) & (tempdf.sess_bin>=3) & tempdf.animal.isin(dls_an)].groupby('animal').shift_t0.mean()
rewarded_post = tempdf[(tempdf.task == 'dls') & (tempdf.reward == 1) & tempdf.animal.isin(dls_an)].groupby('animal').shift_t0.mean()
t_stat, p_val = ttest_rel(rewarded_pre, rewarded_post)

# Unrewarded pre vs post
unrewarded_pre = tempdf[(tempdf.task == 'unstr') & (tempdf.reward == 0) & (tempdf.sess_bin>=3) & tempdf.animal.isin(dls_an)].groupby('animal').shift_t0.mean()
unrewarded_post = tempdf[(tempdf.task == 'dls') & (tempdf.reward == 0) & tempdf.animal.isin(dls_an)].groupby('animal').shift_t0.mean()
t_stat2, p_val2 = ttest_rel(unrewarded_pre, unrewarded_post)

# Add annotations
max_nr = max(ynr_unstr + ynr_unstr_err, ynr_str + ynr_str_err)

# if p_val < 0.05:
#     plt.plot([0 - width/2, 1 - width/2], [max_nr + 0.05, max_nr + 0.05], color='black', linewidth=1)
#     plt.text(0.5, max_nr + 0.055, f'p={p_val:.3f}', ha='center', fontsize=10)
# else:
#     plt.plot([0 - width/2, 1 - width/2], [max_nr + 0.05, max_nr + 0.05], color='black', linewidth=1)
#     plt.text(0.5, max_nr + 0.055, 'ns', ha='center', fontsize=10)

# if p_val2 < 0.05:
#     plt.plot([0 + width/2, 1 + width/2], [max_nr + 0.1, max_nr + 0.1], color='black', linewidth=1)
#     plt.text(0.5, max_nr + 0.105, f'p={p_val2:.3f}', ha='center', fontsize=10)
# else:
#     plt.plot([0 + width/2, 1 + width/2], [max_nr + 0.1, max_nr + 0.1], color='black', linewidth=1)
#     plt.text(0.5, max_nr + 0.105, 'ns', ha='center', fontsize=10)
plt.tight_layout()

In [ ]:
sham_an = an_dict['sham']
yr_unstr = tempdf[(tempdf.task == 'unstr') & (tempdf.reward == 1) & (tempdf.sess_bin>=4) & tempdf.animal.isin(sham_an)].groupby('animal').shift_t0.mean().mean()
ynr_unstr = tempdf[(tempdf.task == 'unstr') & (tempdf.reward == 0) & (tempdf.sess_bin>=4) & tempdf.animal.isin(sham_an)].groupby('animal').shift_t0.mean().mean()

yr_unstr_err = tempdf[(tempdf.task == 'unstr') & (tempdf.reward == 1) & (tempdf.sess_bin>=4) & tempdf.animal.isin(sham_an)].groupby('animal').shift_t0.mean().sem()
ynr_unstr_err = tempdf[(tempdf.task == 'unstr') & (tempdf.reward == 0) & (tempdf.sess_bin>=4)& tempdf.animal.isin(sham_an)].groupby('animal').shift_t0.mean().sem()

yr_str = tempdf[(tempdf.task == 'sham') & (tempdf.reward == 1) & tempdf.animal.isin(sham_an)].groupby('animal').shift_t0.mean().mean()
ynr_str = tempdf[(tempdf.task == 'sham') & (tempdf.reward == 0) & tempdf.animal.isin(sham_an)].groupby('animal').shift_t0.mean().mean()

yr_str_err = tempdf[(tempdf.task == 'sham') & (tempdf.reward == 1) & tempdf.animal.isin(sham_an)].groupby('animal').shift_t0.mean().sem()
ynr_str_err = tempdf[(tempdf.task == 'sham') & (tempdf.reward == 0) & tempdf.animal.isin(sham_an)].groupby('animal').shift_t0.mean().sem()

plt.figure(figsize=(3.5,4))
width = 0.4
plt.bar(0 - width/2, yr_unstr, color = 'red', yerr = yr_unstr_err, width = width)
plt.bar(0 + width/2, ynr_unstr, color = 'blue', yerr = yr_unstr_err, width = width)

plt.bar(1 - width/2, yr_str, color = 'red', yerr = yr_str_err, width = width)
plt.bar(1 + width/2, ynr_str, color = 'blue', yerr = ynr_str_err, width = width)
plt.ylim(0, 0.45)
plt.xticks([0, 1], ['Pre-lesion', 'Post-lesion'], fontsize = 'large', color = 'grey')
plt.ylabel('Switch rate', fontsize = 'x-large')
plt.yticks([0, 0.2, 0.4], ['0.0', 0.2, 0.4], fontsize = 'large', color = 'grey')
sns.despine()

# Statistical annotations
from scipy.stats import ttest_rel

# Rewarded pre vs post
rewarded_pre = tempdf[(tempdf.task == 'unstr') & (tempdf.reward == 1) & (tempdf.sess_bin>=4) & tempdf.animal.isin(sham_an)].groupby('animal').shift_t0.mean()
rewarded_post = tempdf[(tempdf.task == 'sham') & (tempdf.reward == 1) & tempdf.animal.isin(sham_an)].groupby('animal').shift_t0.mean()
t_stat, p_val = ttest_rel(rewarded_pre, rewarded_post)

# Unrewarded pre vs post
unrewarded_pre = tempdf[(tempdf.task == 'unstr') & (tempdf.reward == 0) & (tempdf.sess_bin>=4) & tempdf.animal.isin(sham_an)].groupby('animal').shift_t0.mean()
unrewarded_post = tempdf[(tempdf.task == 'sham') & (tempdf.reward == 0) & tempdf.animal.isin(sham_an)].groupby('animal').shift_t0.mean()
t_stat2, p_val2 = ttest_rel(unrewarded_pre, unrewarded_post)

# Add annotations
max_nr = max(ynr_unstr + ynr_unstr_err, ynr_str + ynr_str_err)-0.02

if p_val < 0.05:
    plt.plot([0 - width/2, 1 - width/2], [max_nr + 0.05, max_nr + 0.05], color='black', linewidth=1)
    plt.text(0.5, max_nr + 0.055, f'p={p_val:.3f}', ha='center', fontsize=10)
else:
    plt.plot([0 - width/2, 1 - width/2], [max_nr + 0.05, max_nr + 0.05], color='black', linewidth=1)
    plt.text(0.5, max_nr + 0.055, 'ns', ha='center', fontsize=10)

if p_val2 < 0.05:
    plt.plot([0 + width/2, 1 + width/2], [max_nr + 0.1, max_nr + 0.1], color='black', linewidth=1)
    plt.text(0.5, max_nr + 0.105, f'p={p_val2:.3f}', ha='center', fontsize=10)
else:
    plt.plot([0 + width/2, 1 + width/2], [max_nr + 0.1, max_nr + 0.1], color='black', linewidth=1)
    plt.text(0.5, max_nr + 0.105, 'ns', ha='center', fontsize=10)
plt.tight_layout()

## trialwise & split by reward

In [ ]:
block = 1
trialsinsess = 100

from utils.dfLoading import subset_trials, add_shift_info, add_block_groups

#################################### distance moved ######################################
from utils.dfLoading import subset_trials, add_block_groups
subset_df = add_block_groups(sessdf)
subset_df = subset_df[(subset_df.week_task>=4) & (subset_df.block_group == 1) & (subset_df.animal.isin(block_an_dict['uncued']))]

subset_df_unbl = add_block_groups(sessdf)
subset_df_unbl = subset_df_unbl[(subset_df_unbl.week_task>=4) & (subset_df_unbl.block_group == 1) & (subset_df_unbl.animal.isin(block_an_dict['cued']))]

subset_df_hsb = add_block_groups(hsb_data)
subset_df_hsb = subset_df_hsb[(subset_df_hsb.block_group == 1)& (subset_df_hsb.week_task>=4)]

trialsinsess = 100
subset_df = subset_trials(subset_df, trialsinsess=trialsinsess, head_trials = trialsinsess)
subset_df_hsb = subset_trials(subset_df_hsb, trialsinsess=trialsinsess, head_trials = trialsinsess)
subset_df_unbl = subset_trials(subset_df_unbl, trialsinsess=trialsinsess, head_trials = trialsinsess)


from utils.dfAnalysisHelpers import get_sessionwise_metric, get_switch_distance_from_ports
port_mean_dict = get_sessionwise_metric(subset_df, metric= 'port', tasks = ['unstr', 'dms', 'dls'])
port_mean_dict_hsb = get_sessionwise_metric(subset_df_hsb, metric = 'port', tasks = ['str'])
rew_mean_dict = get_sessionwise_metric(subset_df, metric = 'reward', tasks = ['unstr', 'dms', 'dls'])
rew_mean_dict_hsb = get_sessionwise_metric(subset_df_hsb, metric = 'reward', tasks = ['str'])

port_mean_dict_hsb_unstr = get_sessionwise_metric(subset_df_hsb[subset_df_hsb.env == 'unstr'], metric= 'port', tasks = ['str'])
port_mean_dict_hsb_str = get_sessionwise_metric(subset_df_hsb[subset_df_hsb.env == 'str'], metric= 'port', tasks = ['str'])
rew_mean_dict_hsb_unstr = get_sessionwise_metric(subset_df_hsb[subset_df_hsb.env == 'unstr'], metric= 'reward', tasks = ['str'])
rew_mean_dict_hsb_str = get_sessionwise_metric(subset_df_hsb[subset_df_hsb.env == 'str'], metric= 'reward', tasks = ['str'])

port_mean_dict_pure_unstr = get_sessionwise_metric(subset_df[subset_df.env == 'unstr'], metric= 'port', tasks = ['unstr'])
port_mean_dict_str_in_unstr = get_sessionwise_metric(subset_df[subset_df.env == 'str'], metric= 'port', tasks = ['unstr'])
rew_mean_dict_pure_unstr = get_sessionwise_metric(subset_df[subset_df.env == 'unstr'], metric= 'reward', tasks = ['unstr'])
rew_mean_dict_str_in_unstr = get_sessionwise_metric(subset_df[subset_df.env == 'str'], metric= 'reward', tasks = ['unstr'])

port_mean_dict_unbl = get_sessionwise_metric(subset_df_unbl, metric = 'port', tasks = ['unstr'])
rew_mean_dict_unbl = get_sessionwise_metric(subset_df_unbl, metric = 'reward', tasks = ['unstr'])

In [243]:
# for single cohort
port_dict = port_mean_dict
rew_dict = rew_mean_dict

avg_reward_switch_0 = np.ones((np.array(list(port_dict.keys())).shape[0], 99))*np.nan
avg_reward_switch_1 = np.ones((np.array(list(port_dict.keys())).shape[0], 99))*np.nan
for idx, (an, env) in enumerate(port_dict.keys()):
    actions = port_dict[(an, env)]
    rewards = rew_dict[(an, env)]
    # get switch trials
    switch_arr = np.diff(actions)
    switch_arr[switch_arr != 0] = 1
    # get trials on which reward was 1 before switch
    reward_switch_1 = (switch_arr == 1) & (rewards[:, :99] == 1)
    # get trials on which reward was 0 before switch
    reward_switch_0 = (switch_arr == 1) & (rewards[:, :99] == 0)
    # average across sessions
    avg_reward_switch_1[idx, :] = np.mean(reward_switch_1, axis=0)
    avg_reward_switch_0[idx, :] = np.mean(reward_switch_0, axis=0)

plt.figure(figsize = (5, 4.5))
ax = plt.subplot(111)
mean_r = np.nanmean(avg_reward_switch_1, axis = 0)
sem_r = sem(avg_reward_switch_1, axis = 0, nan_policy = 'omit')
ax.plot(np.arange(99)+0.5, mean_r, color='red', label = 'Rewarded')
ax.fill_between(x = np.arange(99)+0.5, y1 = mean_r - sem_r, y2 = mean_r + sem_r, color = 'red', alpha = 0.3)

mean_nr = np.nanmean(avg_reward_switch_0, axis = 0)
sem_nr = sem(avg_reward_switch_0, axis = 0, nan_policy = 'omit')
ax.plot(np.arange(99)+0.5, mean_nr, color = 'blue',label = 'Unrewarded')
ax.fill_between(x = np.arange(99)+0.5, y1 = mean_nr - sem_nr, y2 = mean_nr + sem_nr, color = 'blue', alpha = 0.3)
ax.set_ylabel('Switch rate')
ax.set_xlim(0,100)
ax.set_xticks(np.arange(0,101,25))
# ax.set_ylim(0, 0.35)
ax.legend(frameon=False, labelcolor = 'linecolor')
plt.tight_layout()

In [249]:
# pairwise comparison
label1, label2 = 'Uncued', 'Cued'
port_dict1 = port_mean_dict
rew_dict1 = rew_mean_dict
port_dict2 = port_mean_dict_unbl
rew_dict2 = rew_mean_dict_unbl

avg1_reward_switch_0 = np.ones((np.array(list(port_dict1.keys())).shape[0], 99))*np.nan
avg1_reward_switch_1 = np.ones((np.array(list(port_dict1.keys())).shape[0], 99))*np.nan
for idx, (an, env) in enumerate(port_dict1.keys()):
    actions = port_dict1[(an, env)]
    rewards = rew_dict1[(an, env)]
    # get switch trials
    switch_arr = np.diff(actions)
    switch_arr[switch_arr != 0] = 1
    # get trials on which reward was 1 before switch
    reward_switch_1 = (switch_arr == 1) & (rewards[:, :99] == 1)
    # get trials on which reward was 0 before switch
    reward_switch_0 = (switch_arr == 1) & (rewards[:, :99] == 0)
    # average across sessions
    avg1_reward_switch_1[idx, :] = np.mean(reward_switch_1, axis=0)
    avg1_reward_switch_0[idx, :] = np.mean(reward_switch_0, axis=0)

avg2_reward_switch_0 = np.ones((np.array(list(port_dict2.keys())).shape[0], 99))*np.nan
avg2_reward_switch_1 = np.ones((np.array(list(port_dict2.keys())).shape[0], 99))*np.nan
for idx, (an, env) in enumerate(port_dict2.keys()):
    actions = port_dict2[(an, env)]
    rewards = rew_dict2[(an, env)]
    # get switch trials
    switch_arr = np.diff(actions)
    switch_arr[switch_arr != 0] = 1
    # get trials on which reward was 1 before switch
    reward_switch_1 = (switch_arr == 1) & (rewards[:, :99] == 1)
    # get trials on which reward was 0 before switch
    reward_switch_0 = (switch_arr == 1) & (rewards[:, :99] == 0)
    # average across sessions
    avg2_reward_switch_1[idx, :] = np.mean(reward_switch_1, axis=0)
    avg2_reward_switch_0[idx, :] = np.mean(reward_switch_0, axis=0)

plt.figure(figsize = (10, 4.5))
ax = plt.subplot(121)
mean_r = np.nanmean(avg1_reward_switch_1, axis = 0)
sem_r = sem(avg1_reward_switch_1, axis = 0, nan_policy = 'omit')
ax.plot(np.arange(99)+0.5, mean_r, color='red', label = label1)
ax.fill_between(x = np.arange(99)+0.5, y1 = mean_r - sem_r, y2 = mean_r + sem_r, color = 'red', alpha = 0.3)
mean_r = np.nanmean(avg2_reward_switch_1, axis = 0)
sem_r = sem(avg2_reward_switch_1, axis = 0, nan_policy = 'omit')
ax.plot(np.arange(99)+0.5, mean_r, color='xkcd:dark red', label = label2)
ax.fill_between(x = np.arange(99)+0.5, y1 = mean_r - sem_r, y2 = mean_r + sem_r, color = 'xkcd:dark red', alpha = 0.3)
ax.set_ylabel('Switch rate')
ax.set_xlim(0,100)
ax.set_xticks(np.arange(0,101,25))
ax.set_title('Rewarded')
ax.set_ylim(0, 0.35)
ax.legend(frameon=False, labelcolor = 'linecolor')


ax = plt.subplot(122)
mean_nr = np.nanmean(avg1_reward_switch_0, axis = 0)
sem_nr = sem(avg1_reward_switch_0, axis = 0, nan_policy = 'omit')
ax.plot(np.arange(99)+0.5, mean_nr, color = 'blue',label = label1)
ax.fill_between(x = np.arange(99)+0.5, y1 = mean_nr - sem_nr, y2 = mean_nr + sem_nr, color = 'blue', alpha = 0.3)
mean_nr = np.nanmean(avg2_reward_switch_0, axis = 0)
sem_nr = sem(avg2_reward_switch_0, axis = 0, nan_policy = 'omit')
ax.plot(np.arange(99)+0.5, mean_nr, color = 'xkcd:dark blue',label = label2)
ax.fill_between(x = np.arange(99)+0.5, y1 = mean_nr - sem_nr, y2 = mean_nr + sem_nr, color = 'xkcd:dark blue', alpha = 0.3)
ax.set_ylabel('Switch rate')
ax.set_title('Unrewarded')
ax.set_xlim(0,100)
ax.set_xticks(np.arange(0,101,25))
ax.set_ylim(0, 0.35)
ax.legend(frameon=False, labelcolor = 'linecolor')
plt.tight_layout()

## trialwise over multiple blocks 

In [80]:
# blockwise rr diff groups
from scipy.stats import sem
from utils.plotSettings import *

# (block_group, 'first'/'last', n_trials) in x-axis order
SEGMENTS = [(1, 'first', 100), (1, 'last', 25),
            (2, 'first', 100)]
seg_lens = [n for _, _, n in SEGMENTS]
seg_edges = np.cumsum([0] + seg_lens)
n_trials = seg_edges[-1]   # 275
ALLOW_OVERLAP = False      # False: a block needs >= first_n + last_n trials so first/last don't share trials

# set properties
smooth = False
window = 5                    # odd number so the window is centred
smooth_within_blocks = False  # True: don't smooth across segment boundaries

# minimum trials each block needs
need = {}
for bg, _, n in SEGMENTS:
    need[bg] = need.get(bg, 0) + n if not ALLOW_OVERLAP else max(need.get(bg, 0), n)

def session_matrix(df, sess_col='session'):
    """Returns {animal: n_sessions x n_trials reward matrix}; only sessions where every block is long enough."""
    out = {}
    df = df.sort_values('datetime')
    for animal, adf in df.groupby('animal'):
        rows = []
        for _, sdf in adf.groupby(sess_col):
            blocks = {bg: sdf.loc[sdf.block_group == bg, 'shift_t0'].to_numpy() for bg in need}
            if any(len(blocks[bg]) < need[bg] for bg in need):
                continue
            parts = [blocks[bg][:n] if which == 'first' else blocks[bg][-n:] for bg, which, n in SEGMENTS]
            rows.append(np.concatenate(parts))
        if rows:
            out[animal] = np.vstack(rows).astype(float)
    return out

def smooth_rows(mat, window, within_blocks=True):
    """Moving average; near segment edges, averages only over trials inside the segment (no NaNs)."""
    out = np.full_like(mat, np.nan)
    edges = seg_edges if within_blocks else [0, mat.shape[1]]
    k = np.ones(window)
    for s, e in zip(edges[:-1], edges[1:]):
        seg = mat[:, s:e]
        norm = np.convolve(np.ones(seg.shape[1]), k, 'same')   # trials actually inside the window
        out[:, s:e] = np.array([np.convolve(r, k, 'same') for r in seg]) / norm
    return out

# select data 
sessdf = add_shift_info(sessdf)
unstr_mats = session_matrix(sessdf[(sessdf.day_task > 30) & (sessdf.task == 'unstr') & (sessdf.animal.isin(block_an_dict['cued']+block_an_dict['uncued']))], sess_col='session')
hsb_data = add_shift_info(hsb_data)
str_mats   = session_matrix(hsb_data[(hsb_data.day_task > 30) & (hsb_data.task == 'str')], sess_col='session')


# average over sessions (rows) per animal -> animals x n_trials
temp_unstr = np.array([m.mean(axis=0) for m in unstr_mats.values()])
temp_str   = np.array([m.mean(axis=0) for m in str_mats.values()])

if smooth:
    temp_unstr = smooth_rows(temp_unstr, window, smooth_within_blocks)
    temp_str = smooth_rows(temp_str, window, smooth_within_blocks)

mean_unstr = np.nanmean(temp_unstr, axis=0)
mean_str = np.nanmean(temp_str, axis=0)

# plot
plt.figure(figsize=(9, 4.5))
ax = plt.gca()
color1, color2 = 'xkcd:barney purple', 'xkcd:emerald green'
label1, label2 = 'Uncorrelated', 'Correlated'

# split lines only at axis breaks (first 100 -> last 25), not at block switches
break_pos = [b for (_, which, _), b in zip(SEGMENTS[1:], seg_edges[1:-1]) if which == 'last']
switch_pos = [b for (_, which, _), b in zip(SEGMENTS[1:], seg_edges[1:-1]) if which == 'first']
runs = list(zip([0] + break_pos, break_pos + [n_trials]))

# shift x after each axis break so the data has a visible gap there
gap = 2.5   # half-width of the break, in trials
def xmap(t):
    return np.asarray(t) + 2 * gap * np.searchsorted(break_pos, t, side='right')
x = xmap(np.arange(n_trials))

for i, (s, e) in enumerate(runs):
    for mean, temp, c, lab in [(mean_unstr, temp_unstr, color1, label1), (mean_str, temp_str, color2, label2)]:
        err = sem(temp[:, s:e], nan_policy='omit')
        plt.plot(x[s:e], mean[s:e], color=c, label=lab if i == 0 else None)
        plt.fill_between(x[s:e], mean[s:e] + err, mean[s:e] - err, color=c, alpha=0.2)

# block switches: dashed vertical line
for b in switch_pos:
    plt.axvline(xmap(b), color='grey', linestyle='--', linewidth=0.75)

# axis break where the middle of a block is skipped
trans = ax.get_xaxis_transform()   # x in data units, y in axes units
for b in xmap(break_pos) - gap - 0.5:   # centre of each gap in the data
    # patch that cuts the x-axis spine
    ax.add_patch(plt.Rectangle((b - gap, -0.02), 2 * gap, 0.04, transform=trans,
                               facecolor=plt.gcf().get_facecolor(), edgecolor='none',
                               clip_on=False, zorder=3))
    # the two diagonal break marks
    ax.plot([b - gap, b + gap], [0, 0], transform=trans, linestyle='none',
            marker=[(-1, -1.5), (1, 1.5)], markersize=8, markeredgewidth=1,
            color='k', clip_on=False, zorder=4)

plt.xlabel('Trials in session', fontsize='x-large')
plt.ylabel('Switch rate', fontsize='x-large')
plt.xlim(0, xmap(n_trials - 1) + 1)
ticks = np.arange(0, n_trials + 1, 25)
plt.xticks(xmap(ticks), ticks, color='grey', fontsize='large')
leg = plt.legend(frameon=False, loc='upper left')
for line, text in zip(leg.get_lines(), leg.get_texts()):
    text.set_color(line.get_color())
plt.tight_layout()


c:\Users\dlab\banditAnalysis\utils\dfLoading.py:177: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['shift_t0'] = (df['choice_t1']==df['port']).replace({True: 0, False: 1})
c:\Users\dlab\banditAnalysis\utils\dfLoading.py:178: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['shift_t1'] = (df['choice_t2']==df['port']).replace({True: 0, False: 1})
c:\Users\dlab\banditAnalysis\utils\dfLoading.py:177: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, e

## split by distance

In [28]:
# P(switch to distance d | reward), one figure per task
from scipy.stats import sem
from utils.dfLoading import add_shift_info

unstr_df = sessdf[(sessdf.day_task > 30) & (sessdf.task == 'unstr') &
                  (sessdf.animal.isin(block_an_dict['cued'] + block_an_dict['uncued']))].copy()
str_df = hsb_data[(hsb_data.day_task > 30) & (hsb_data.task == 'str')].copy()

def switch_dist_stats(df):
    """Per animal and reward: P(|disp| = d | reward) over all trials, and chance."""
    df = add_shift_info(df.sort_values('datetime'))
    df = df[(df.port > 0) & df.choice_t1.notna()]
    ports = np.sort(df.port.unique())
    dists = np.arange(1, ports.max() - ports.min() + 1)
    obs, chance = {0: [], 1: []}, {0: [], 1: []}
    for (animal, rew), d in df.groupby(['animal', 'reward']):
        absd = d.disp.abs().to_numpy()
        obs[rew].append([(absd == k).mean() for k in dists])
        # chance: keep the animal's switch rate, pick the destination uniformly among other ports
        sw = d[d.shift_t0 == 1]
        if len(sw):
            others = np.abs(ports[None, :] - sw.port.to_numpy()[:, None])
            ch = np.array([(others == k).sum(1) / (len(ports) - 1) for k in dists]).T.mean(0)
        else:
            ch = np.zeros(len(dists))
        chance[rew].append(d.shift_t0.mean() * ch)
    return dists, {k: np.array(v) for k, v in obs.items()}, {k: np.array(v) for k, v in chance.items()}
w = 0.38
LOG_Y = True   # log-scale y-axis for small probabilities
for title, df in [('Correlated', str_df), ('Uncorrelated', unstr_df)]:
    dists, obs, chance = switch_dist_stats(df)
    fig, ax = plt.subplots(figsize=(6, 4))
    for rew, off, c, lab in [(1, -w / 2, 'red', 'Rewarded'), (0, w / 2, 'blue', 'Unrewarded')]:
        m = obs[rew]
        ax.bar(dists + off, m.mean(0), width=w, yerr=sem(m, axis=0), color=c, label=lab,
               error_kw=dict(lw=2, capsize=0))
        ax.hlines(chance[rew].mean(0), dists + off - w / 2, dists + off + w / 2,
                  color='k', linestyle='--', linewidth=1, label='Chance' if rew == 1 else None)
    ax.set_xticks(dists)
    ax.set_xlabel('Switch distance (ports)', fontsize='x-large')
    ax.set_ylabel('P(switch)', fontsize='x-large')
    if LOG_Y:
        ax.set_yscale('log', nonpositive='clip')
    ax.set_title(title, fontsize='x-large')
    ax.legend(frameon=False)
    plt.tight_layout()


c:\Users\dlab\banditAnalysis\utils\dfLoading.py:177: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['shift_t0'] = (df['choice_t1']==df['port']).replace({True: 0, False: 1})
c:\Users\dlab\banditAnalysis\utils\dfLoading.py:178: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['shift_t1'] = (df['choice_t2']==df['port']).replace({True: 0, False: 1})
c:\Users\dlab\banditAnalysis\utils\dfLoading.py:177: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, e

# metric vs/ metric correlation

In [ ]:
%matplotlib qt
# %matplotlib inline
window = 7
trialsinsess = 100

from utils.dfLoading import subset_trials, add_shift_info, add_block_groups
subset_df = add_block_groups(sessdf)
subset_df = subset_df[(subset_df.task == 'unstr') & (subset_df.animal.isin(block_an_dict['cued']+block_an_dict['uncued'])) & (subset_df.sess_bin>= 4)] #& (subset_df.block_group == 1)]
# subset_df = add_shift_info(df)
subset_df = subset_trials(subset_df, trialsinsess=trialsinsess)

hsb_data = hsb_data.sort_values(['animal', 'session', 'eptime'])
subset_df_hsb = add_block_groups(hsb_data)
# subset_df_hsb = subset_df_hsb[(subset_df_hsb.block_group == 1)]
subset_df_hsb = subset_trials(subset_df_hsb, trialsinsess=trialsinsess)

# subset_df['rr'] = subset_df.groupby(['animal', 'session'], as_index = False).reward.rolling(window, center=True).mean().reward
# subset_df_hsb['rr'] = subset_df_hsb.groupby(['animal', 'session'], as_index = False).reward.rolling(window, center=True).mean().reward
# # subset_df_unbl['rr'] = subset_df_unbl.groupby(['animal', 'session'], as_index = False).reward.rolling(window, center=True).mean().reward


# from utils.dfAnalysisHelpers import session_averager
# rr_mean_dict = session_averager(subset_df, metric= 'rr', tasks = ['unstr'])
# rr_mean_dict_hsb = session_averager(subset_df_hsb, metric= 'rr', tasks = ['str'])


hsb_entropy_vals = subset_df_hsb.groupby('animal').port.value_counts(normalize = True).sort_index().reset_index().groupby('animal').apply(lambda x:entropy(x.proportion, base = 2))
rs_entropy_vals = subset_df.groupby('animal').port.value_counts(normalize = True).sort_index().reset_index().groupby('animal').apply(lambda x:entropy(x.proportion, base = 2))


C:\Users\dlab\AppData\Local\Temp\ipykernel_8636\2206612637.py:27: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  hsb_entropy_vals = subset_df_hsb.groupby('animal').port.value_counts(normalize = True).sort_index().reset_index().groupby('animal').apply(lambda x:entropy(x.proportion, base = 2))
C:\Users\dlab\AppData\Local\Temp\ipykernel_8636\2206612637.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  rs_ent

In [ ]:
from scipy.stats import mannwhitneyu
mannwhitneyu(hsb_entropy_vals.values, rs_entropy_vals.values)

MannwhitneyuResult(statistic=np.float64(23.0), pvalue=np.float64(0.09483266991006929))

In [ ]:
pd.concat([hsb_entropy_vals, rs_entropy_vals]).plot(kind = 'bar')
plt.axvline(5.5, color = 'k', ls = '--')
plt.ylabel('Inverse bias')
plt.tight_layout()

In [ ]:
# plot bias v performance
rr_val_unstr = []
for key, value in entropy_mean_dict.items():
    rr_val_unstr.append([np.nanmean(value[80:]), rs_entropy_vals[key[0]]])
rr_val_unstr = np.array(rr_val_unstr)
rr_val_str = []
for key, value in entropy_mean_dict_hsb.items():
    rr_val_str.append([np.nanmean(value[80:]), hsb_entropy_vals[key[0]]])
rr_val_str = np.array(rr_val_str)

plt.scatter(rr_val_str[:, 0], rr_val_str[:, 1], color = 'xkcd:emerald green', label = 'Structured')
plt.scatter(rr_val_unstr[:, 0], rr_val_unstr[:, 1], color = 'xkcd:pumpkin orange', label = 'Unstructured')
plt.xlabel("Sw prob in end of session")
plt.ylabel("Inverse bias (bits)")
# leg = plt.legend(frameon=False)
# for line, text in zip(leg.get_patches(), leg.get_texts()):
#     text.set_color(line.get_color())
plt.legend()
plt.tight_layout()

In [ ]:
#plot number of times each port was chosen/total trials in block as a function of rewprob at that port, for all trials indf; average across blocks
port_choices = [1, 2, 3, 4]
# group by unique rewprob values, find unique rewprob values across all ports first
colors_ports = ['red', 'green', 'blue', 'purple']
# colors_ports = ['xkcd:wine', 'xkcd:grass green', 'xkcd:cerulean', 'xkcd:lighter purple']
# unique_rewprobs = np.unique(tempdf['rewprob'])
# print(unique_rewprobs)
#remove nan from unique_rewprobs
# unique_rewprobs = unique_rewprobs[~np.isnan(unique_rewprobs)]
# print(unique_rewprobs)
pn = 1
plt.figure(figsize = (10,7))
#remove rows with nan rewprob
for an, tempdf in subset_df_hsb.groupby('animal'):

    tempdf = tempdf[~tempdf['rewprob'].isna()]

    #for every block, calculate number of times each port was chosen/total trials in block for each unique rewprob value
    nblocks = tempdf['session'].nunique()
    unique_rewprobs = np.unique(tempdf['rewprob'])
    unique_rewprobs = unique_rewprobs[~np.isnan(unique_rewprobs)]
    port_choice_matrix = np.zeros((len(unique_rewprobs), len(port_choices), nblocks))
    #for all blocks
    #save all rewprobfull per block in a list
    for block, num in enumerate(tempdf['session'].unique()):
        block_data = tempdf[tempdf['session'] == num]
        #for all ports
        for p, port in enumerate(port_choices):
            port_data = block_data[block_data['port'] == port]
            #for all unique rewprobs
            for r, rewprob in enumerate(unique_rewprobs):
                n_choices = len(port_data[port_data['rewprob'] == rewprob])
                port_choice_matrix[r, p, block] = n_choices

    bias_matrix = np.zeros((port_choice_matrix.shape[0], port_choice_matrix.shape[1]))
    counter = np.zeros((port_choice_matrix.shape[0], port_choice_matrix.shape[1]))

    for p in range(port_choice_matrix.shape[1]):
        for r in range(port_choice_matrix.shape[0]):
            bias_matrix[r, p] = 0
            for b in range(port_choice_matrix.shape[2]):
                ntrls = np.sum(port_choice_matrix[:,:,b])
                if ntrls != 0:
                    bias_matrix[r, p] += port_choice_matrix[r, p, b] / ntrls
                if port_choice_matrix[r, p, b] > 0:
                    counter[r, p] += 1

    #divide fin_mat by counter
    for p in range(port_choice_matrix.shape[1]):
        for r in range(port_choice_matrix.shape[0]):
            if counter[r, p] > 0:
                bias_matrix[r, p] = bias_matrix[r, p] / counter[r, p]
    print('Bias matrix:', bias_matrix)


    #plot as line plots for each port
    # plt.figure(figsize=(7, 6))
    ax = plt.subplot(2,3,pn)
    for p, port in enumerate(port_choices):
        ax.plot([0,1,2,3], bias_matrix[:, p], label=f'Port {port}', marker='o', linewidth=2, color=colors_ports[p])
    ax.set_xlabel('Rew. prob.')
    ax.set_xticks([0,1,2,3], labels=[int(r) for r in unique_rewprobs])
    # ax.set_ylabel(f'Fraction chosen (averaged across {nblocks} blocks)')
    ax.set_ylabel(f'{nblocks} blocks')
    ax.set_title(f'{an}')
    ax.set_ylim(0, 0.9)
    pn+=1
    
plt.tight_layout()
    

Bias matrix: [[0.04975962 0.08687494 0.08836152 0.03129964]
 [0.04589626 0.09217039 0.14843437 0.06395682]
 [0.23797001 0.19483884 0.6378567  0.12195234]
 [0.73675359 0.54920584 0.87767973 0.36494481]]
Bias matrix: [[0.06574853 0.13132345 0.09036055 0.04867708]
 [0.05569468 0.13545475 0.14562411 0.06046239]
 [0.15211948 0.59567235 0.27917312 0.10221663]
 [0.46919631 0.76222535 0.53125143 0.63047106]]
Bias matrix: [[0.05921806 0.02790698 0.05981966 0.04330468]
 [0.05191776 0.07081318 0.07295392 0.03874303]
 [0.14802222 0.20147986 0.18183891 0.13844523]
 [0.69559916 0.78738634 0.73611393 0.71764018]]
Bias matrix: [[0.05995081 0.07518393 0.07277742 0.06905383]
 [0.07809739 0.11147881 0.05508976 0.04917793]
 [0.18488405 0.29353647 0.26244049 0.08677344]
 [0.77131076 0.77169489 0.60361523 0.49829125]]
Bias matrix: [[0.03322692 0.01477833 0.07414032 0.06012252]
 [0.05335489 0.04641005 0.07179366 0.06939124]
 [0.26505455 0.1208359  0.22534452 0.22255014]
 [0.82384893 0.55677095 0.68223607 0.7

In [ ]:
# calculate entropy per session
# tempdf[(tempdf.task == 'unstr')&(tempdf.sess_bin>=4)].groupby(['animal', 'session']).entropy.mean().unstack().mean(axis = 1)

# convert to numpy array
# g = tempdf.groupby(['animal', 'session']).cumcount()
# L = np.array(tempdf.set_index(['session',g])
#         .unstack(fill_value=0)
#         .stack().groupby(level=0)
#         .apply(lambda x: x.port.values.tolist())
#         .tolist())
# # mean
# entropy_trial = [entropy(calc_prob(L[:, col]), base = 2) for col in range(L.shape[1])]

y = tempdf[(tempdf.task == 'unstr')&(tempdf.sess_bin>=4)].groupby(['animal', 'session']).port.apply(lambda x: entropy(calc_prob(x[:10]), base = 2)).unstack().mean(axis = 1)

x = tempdf[(tempdf.task == 'unstr')&(tempdf.sess_bin>=4)].groupby(['animal', 'session']).rr.mean().unstack().mean(axis = 1)
# sns.regplot(x =x, y=y, ci = None)
# regress x against y and draw a line with coefficient and intercept from model
from sklearn.linear_model import LinearRegression
from scipy.stats import linregress
model = LinearRegression()
model.fit(x.values.reshape(-1, 1), y.values)
r_sq = model.score(x.values.reshape(-1, 1), y.values)
plt.figure()
plt.scatter(x, y, color = 'grey')
plt.plot(x, model.predict(x.values.reshape(-1, 1)), color = 'red')
slope, intercept, r_value, p_value, std_err = linregress(x,y)
print(p_value, slope, intercept)
print(model.coef_, model.intercept_)
plt.title(fr'$R^2$ = {round(r_sq, 4)}, p = {round(p_value, 4)}', fontsize = 'large')
plt.xlabel('Reward rate', fontsize = 'x-large')

plt.ylabel('Entropy', fontsize = 'x-large')
sns.despine()
plt.tight_layout()

# sns.regplot(x = x, y = y, ci = None)

0.018206987697983328 -1.7331756284868047 1.734430831227499
[-1.73317563] 1.734430831227499


In [ ]:
y = tempdf[(tempdf.task == 'unstr')&(tempdf.sess_bin>=4)].groupby(['animal', 'session']).shift_t0.mean().unstack().mean(axis = 1)
from sklearn.linear_model import LinearRegression
from scipy.stats import linregress
model = LinearRegression()
model.fit(x.values.reshape(-1, 1), y.values)
r_sq = model.score(x.values.reshape(-1, 1), y.values)
plt.figure()
plt.scatter(x, y, color = 'grey')
plt.plot(x, model.predict(x.values.reshape(-1, 1)), color = 'red')
slope, intercept, r_value, p_value, std_err = linregress(x,y)
print(p_value, slope, intercept)
print(model.coef_, model.intercept_)
plt.title(fr'$R^2$ = {round(r_sq, 4)}, p = {round(p_value, 4)}', fontsize = 'large')
plt.xlabel('Performance index', fontsize = 'x-large')
plt.ylabel('Average switch probability', fontsize = 'x-large')
sns.despine()
plt.tight_layout()

In [ ]:
y = tempdf[(tempdf.task == 'unstr')&(tempdf.sess_bin>=4)&(tempdf.shift_t0 == 1)].groupby(['animal', 'session']).disp.apply(lambda x: np.mean(abs(x))).unstack().mean(axis = 1)
from sklearn.linear_model import LinearRegression
from scipy.stats import linregress
model = LinearRegression()
model.fit(x.values.reshape(-1, 1), y.values)
r_sq = model.score(x.values.reshape(-1, 1), y.values)
plt.figure()
plt.scatter(x, y, color = 'grey')
plt.plot(x, model.predict(x.values.reshape(-1, 1)), color = 'red')
slope, intercept, r_value, p_value, std_err = linregress(x,y)
print(p_value, slope, intercept)
print(model.coef_, model.intercept_)
plt.title(fr'$R^2$ = {round(r_sq, 4)}, p = {round(p_value, 4)}', fontsize = 'large')
plt.xlabel('Performance index', fontsize = 'x-large')
plt.ylabel('Average switch distance', fontsize = 'x-large')
sns.despine()
plt.tight_layout()

0.12515671710251056 -2.634573790691113 3.363466412685049
[-2.63457379] 3.363466412685049


# Intervals of trials

In [ ]:
if 'poke_in_time' in sessdf.columns:
    from utils.dfLoading import add_shift_info, subset_trials
    # sessdf = add_shift_info(sessdf)
    mask = (sessdf.task == 'unstr') & (sessdf.week_task>=4) & (sessdf.animal.isin(block_an_dict['cued']+block_an_dict['uncued'])) & (sessdf.block_group==1)
    tempdf = subset_trials(sessdf[mask], trialsinsess = 100, head_trials = 100)
    stay_dict = {}
    switch_dict = {}
    for animal, grp in tempdf.groupby('animal'):
        group = grp.iloc[1:, :]
        stay_curr_trial_in = np.zeros(group.shape[0])*np.nan
        stay_prev_trial_out = np.zeros(group.shape[0])*np.nan
        sw_curr_trial_in = np.zeros(group.shape[0])*np.nan
        sw_prev_trial_out = np.zeros(group.shape[0])*np.nan
        for ii, (jj, tr) in enumerate(group.iterrows()):
            try:
                # if tr.reward == 1:
                if tr.shift_t0 == 0:
                    stay_curr_trial_in[ii] = tr.poke_in_time
                    stay_prev_trial_out[ii] = grp.loc[jj-1, 'last_lick_time']
                elif tr.shift_t0 == 1:
                    sw_curr_trial_in[ii] = tr.poke_in_time
                    sw_prev_trial_out[ii] = grp.loc[jj-1, 'last_lick_time']
                # elif tr.reward == 1:
                #     sw_curr_trial_in[ii] = tr.poke_in_time
                #     sw_prev_trial_out[ii] = grp.loc[jj-1, 'last_lick_time']
                #     sw_curr_trial_in[ii] = tr.poke_in_time
                #     sw_prev_trial_out[ii] = grp.loc[jj-1, 'last_lick_time']
            except KeyError:
                pass
        # STORE PER ANIMAL
        stay_dict[animal] = stay_curr_trial_in-stay_prev_trial_out
        switch_dict[animal] = sw_curr_trial_in-sw_prev_trial_out
        # break            
        
    # curr_trial = tempdf[(tempdf.reward == 0)].dropna(subset = 'reward')
    # prev_trial = tempdf.loc[((tempdf.reward ==0).shift(-1).fillna(False))]
    # stay_time = curr_trial.poke_in_time.to_numpy()[1:] - prev_trial.last_lick_time.to_numpy()

    # # switch
    # curr_trial = tempdf[(tempdf.reward==1)].dropna(subset = 'reward')
    # prev_trial = tempdf.loc[((tempdf.reward == 1).shift(-1).fillna(False))]
    # switch_time = curr_trial.poke_in_time.to_numpy()- prev_trial.last_lick_time.to_numpy()

    # block 1 v block 2 starting
else:
    pass

## trialwise

In [6]:
%matplotlib qt
# %matplotlib inline
window = 7
trialsinsess = 100
exclude_an = ['test05022023', 'Jirachi', 'Xatu', 'Alakazam', 'Emolga', 'Giratina', 'Torchic', 'Whismur', 'Vulpix', 'Yamper']

mask = (sessdf.task == 'unstr') & (sessdf.week_task>=4) & (~sessdf.animal.isin(exclude_an))
diff_start_df = sessdf[mask]
diff_start_df['iti'] = sessdf[mask].groupby(['animal', 'block_num']).poke_in_time.diff()

from utils.dfLoading import subset_trials, add_shift_info, add_block_groups
subset_df = add_block_groups(diff_start_df)
# subset_df = subset_df[(subset_df.sess_bin>=3)] #& (subset_df.block_group == 1)] #& (subset_df.animal.isin(block_an_dict['cued']+block_an_dict['uncued']))]
subset_df = subset_df[(subset_df.week_task>=4) & (subset_df.block_group == 1)& (subset_df.animal.isin(block_an_dict['uncued']+block_an_dict['cued']+block_an_dict['unblocked']))]
# subset_df = add_shift_info(df)

subset_df = subset_trials(subset_df, trialsinsess=trialsinsess, head_trials = trialsinsess)
from utils.dfAnalysisHelpers import get_sessionwise_averages
iti_mean_dict = get_sessionwise_averages(subset_df, metric= 'iti', tasks = ['unstr'])

C:\Users\dlab\AppData\Local\Temp\ipykernel_16444\1217784936.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  diff_start_df['iti'] = sessdf[mask].groupby(['animal', 'block_num']).poke_in_time.diff()
c:\Users\dlab\banditAnalysis\utils\dfLoading.py:121: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sess_block'] = 0
c:\Users\dlab\banditAnalysis\utils\dfLoading.py:124: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = valu

In [97]:
iti_mean_dict.keys()

dict_keys([('Bayleef', 'unstr'), ('Blissey', 'unstr'), ('Chikorita', 'unstr'), ('Cresselia', 'unstr'), ('Darkrai', 'unstr'), ('Eevee', 'unstr'), ('Goldeen', 'unstr'), ('Haxorus', 'unstr'), ('Hoppip', 'unstr'), ('Inkay', 'unstr'), ('Ivysaur', 'unstr'), ('Jigglypuff', 'unstr'), ('Kirlia', 'unstr'), ('Lugia', 'unstr'), ('Mesprit', 'unstr'), ('Nidorina', 'unstr'), ('Oddish', 'unstr'), ('Phione', 'unstr'), ('Quaxly', 'unstr'), ('Quilava', 'unstr'), ('Raltz', 'unstr'), ('Shinx', 'unstr'), ('Umbreon', 'unstr'), ('Uxie', 'unstr'), ('Vanillish', 'unstr'), ('Yanma', 'unstr'), ('Zorua', 'unstr')])

In [7]:
from scipy.optimize import curve_fit
%matplotlib qt
from scipy.stats import sem
from itertools import permutations
# plot average reponse
from utils.plotSettings import *
plt.figure(figsize = (5,4.5))
temp_unstr, temp_str = [], []

# set properties
smooth = False
window = 3
label1, label2 = 'Pre-lesion', 'Post-lesion'
# label1, label2 = 'Pure unstructured', 'Pure structured'
# label1, label2 = 'Block 1', 'Block 2'
color1, color2 = 'xkcd:cornflower', 'xkcd:slate grey'

# set data for plotting
for (animal, env) in iti_mean_dict.keys():
    if env == 'unstr':
        temp_unstr.append(iti_mean_dict[(animal, env)])


temp_unstr = np.array(temp_unstr)
# temp_str = np.array(temp_str)

# smoothing details
if smooth:           
    for i, row in enumerate(temp_unstr):
        temp_unstr[i] = np.convolve(row, np.ones(window)/window, 'same')
        # try:
        #     # temp_str[i] = np.convolve(temp_str[i], np.ones(window)/window, 'same')
        # except IndexError:
        #     pass
    
    temp_unstr = np.array(temp_unstr)
    # temp_str = np.array(temp_str)
    # 
    # works only for odd windows
    temp_unstr[:, 0:(window//2)] = np.nan
    temp_unstr[:, (trialsinsess-window//2):trialsinsess] = np.nan
    
    # temp_str[:, 0:(window//2)] = np.nan
    # temp_str[:, (trialsinsess-window//2):trialsinsess] = np.nan

# take mean of smoothed data   
mean_unstr = (np.nanmean(np.array(temp_unstr), axis = 0))
# mean_str = (np.nanmean(np.array(temp_str), axis = 0))

# plot line 1
plt.plot(np.arange(trialsinsess), mean_unstr, color = color1, label = label1)#, marker = '.') 
plt.fill_between(np.arange(trialsinsess), mean_unstr+sem(temp_unstr, nan_policy = 'omit'),
                mean_unstr-sem(temp_unstr, nan_policy = 'omit'), color = color1, alpha = 0.2)

# plot line 2 (or comment)
# plt.plot(np.arange(trialsinsess), mean_str, color = color2, label = label2)
# plt.fill_between(np.arange(trialsinsess), mean_str+sem(temp_str, nan_policy = 'omit'), 
#                 mean_str-sem(temp_str, nan_policy = 'omit'), color = color2, alpha = 0.2)

# plot significance
# from scipy.stats import ttest_ind
# pvals = ttest_ind(temp_str, temp_unstr)[1]
# pvals_ind = [ind for ind, i in enumerate(pvals) if i < 0.05]
# plt.scatter(pvals_ind, [0.45]*len(pvals_ind), marker = '.', color = 'r', alpha = 0.4) #(6, 2, 60) for asterisks

# chance calc
prob_comb = np.array(list(permutations([80, 46, 15, 10]))+list(permutations([46, 80, 46, 15])))
str_prob_comb = np.array([[80, 46, 15, 10], [46, 80, 46, 15], [15, 46, 80, 46], [10, 15, 46, 80]])
unstr_prob_comb = prob_comb[~(prob_comb[:, None, :] == str_prob_comb[None, :, :]).all(axis = 2).any(axis = 1)]
hsb_prob_comb = [42.06]
# np.mean(unstr_prob_comb)
plt.axhline(np.mean(prob_comb)/100, color = color1, linestyle = '--')
# plt.axhline(np.mean(str_prob_comb)/100, color = color2, linestyle = '--')

# plot settings
plt.xlabel('Trials in session', fontsize = 'x-large')
plt.ylabel('ITI', fontsize = 'x-large')
plt.xlim(0, trialsinsess)
# plt.ylim(0.4, 0.75)
plt.xticks([0, 25, 50, 75, 100], [1, '', '', '', 100], color = 'grey', fontsize = 'large')
# plt.yticks([0.4, 0.5, 0.6, 0.7], [0.4, '', '', 0.7], color = 'grey', fontsize = 'large')
plt.title(f'n = {temp_unstr.shape[0]}')

leg = plt.legend(frameon=False, loc = 'upper left')
for line, text in zip(leg.get_lines(), leg.get_texts()):
    text.set_color(line.get_color())
sns.despine()
# sns.set_context('talk')
plt.tight_layout()
# plt.savefig('C:/Users/dlab/OneDrive - Indian Institute of Science/Drawings/awp1/entropy_n_21.png', dpi = 300)


# if fitting exp curve is reqd
def logFunc(x, a, b):
    return a + b*np.log(x)

# popt_str, _ = curve_fit(logFunc, np.arange(150), temp_str, nan_policy = 'omit')
# popt_unstr, _ = curve_fit(logFunc, np.arange(trialsinsess), mean_unstr, nan_policy = 'omit')
# plt.plot(np.arange(150), logFunc(np.arange(150), *popt_str), '#408ec6', linestyle= 'dotted')
# plt.plot(np.arange(trialsinsess), logFunc(np.arange(trialsinsess), *popt_unstr), color1, linestyle= 'dotted')

C:\Users\dlab\AppData\Local\Temp\ipykernel_16444\1944854747.py:47: RuntimeWarning: Mean of empty slice
  mean_unstr = (np.nanmean(np.array(temp_unstr), axis = 0))
C:\Users\dlab\AppData\Local\Temp\ipykernel_16444\1944854747.py:52: SmallSampleWarning: After omitting NaNs, one or more axis-slices of one or more sample arguments is too small; corresponding elements of returned arrays will be NaN. See documentation for sample size requirements.
  plt.fill_between(np.arange(trialsinsess), mean_unstr+sem(temp_unstr, nan_policy = 'omit'),
C:\Users\dlab\AppData\Local\Temp\ipykernel_16444\1944854747.py:53: SmallSampleWarning: After omitting NaNs, one or more axis-slices of one or more sample arguments is too small; corresponding elements of returned arrays will be NaN. See documentation for sample size requirements.
  mean_unstr-sem(temp_unstr, nan_policy = 'omit'), color = color1, alpha = 0.2)


## lagged analysis

In [8]:
exclude_an = ['test05022023', 'Jirachi', 'Xatu', 'Alakazam', 'Emolga', 'Giratina', 'Torchic', 'Whismur', 'Vulpix', 'Yamper']

mask = (sessdf.task == 'unstr') & (sessdf.week_task>=4) & (~sessdf.animal.isin(exclude_an))
diff_start_df = sessdf[mask]
diff_start_df['iti'] = sessdf[mask].groupby(['animal', 'block_num']).poke_in_time.diff()

C:\Users\dlab\AppData\Local\Temp\ipykernel_16444\3671713491.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  diff_start_df['iti'] = sessdf[mask].groupby(['animal', 'block_num']).poke_in_time.diff()


In [6]:
# sns.displot(data = diff_start_df, x = 'iti', col = 'animal', col_wrap = 5, kind = 'hist')
plt.figure(figsize = (20, 10))
i = 1
for an, group in diff_start_df.groupby('animal'):
    ax = plt.subplot(6,8,i)
    try:
        ax.hist(group['iti'], bins = 30)#, range = (0, 1*60*1000))
    except ValueError:
        pass
    i+=1
    ax.set_title(an)
plt.tight_layout()

c:\Users\dlab\miniforge3\envs\rishika\lib\site-packages\matplotlib\axes\_axes.py:7101: RuntimeWarning: All-NaN slice encountered
  xmin = min(xmin, np.nanmin(xi))
c:\Users\dlab\miniforge3\envs\rishika\lib\site-packages\matplotlib\axes\_axes.py:7102: RuntimeWarning: All-NaN slice encountered
  xmax = max(xmax, np.nanmax(xi))


In [7]:
diff_start_df['iti_bin'] = pd.cut(diff_start_df.iti, np.array([0.01,5,10,30,60,120,300,1500])*1000, labels = ['<5s', '5-10s', '10-30s', '30-60s', '1-2m', '2-5m', '5-15m'])

C:\Users\dlab\AppData\Local\Temp\ipykernel_12236\385822470.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  diff_start_df['iti_bin'] = pd.cut(diff_start_df.iti, np.array([0.01,5,10,30,60,120,300,1500])*1000, labels = ['<5s', '5-10s', '10-30s', '30-60s', '1-2m', '2-5m', '5-15m'])


In [73]:
window = 4
prev_trial_lag = 7
next_trial_lag = 10

for index, trial in diff_start_df[(diff_start_df.iti_bin=='10-30s') & (diff_start_df.animal=='Zorua')].iterrows():
    prev_ports = sessdf.loc[index-prev_trial_lag:index, 'port'] # og port not included
    next_ports = sessdf.loc[index:index+next_trial_lag, 'port'] # og port included 
    # calc entropy over these ports?
    prev_entropy = prev_ports.rolling(window, center=True).apply(lambda x: entropy(calc_prob(x), base = 2))
    next_entropy = next_ports.rolling(window, center=True).apply(lambda x: entropy(calc_prob(x), base = 2))


In [231]:
from numpy.lib.stride_tricks import sliding_window_view
time_bins = [10000, 30000, 60000, 60000*5]
prev_trial_lag = 10
next_trial_lag = 20 # includes current trial
window_size = 3
metric = 'swprob'
# for every time bin
# time_bins = diff_start_df.iti_bin.unique()
an_list = ['Zorua']
prev_entropy = np.zeros((len(time_bins), len(an_list), prev_trial_lag))*np.nan
next_entropy = np.zeros((len(time_bins), len(an_list), next_trial_lag))*np.nan
dataset = diff_start_df[diff_start_df.animal.isin(an_list)]
prev_metric_dict = {}
next_metric_dict = {}
for an_idx, (animal, group_data) in enumerate(dataset.groupby('animal')):

    for t_idx, time_bin in enumerate(time_bins):
        #  for every animal
        iti_trial = group_data[(group_data.iti<=time_bin)]
        prev_ports = np.zeros((iti_trial.shape[0], prev_trial_lag))
        next_ports = np.zeros((iti_trial.shape[0], next_trial_lag))

        for row_idx, (index, trial) in enumerate(iti_trial.iterrows()):
            prev_ports[row_idx, 0:sessdf.loc[index-prev_trial_lag:index-1, 'port'].shape[0]] = sessdf.loc[index-prev_trial_lag:index-1, 'port'] # og port not included
            next_ports[row_idx, 0:sessdf.loc[index:index+next_trial_lag-1, 'port'].shape[0]] = sessdf.loc[index:index+next_trial_lag-1, 'port'] # og port included 
            
        if metric == 'entropy':
            # input matrix - n trials x n lags 
            # for every trial x lag combination, compute Shannon entropy after calculating probability of events in a window of 3
            # for edge cases where window size is too small, output np.nan
            # output matrix - n trials x n lags
            prev_entropy = np.zeros_like(prev_ports)*np.nan
            for row_idx, row in enumerate(prev_ports):
                for window_pos, window in enumerate(sliding_window_view(row, window_size)):
                    idx = window_size - (window_size//2) -1
                    # print(row_idx, idx, window_pos+idx, entropy(calc_prob(window), base = 2), window)
                    prev_entropy[row_idx, window_pos+idx] = entropy(calc_prob(window), base = 2)

            next_entropy = np.zeros_like(next_ports)*np.nan
            for row_idx, row in enumerate(next_ports):
                for window_pos, window in enumerate(sliding_window_view(row, window_size)):
                    idx = window_size - (window_size//2) -1
                    next_entropy[row_idx, window_pos+idx] = entropy(calc_prob(window), base = 2)
            prev_metric_dict[(animal, time_bin)] = prev_entropy
            next_metric_dict[(animal, time_bin)] = next_entropy

        elif metric == 'swprob':
            prev_swprob = np.diff(prev_ports, axis = 1)
            prev_swprob[prev_swprob != 0] = 1
            next_swprob = np.diff(next_ports, axis = 1)
            next_swprob[next_swprob != 0] = 1
            prev_metric_dict[(animal, time_bin)] = prev_swprob
            next_metric_dict[(animal, time_bin)] = next_swprob
        print(animal, time_bin)        
    

Zorua 10000
Zorua 30000
Zorua 60000
Zorua 300000


In [ ]:
from utils.sessdf_indexing_fix import prepare_sessdf_positions, build_lagged_port_arrays
from numpy.lib.stride_tricks import sliding_window_view
time_bins = [2000, 5000, 10000, 30000, 60000, 60000*2, 60000*5, 60000*40]
prev_trial_lag = 10
next_trial_lag = 20
window_size = 3
metric = 'reward'
an_list =  diff_start_df.animal.unique()
# Rewrite of the lag-window block using explicit row positions instead of index arithmetic.
diff_start_df_ordered = prepare_sessdf_positions(diff_start_df)
dataset = diff_start_df_ordered[diff_start_df_ordered.animal.isin(an_list)]

prev_metric_dict = {}
next_metric_dict = {}

for an_idx, (animal, group_data) in enumerate(dataset.groupby('animal')):
    for t_idx, time_bin in enumerate(time_bins):
        iti_trial, prev_ports, next_ports = build_lagged_port_arrays(
            group_data,
            least_time_bin=time_bins[t_idx-1] if t_idx-1>-1 else 0,
            most_time_bin=time_bin,
            prev_trial_lag=prev_trial_lag,
            next_trial_lag=next_trial_lag,
            metric_col=metric,
            time_col='iti',
        )

        if metric == 'entropy':
            prev_entropy = np.zeros_like(prev_ports) * np.nan
            for row_idx, row in enumerate(prev_ports):
                for window_pos, window in enumerate(sliding_window_view(row, window_size)):
                    idx = window_size - (window_size // 2) - 1
                    prev_entropy[row_idx, window_pos + idx] = entropy(calc_prob(window), base=2)

            next_entropy = np.zeros_like(next_ports) * np.nan
            for row_idx, row in enumerate(next_ports):
                for window_pos, window in enumerate(sliding_window_view(row, window_size)):
                    idx = window_size - (window_size // 2) - 1
                    next_entropy[row_idx, window_pos + idx] = entropy(calc_prob(window), base=2)

            prev_metric_dict[(animal, time_bin)] = prev_entropy
            next_metric_dict[(animal, time_bin)] = next_entropy

        elif metric == 'swprob':
            prev_ports[np.isnan(prev_ports)] = 0
            prev_swprob = np.diff(prev_ports, axis=1)
            prev_swprob[prev_swprob != 0] = 1
            next_ports[np.isnan(next_ports)] = 0
            next_swprob = np.diff(next_ports, axis=1)
            next_swprob[next_swprob != 0] = 1
            prev_metric_dict[(animal, time_bin)] = prev_swprob
            next_metric_dict[(animal, time_bin)] = next_swprob

        elif metric == 'reward':
            prev_metric_dict[(animal, time_bin)] = prev_ports
            next_metric_dict[(animal, time_bin)] = next_ports

        print(animal, time_bin)

Bayleef 2000
Bayleef 5000
Bayleef 10000
Bayleef 30000
Bayleef 60000
Bayleef 120000
Bayleef 300000
Bayleef 2400000
Blissey 2000
Blissey 5000
Blissey 10000
Blissey 30000
Blissey 60000
Blissey 120000
Blissey 300000
Blissey 2400000
Chikorita 2000
Chikorita 5000
Chikorita 10000
Chikorita 30000
Chikorita 60000
Chikorita 120000
Chikorita 300000
Chikorita 2400000
Cresselia 2000
Cresselia 5000
Cresselia 10000
Cresselia 30000
Cresselia 60000
Cresselia 120000
Cresselia 300000
Cresselia 2400000
Darkrai 2000
Darkrai 5000
Darkrai 10000
Darkrai 30000
Darkrai 60000
Darkrai 120000
Darkrai 300000
Darkrai 2400000
Eevee 2000
Eevee 5000
Eevee 10000
Eevee 30000
Eevee 60000
Eevee 120000
Eevee 300000
Eevee 2400000
Goldeen 2000
Goldeen 5000
Goldeen 10000
Goldeen 30000
Goldeen 60000
Goldeen 120000
Goldeen 300000
Goldeen 2400000
Haxorus 2000
Haxorus 5000
Haxorus 10000
Haxorus 30000
Haxorus 60000
Haxorus 120000
Haxorus 300000
Haxorus 2400000
Hoppip 2000
Hoppip 5000
Hoppip 10000
Hoppip 30000
Hoppip 60000
Hoppip 12

In [10]:
plt.figure()
hist_2d = np.zeros((len(time_bins), dataset.animal.unique().shape[0])) 
for t_idx, time_bin in enumerate(time_bins):
    l = []

    ax = plt.subplot(2,4,t_idx+1)
    ax.set_ylim(0., 1)
    ax.set_title(f'{time_bin/1000}s')
    for an_idx, (animal, group_data) in enumerate(dataset.groupby('animal')):   
        ax.plot(
            np.hstack([
                np.nanmean(prev_metric_dict[(animal, time_bin)], axis = 0),
                np.nanmean(next_metric_dict[(animal, time_bin)], axis = 0)
            ]),
            marker = '.',
            alpha = 0.2,
            label = f'{animal} {prev_metric_dict[(animal, time_bin)].shape[0]} trials'
        )
        l.append(np.hstack([
                np.nanmean(prev_metric_dict[(animal, time_bin)], axis = 0),
                np.nanmean(next_metric_dict[(animal, time_bin)], axis = 0)
            ]))
        if an_idx == 0:
            ax.axvline(prev_metric_dict[(animal, time_bin)].shape[1], color = 'k', alpha = 0.7)
        hist_2d[t_idx, an_idx] = prev_metric_dict[(animal, time_bin)].shape[0]
    ax.plot(np.nanmean(l, axis = 0), '.-k')
    if metric == 'swprob':
        ax.set_xticks([-1, 9, 29], [-10, 0, 20])  
    elif metric == 'reward':
        ax.set_xticks([0, 10, 30], [-10, 0, 20])  
    ax.set_xlabel('Trials from iti')
    ax.set_ylabel('Reward rate')
        # ax.legend()

plt.tight_layout()

C:\Users\dlab\AppData\Local\Temp\ipykernel_16444\1952245487.py:12: RuntimeWarning: Mean of empty slice
  np.nanmean(prev_metric_dict[(animal, time_bin)], axis = 0),
C:\Users\dlab\AppData\Local\Temp\ipykernel_16444\1952245487.py:13: RuntimeWarning: Mean of empty slice
  np.nanmean(next_metric_dict[(animal, time_bin)], axis = 0)
C:\Users\dlab\AppData\Local\Temp\ipykernel_16444\1952245487.py:20: RuntimeWarning: Mean of empty slice
  np.nanmean(prev_metric_dict[(animal, time_bin)], axis = 0),
C:\Users\dlab\AppData\Local\Temp\ipykernel_16444\1952245487.py:21: RuntimeWarning: Mean of empty slice
  np.nanmean(next_metric_dict[(animal, time_bin)], axis = 0)
C:\Users\dlab\AppData\Local\Temp\ipykernel_16444\1952245487.py:13: RuntimeWarning: Mean of empty slice
  np.nanmean(next_metric_dict[(animal, time_bin)], axis = 0)
C:\Users\dlab\AppData\Local\Temp\ipykernel_16444\1952245487.py:21: RuntimeWarning: Mean of empty slice
  np.nanmean(next_metric_dict[(animal, time_bin)], axis = 0)
C:\Users\dlab\

In [83]:
from matplotlib.colors import LogNorm
plt.figure()
sns.heatmap(hist_2d.T,  cmap = 'viridis', xticklabels = np.array(time_bins)/1000, yticklabels=dataset.animal.unique(), norm = LogNorm())
# plt.imshow(np.log10(hist_2d.T),  cmap = 'cividis',norm = LogNorm())
plt.tight_layout()

# Learning rate with block changes | distances

In [511]:
# find highest rp for block 1, given any block 2 
an_list = block_an_dict['cued']+block_an_dict['uncued']
subset_df = add_block_groups(sessdf)
subset_df = subset_df[(subset_df.sess_bin>=4) & (subset_df.block_group == 2) & (subset_df.animal.isin(an_list)) & (subset_df.task == 'unstr')]
subset_df = subset_trials(subset_df, trialsinsess=trialsinsess, head_trials = trialsinsess)
# rr_mean_dict = get_sessionwise_averages(subset_df_unbl, metric= 'rr', tasks = ['unstr', 'dms', 'dls'])

# get rps of every block 2 that you need
bl2_rps = subset_df.groupby(['animal', 'session']).rewprobfull.head(1)
bl2_rps_arr = np.zeros((bl2_rps.shape[0], 4))
sess_arr = subset_df.groupby(['animal', 'session']).session.head(1).to_numpy()
an_arr = subset_df.groupby(['animal', 'session']).animal.head(1).to_numpy()
for idx, row in enumerate(bl2_rps):
    bl2_rps_arr[idx, :] = np.array(row.strip('[]').split(' '), dtype = float)

# create lookup table for block 1 rps
bl1_lookup = sessdf[sessdf.block_group == 1].groupby(['animal', 'session'])['rewprobfull'].first()
# map values
bl1_rps_arr = np.zeros((bl2_rps.shape[0], 4))
for i in range(len(sess_arr)):
    try:
        val = bl1_lookup[(an_arr[i], sess_arr[i])]
        bl1_rps_arr[i, :] = np.array(val.strip('[]').split(' '), dtype=float)
    except KeyError:
        bl1_rps_arr[i, :] = np.ones(4)*np.nan

# distance between diff blocks
distance_bw_blocks = np.abs(np.argmax(bl1_rps_arr==80, axis = 1) - np.argmax(bl2_rps_arr==80, axis = 1))
# map with an and sess info
dist_info = pd.DataFrame(np.vstack([an_arr, sess_arr, distance_bw_blocks]).T, index = bl2_rps.index, columns=['animal', 'session', 'bl_dist'])
# merge!
subset_df = subset_df.merge(dist_info, on = ['animal', 'session'], how = 'left')

# create diff subsets by subselection
subset_df_d0 = subset_df[subset_df.bl_dist == 0]
subset_df_d1 = subset_df[subset_df.bl_dist == 1]
subset_df_d2 = subset_df[subset_df.bl_dist == 2]
subset_df_d3 = subset_df[subset_df.bl_dist == 3]

rr_mean_dict_d0 = get_sessionwise_averages(subset_df_d0, metric= 'shift_t0', tasks = ['unstr'])
rr_mean_dict_d1 = get_sessionwise_averages(subset_df_d1, metric= 'shift_t0', tasks = ['unstr'])
rr_mean_dict_d2 = get_sessionwise_averages(subset_df_d2, metric= 'shift_t0', tasks = ['unstr'])
rr_mean_dict_d3 = get_sessionwise_averages(subset_df_d3, metric= 'shift_t0', tasks = ['unstr'])

In [514]:
from scipy.optimize import curve_fit
%matplotlib qt
from scipy.stats import sem
from itertools import permutations
# plot average reponse
from utils.plotSettings import *
plt.figure(figsize = (5,4.5))
trialsinsess = 100
# set properties
smooth = False
include_hsb = False
window = 3
label1, label2, label3, label4 = 'unchanged high', 'dist 1', 'dist 2', 'dist 3'
colors = plt.cm.viridis(np.linspace(0, 1, 5))

# set data for plotting
d0_data = np.array(list(rr_mean_dict_d0.values()))
d1_data = np.array(list(rr_mean_dict_d1.values()))
d2_data = np.array(list(rr_mean_dict_d2.values()))
d3_data = np.array(list(rr_mean_dict_d3.values()))

mean_d0 = np.nanmean(d0_data, axis = 0)
mean_d1 = np.nanmean(d1_data, axis = 0)
mean_d2 = np.nanmean(d2_data, axis = 0)
mean_d3 = np.nanmean(d3_data, axis = 0)

# plot line 1
plt.plot(np.arange(trialsinsess), mean_d0, color = colors[0], label = label1)#, marker = '.') 
plt.fill_between(np.arange(trialsinsess), mean_d0+sem(d0_data, nan_policy = 'omit'),
                mean_d0-sem(d0_data, nan_policy = 'omit'), color = colors[0], alpha = 0.2)
    
plt.plot(np.arange(trialsinsess), mean_d1, color = colors[1], label = label2)#, marker = '.') 
plt.fill_between(np.arange(trialsinsess), mean_d1+sem(d1_data, nan_policy = 'omit'),
                mean_d1-sem(d1_data, nan_policy = 'omit'), color = colors[1], alpha = 0.2)

plt.plot(np.arange(trialsinsess), mean_d2, color = colors[2], label = label3)#, marker = '.') 
plt.fill_between(np.arange(trialsinsess), mean_d2+sem(d2_data, nan_policy = 'omit'),
                mean_d2-sem(d2_data, nan_policy = 'omit'), color = colors[2], alpha = 0.2)

plt.plot(np.arange(trialsinsess), mean_d3, color = colors[3], label = label4)#, marker = '.') 
plt.fill_between(np.arange(trialsinsess), mean_d3+sem(d3_data, nan_policy = 'omit'),
                mean_d3-sem(d3_data, nan_policy = 'omit'), color = colors[3], alpha = 0.2)


# smoothing details
# if smooth:           
#     for i, row in enumerate(temp_unstr):
#         temp_unstr[i] = np.convolve(row, np.ones(window)/window, 'same')
#         try:
#             temp_str[i] = np.convolve(temp_str[i], np.ones(window)/window, 'same')
#         except IndexError:
#             pass
    
#     temp_unstr = np.array(temp_unstr)
#     temp_str = np.array(temp_str)
#     # 
#     # works only for odd windows
#     temp_unstr[:, 0:(window//2)] = np.nan
#     temp_unstr[:, (trialsinsess-window//2):trialsinsess] = np.nan
    
#     temp_str[:, 0:(window//2)] = np.nan
#     temp_str[:, (trialsinsess-window//2):trialsinsess] = np.nan

# plot significance
# from scipy.stats import ttest_ind
# pvals = ttest_ind(temp_str, temp_unstr)[1]
# pvals_ind = [ind for ind, i in enumerate(pvals) if i < 0.05]
# plt.scatter(pvals_ind, [0.45]*len(pvals_ind), marker = '.', color = 'r', alpha = 0.4) #(6, 2, 60) for asterisks

# chance calc
prob_comb = np.array(list(permutations([80, 46, 15, 10]))+list(permutations([46, 80, 46, 15])))
str_prob_comb = np.array([[80, 46, 15, 10], [46, 80, 46, 15], [15, 46, 80, 46], [10, 15, 46, 80]])
unstr_prob_comb = prob_comb[~(prob_comb[:, None, :] == str_prob_comb[None, :, :]).all(axis = 2).any(axis = 1)]
hsb_prob_comb = [42.06]
# np.mean(unstr_prob_comb)
# plt.axhline(np.mean(prob_comb)/100, color = 'k', linestyle = '--')
# plt.axhline(np.mean(str_prob_comb)/100, color = color2, linestyle = '--')

# plot settings
plt.xlabel('Trials in session', fontsize = 'x-large')
plt.ylabel('Switch rate', fontsize = 'x-large')
plt.xlim(0, trialsinsess)
# plt.ylim(0.4, 0.75)
plt.xticks([0, 25, 50, 75, 100], [1, '', '', '', 100], color = 'grey', fontsize = 'large')
# plt.yticks(np.arange(2,9)*0.1, [0.2, '', '', '','','', 0.8], color = 'grey', fontsize = 'large')
# plt.title(f'n = {d0_data.shape[0]}')

leg = plt.legend(frameon=False, loc = 'best')
for line, text in zip(leg.get_lines(), leg.get_texts()):
    text.set_color(line.get_color())
sns.despine()
# sns.set_context('talk')
plt.tight_layout()
# plt.savefig('C:/Users/dlab/OneDrive - Indian Institute of Science/Drawings/awp1/entropy_n_21.png', dpi = 300)


# if fitting exp curve is reqd
def logFunc(x, a, b):
    return a + b*np.log(x)

# popt_str, _ = curve_fit(logFunc, np.arange(150), temp_str, nan_policy = 'omit')
# popt_unstr, _ = curve_fit(logFunc, np.arange(trialsinsess), mean_unstr, nan_policy = 'omit')
# plt.plot(np.arange(150), logFunc(np.arange(150), *popt_str), '#408ec6', linestyle= 'dotted')
# plt.plot(np.arange(trialsinsess), logFunc(np.arange(trialsinsess), *popt_unstr), color1, linestyle= 'dotted')

# LZW compressibility

In [74]:
#################################### rr ######################################
# set_cwd('/home/rishika/sim/')
%matplotlib qt
# %matplotlib inline
window = 7
trialsinsess = 100
from utils.dfLoading import subset_trials, add_shift_info, add_block_groups
subset_df = add_block_groups(sessdf)
# subset_df = subset_df[(subset_df.sess_bin>=3)] #& (subset_df.block_group == 1)] #& (subset_df.animal.isin(block_an_dict['cued']+block_an_dict['uncued']))]
subset_df = subset_df[(subset_df.week_task>=4) & (subset_df.block_group == 1)& (subset_df.animal.isin(block_an_dict['uncued']+block_an_dict['cued']))]
#+block_an_dict['cued']))]
# subset_df = add_shift_info(df)
subset_df = subset_trials(subset_df, trialsinsess=trialsinsess, head_trials = trialsinsess)
from utils.dfAnalysisHelpers import get_sessionwise_metric
port_dict = get_sessionwise_metric(subset_df, metric= 'port', tasks = ['unstr', 'dms', 'dls', 'sham'])
subset_df_unbl = add_block_groups(sessdf)
subset_df_unbl = subset_df_unbl[(subset_df_unbl.sess_bin>=4) & (subset_df_unbl.block_group == 2) & (subset_df_unbl.animal.isin(block_an_dict['uncued']+block_an_dict['cued']))]#& (subset_df_unbl.animal.isin(block_an_dict['unblocked']))]
# # subset_df = add_shift_info(df)
subset_df_unbl = subset_trials(subset_df_unbl, trialsinsess=trialsinsess, head_trials = trialsinsess)
port_dict_unbl = get_sessionwise_metric(subset_df_unbl, metric= 'port', tasks = ['unstr'])

from utils.lzw_helpers import lzw_compress
lzw_port_dict = {}
for entry, val in port_dict.items():
    lzw_port_dict[entry] = np.array([lzw_compress("".join(str(num) for num in row), return_random_ratio=True)[1] for row in port_dict[entry].T])
lzw_port_dict_unbl = {}
for entry, val in port_dict_unbl.items():
    lzw_port_dict_unbl[entry] = np.array([lzw_compress("".join(str(num) for num in row), return_random_ratio=True)[1] for row in port_dict_unbl[entry].T])

In [85]:
lzw_compress("4132112222222222221333212334433332332334444444444211221132222443232121122412343333211111222213221222"
, return_random_ratio=True, n_random_samples=500)

([0,
  1,
  2,
  3,
  1,
  1,
  3,
  10,
  11,
  12,
  10,
  5,
  2,
  6,
  9,
  16,
  0,
  0,
  16,
  16,
  3,
  23,
  19,
  20,
  27,
  28,
  7,
  9,
  30,
  6,
  11,
  20,
  6,
  17,
  30,
  10,
  4,
  24,
  21,
  22,
  30,
  8,
  31,
  14,
  33,
  46],
 1.155608695652174)

In [82]:
from scipy.optimize import curve_fit
%matplotlib qt
from scipy.stats import sem
from itertools import permutations
# plot average reponse
from utils.plotSettings import *
plt.figure(figsize = (5,4.5))
temp_unstr, temp_str = [], []

# set properties
smooth = False
window = 3
# label1, label2 = 'Pre-lesion', 'Post-lesion'
# label1, label2 = 'Pure unstructured', 'Pure structured'
label1, label2 = 'Block 1', 'Block 2'
# color1, color2 = 'xkcd:cornflower', 'xkcd:slate grey'
# color1, color2 = 'xkcd:cornflower', 'xkcd:pale purple'
color1, color2 = sns.color_palette('crest')[0], sns.color_palette('crest')[4]
# color1, color2 = 'xkcd:pumpkin orange', 'xkcd:emerald green'
# color1, color2 = 'xkcd:olive green', 'xkcd:emerald green'
# # color1, color2 = '#7a2048', '#408ec6'
sel_animal = 'Blissey'
cond = 'unstr'
# set data for plotting
for (animal, env) in lzw_port_dict.keys():
    if animal in an_dict[cond]:
        if env == 'unstr':
            temp_unstr.append(lzw_port_dict[(animal, env)])
        # plt.plot(lzw_port_dict[animal, env], alpha = 0.2)

for (animal, env) in lzw_port_dict_unbl.keys():
    if animal in an_dict[cond]:
        if env == cond:
            temp_str.append(lzw_port_dict_unbl[(animal, env)])


temp_unstr = np.array(temp_unstr)
temp_str = np.array(temp_str)

# smoothing details
if smooth:           
    for i, row in enumerate(temp_unstr):
        temp_unstr[i] = np.convolve(row, np.ones(window)/window, 'same')
        try:
            temp_str[i] = np.convolve(temp_str[i], np.ones(window)/window, 'same')
        except IndexError:
            pass
    
    temp_unstr = np.array(temp_unstr)
    temp_str = np.array(temp_str)
    # 
    # works only for odd windows
    temp_unstr[:, 0:(window//2)] = np.nan
    temp_unstr[:, (trialsinsess-window//2):trialsinsess] = np.nan
    
    temp_str[:, 0:(window//2)] = np.nan
    temp_str[:, (trialsinsess-window//2):trialsinsess] = np.nan

# take mean of smoothed data   
mean_unstr = (np.nanmean(np.array(temp_unstr), axis = 0))
mean_str = (np.nanmean(np.array(temp_str), axis = 0))

# plot line 1
plt.plot(np.arange(trialsinsess), mean_unstr, color = color1, label = label1)#, marker = '.') 
plt.fill_between(np.arange(trialsinsess), mean_unstr+sem(temp_unstr, nan_policy = 'omit'),
                mean_unstr-sem(temp_unstr, nan_policy = 'omit'), color = color1, alpha = 0.2)

# plot line 2 (or comment)
plt.plot(np.arange(trialsinsess), mean_str, color = color2, label = label2)
plt.fill_between(np.arange(trialsinsess), mean_str+sem(temp_str, nan_policy = 'omit'), 
                mean_str-sem(temp_str, nan_policy = 'omit'), color = color2, alpha = 0.2)

# plot significance
# from scipy.stats import ttest_ind
# pvals = ttest_ind(temp_str, temp_unstr)[1]
# pvals_ind = [ind for ind, i in enumerate(pvals) if i < 0.05]
# plt.scatter(pvals_ind, [0.45]*len(pvals_ind), marker = '.', color = 'r', alpha = 0.4) #(6, 2, 60) for asterisks

# chance calc
prob_comb = np.array(list(permutations([80, 46, 15, 10]))+list(permutations([46, 80, 46, 15])))
str_prob_comb = np.array([[80, 46, 15, 10], [46, 80, 46, 15], [15, 46, 80, 46], [10, 15, 46, 80]])
unstr_prob_comb = prob_comb[~(prob_comb[:, None, :] == str_prob_comb[None, :, :]).all(axis = 2).any(axis = 1)]
hsb_prob_comb = [42.06]
# np.mean(unstr_prob_comb)
# plt.axhline(np.mean(prob_comb)/100, color = color1, linestyle = '--')
# plt.axhline(np.mean(str_prob_comb)/100, color = color2, linestyle = '--')
plt.axhline(1, color = 'k', linestyle = '--', linewidth = 0.75)

# plot settings
plt.xlabel('Trials in session', fontsize = 'x-large')
plt.ylabel('Compressibility', fontsize = 'x-large')
plt.xlim(0, trialsinsess)
# plt.ylim(0.4, 0.75)
plt.xticks([0, 25, 50, 75, 100], [1, '', '', '', 100], color = 'grey', fontsize = 'large')
# plt.yticks([0.4, 0.5, 0.6, 0.7], [0.4, '', '', 0.7], color = 'grey', fontsize = 'large')
# plt.title(f'n = {temp_unstr.shape[0]}')

leg = plt.legend(frameon=False, loc = 'upper left')
for line, text in zip(leg.get_lines(), leg.get_texts()):
    text.set_color(line.get_color())
sns.despine()
# sns.set_context('talk')
plt.tight_layout()
# plt.savefig('C:/Users/dlab/OneDrive - Indian Institute of Science/Drawings/awp1/entropy_n_21.png', dpi = 300)


# if fitting exp curve is reqd
def logFunc(x, a, b):
    return a + b*np.log(x)

# popt_str, _ = curve_fit(logFunc, np.arange(150), temp_str, nan_policy = 'omit')
# popt_unstr, _ = curve_fit(logFunc, np.arange(trialsinsess), mean_unstr, nan_policy = 'omit')
# plt.plot(np.arange(150), logFunc(np.arange(150), *popt_str), '#408ec6', linestyle= 'dotted')
# plt.plot(np.arange(trialsinsess), logFunc(np.arange(trialsinsess), *popt_unstr), color1, linestyle= 'dotted')

In [67]:
lzw_port_dict = {}
for entry, val in port_dict.items():
    lzw_port_dict[entry] = np.array([lzw_compress("".join(str(num) for num in row), True)[1] for row in port_dict[entry]])

rat = 'Quaxly'
# session-wise lzw comp ratio 
plt.bar(np.arange(lzw_port_dict[(rat, 'unstr')].shape[0]), lzw_port_dict[(rat, 'unstr')])

<BarContainer object of 125 artists>

In [71]:
lzw_compress(''.join(port_dict['Quaxly', 'unstr'][7].astype(str)), True)

([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 8], 7.142857142857143)